In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# This tells the website we're a regular browser
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

# The URL you gave me
url = "https://oeil.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD"

# Try to fetch the page
print("Fetching the page...")
response = requests.get(url, headers=HEADERS)
print(f"Status code: {response.status_code}")  # 200 means success!

# Look at the first 500 characters of HTML
print("\nFirst 500 characters of HTML:")
print(response.text[:500])

Fetching the page...
Status code: 200

First 500 characters of HTML:
<!doctype html>
<html lang="en">


<head>

    
    <title>Search | Legislative Observatory | European Parliament</title>

    <!-- Required meta tags -->
    <meta charset="utf-8">
    <meta content="@project.version@" name="project_version"/>
    <meta content="@build.number@" name="build_number"/>
    <meta content="@build.time@" name="build_time"/>

    <!-- Commons meta tags -->
    <meta content="https://www.europarl.europa.eu">
    <meta name="viewport" content="width=device-width, initia


In [2]:
# Parse the HTML
soup = BeautifulSoup(response.content, 'html.parser')

# Find all links that go to procedure pages
procedure_links = []
for link in soup.find_all('a', href=True):
    href = link['href']
    if '/procedure-file?reference=' in href:
        full_url = "https://oeil.europarl.europa.eu" + href
        procedure_links.append(full_url)

# Remove duplicates (just in case)
procedure_links = list(set(procedure_links))

print(f"Found {len(procedure_links)} procedure links on the first page!")
print("\nFirst 5 links:")
for link in procedure_links[:5]:
    print(link)

Found 30 procedure links on the first page!

First 5 links:
https://oeil.europarl.europa.eu/oeil/en/procedure-file?reference=2025/0429(COD)
https://oeil.europarl.europa.eu/oeil/en/procedure-file?reference=2025/0421(COD)
https://oeil.europarl.europa.eu/oeil/en/procedure-file?reference=2026/0045(COD)
https://oeil.europarl.europa.eu/oeil/en/procedure-file?reference=2026/0011(COD)
https://oeil.europarl.europa.eu/oeil/en/procedure-file?reference=2025/0408(COD)


In [3]:
def parse_search_results(html):
    """Extract all available information from search page HTML"""
    soup = BeautifulSoup(html, 'html.parser')
    procedures = []
    
    for doc in soup.find_all('div', class_='es_document-wrapper'):
        proc = {}
        
        # Find the main title/header area
        title_tag = doc.find('h3', class_='es_document-title')
        if title_tag:
            link_tag = title_tag.find('a')
            if link_tag:
                proc['url'] = "https://oeil.europarl.europa.eu" + link_tag['href']
                # Extract reference from URL
                import re
                match = re.search(r'reference=([^&]+)', link_tag['href'])
                if match:
                    proc['reference'] = match.group(1)
            
            # Extract committees from badges inside the title
            committees = []
            badge_spans = title_tag.find_all('span', class_='es_badge-committee')
            for badge in badge_spans:
                committees.append(badge.get_text(strip=True))
            if committees:
                proc['committees'] = ', '.join(committees)
        
        # Get the title text
        body = doc.find('div', class_='es_document-body')
        if body:
            title_p = body.find('p')
            if title_p:
                proc['title'] = title_p.get_text(strip=True)
        
        # Extract rapporteur(s) - improved
        rapporteur_div = doc.find('div', class_='es_document-subtitle', 
                                string=lambda text: text and 'Rapporteur' in text)
        if rapporteur_div:
            # Find all author spans
            author_spans = rapporteur_div.find_all('span', class_='es_document-subtitle-author')
            rapporteurs = [span.get_text(strip=True) for span in author_spans]
            if rapporteurs:
                proc['rapporteur'] = '; '.join(rapporteurs)
        
        # Only add if we found something
        if proc:
            procedures.append(proc)
    
    return procedures

In [4]:
def get_total_results_from_page(html):
    soup = BeautifulSoup(html, 'html.parser')
    total_span = soup.find('span', class_='document-search-total-results-count')
    if total_span:
        # Remove commas and convert to int
        total_text = total_span.get_text(strip=True).replace(',', '')
        return int(total_text)
        print(int(total_text))
    return None

In [5]:
def fetch_all_pages():
    all_procedures = []
    page = 1
    results_per_page = 30
    
    # First, fetch page 1 to get total
    first_url = f"https://oeil.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD&pageIndex=1"
    print("Fetching first page to get total...")
    response = requests.get(first_url, headers=HEADERS, timeout=10)
    response.raise_for_status()
    
    total_results = get_total_results_from_page(response.text)
    if total_results is None:
        print("Could not find total results count. Using hardcoded 22062.")
        total_results = 22062
    
    total_pages = (total_results + results_per_page - 1) // results_per_page
    print(f"Total results: {total_results}, total pages: {total_pages}")
    
    # Parse first page
    first_page_results = parse_search_results(response.text)
    all_procedures.extend(first_page_results)
    print(f"  Page 1: found {len(first_page_results)} procedures")
    
    # Loop for remaining pages
    page = 2
    while page <= total_pages:
        print(f"Fetching page {page}/{total_pages}...")
        url = f"https://oeil.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD&pageIndex={page}"
        
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
            response.raise_for_status()
            
            page_results = parse_search_results(response.text)
            all_procedures.extend(page_results)
            
            print(f"  Found {len(page_results)} procedures on this page")
            
            page += 1
            time.sleep(1)
            
        except Exception as e:
            print(f"Error on page {page}: {e}")
            break
    
    return all_procedures

In [6]:
# Test on first page
url = "https://oeil.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD&pageIndex=1"
response = requests.get(url, headers=HEADERS)
first_page_results = parse_search_results(response.text)

# Check how many have rapporteur
with_rapporteur = [p for p in first_page_results if 'rapporteur' in p]
print(f"Total on first page: {len(first_page_results)}")
print(f"With rapporteur: {len(with_rapporteur)}")
print("Sample with rapporteur:")
for p in with_rapporteur[:3]:
    print(p.get('reference'), p.get('rapporteur'))

Total on first page: 30
With rapporteur: 0
Sample with rapporteur:


In [ ]:
print("Starting to fetch all procedures...")
all_data = fetch_all_pages()
print(f"\nTotal procedures collected: {len(all_data)}")

# Create DataFrame
df_all = pd.DataFrame(all_data)
print(f"DataFrame shape: {df_all.shape}")
print("\nFirst 5 rows:")
print(df_all.head())

# # Save to CSV
# df_all.to_csv('europarl_all_procedures.csv', index=False)
# print("\nSaved all data to 'europarl_all_procedures.csv'")

In [ ]:
df_all

In [ ]:
df_all.to_csv('oeil_all_procedures_landing_page.csv', index=False)

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from typing import List, Optional, Union, Tuple

# ---------- existing functions (unchanged, kept for reference) ----------
def fetch_html(url: str) -> str:
    response = requests.get(url)
    response.raise_for_status()
    return response.text

def find_sections_by_header(soup: BeautifulSoup, header_text: str) -> List[BeautifulSoup]:
    sections = []
    pattern = re.compile(re.escape(header_text), re.IGNORECASE)
    for elem in soup.find_all(string=pattern):
        parent = elem.parent
        while parent and parent.name not in ['h1','h2','h3','h4','h5','h6','p','div','section']:
            parent = parent.parent
        if not parent:
            parent = elem.parent
        next_div = parent.find_next('div', class_='table-responsive')
        if next_div and next_div not in sections:
            sections.append(next_div)
    return sections

def extract_committee_from_table(procedure_div: BeautifulSoup, label_texts: Union[str, List[str]]) -> Optional[str]:
    if isinstance(label_texts, str):
        label_texts = [label_texts]
    table = procedure_div.find('table')
    if not table:
        return None
    rows = table.find_all('tr')
    for row in rows:
        cells = row.find_all('td')
        if len(cells) < 2:
            continue
        label = cells[0].get_text(strip=True)
        if any(lt.lower() in label.lower() for lt in label_texts):
            p = cells[1].find('p')
            if p:
                return p.get_text(strip=True)
            return cells[1].get_text(strip=True)
    return None

def count_rollcall_sections(soup: BeautifulSoup, header_text: str) -> int:
    pattern = re.compile(re.escape(header_text), re.IGNORECASE)
    return len(list(soup.find_all(string=pattern)))

# ---------- new helper functions ----------
def find_all_divs_after_header(soup: BeautifulSoup, header_text: str, class_name: str = 'table-responsive') -> List[BeautifulSoup]:
    """
    Find all divs with class `class_name` that appear after an element containing header_text.
    Returns a list of those divs.
    """
    divs = []
    pattern = re.compile(re.escape(header_text), re.IGNORECASE)
    for elem in soup.find_all(string=pattern):
        parent = elem.parent
        while parent and parent.name not in ['h1','h2','h3','h4','h5','h6','p','div','section']:
            parent = parent.parent
        if not parent:
            parent = elem.parent
        # find all following divs with the given class
        for sibling in parent.find_all_next('div', class_=class_name):
            # stop if we hit another header? For simplicity we collect all until end of section.
            # But the page structure is consistent: all such divs under the same heading belong to it.
            if sibling not in divs:
                divs.append(sibling)
    return divs

def extract_committees_from_table(procedure_div: BeautifulSoup, label_texts: Union[str, List[str]]) -> List[str]:
    """
    Extract all committee codes from a procedure table.
    Finds the row whose first cell contains any of the label_texts (case‑insensitive),
    then collects the text from all subsequent cells (or their first paragraphs).
    Returns a list of committee codes.
    """
    if isinstance(label_texts, str):
        label_texts = [label_texts]
    table = procedure_div.find('table')
    if not table:
        return []
    rows = table.find_all('tr')
    for row in rows:
        cells = row.find_all('td')
        if len(cells) < 2:
            continue
        label = cells[0].get_text(strip=True)
        if any(lt.lower() in label.lower() for lt in label_texts):
            committees = []
            # start from cell 1 (index 1) because cell 0 is the label
            for cell in cells[1:]:
                # committee code is usually in the first paragraph, otherwise whole cell text
                p = cell.find('p')
                text = p.get_text(strip=True) if p else cell.get_text(strip=True)
                if text:  # ignore empty cells
                    committees.append(text)
            return committees
    return []

# ---------- new main function ----------
def get_committee_votes_with_html(url: str, procedure_number: str) -> pd.DataFrame:
    """
    Fetch the HTML from the given Europarl document URL and extract
    information about responsible and opinion committees, including the
    full HTML of the procedure and roll‑call tables.

    Parameters
    ----------
    url : str
        The URL of the document (e.g. https://www.europarl.europa.eu/doceo/document/...)
    procedure_number : str
        The procedure identifier, e.g. '2021/0366(COD)'

    Returns
    -------
    pd.DataFrame
        Columns: procedure, committee, designation, procedure_available,
                 rollcall_available, procedure_string, rollcall_string
    """
    html = fetch_html(url)
    soup = BeautifulSoup(html, 'html.parser')

    # ----- Check for "document not in HTML" placeholder -----
    placeholder = soup.select_one('div.card.mb-3 div.card-body p.bg-white.alert.alert-info.mb-3')
    if placeholder and "This document does not exist in HTML but is available in the following formats:" in placeholder.get_text():
        return pd.DataFrame([{
            'procedure': procedure_number,
            'committee': 'other format',
            'designation': 'other format',
            'procedure_available': False,
            'rollcall_available': False,
            'procedure_string': '',
            'rollcall_string': ''
        }])

    # ---- Collect responsible committee and its procedure table ----
    resp_procedure_div = None
    resp_committee = None

    # Try specific header first
    resp_sections = find_sections_by_header(soup, "PROCEDURE – COMMITTEE RESPONSIBLE")
    for div in resp_sections:
        code = extract_committee_from_table(div, "Committee responsible")
        if code:
            resp_committee = code
            resp_procedure_div = div
            break

    # If not found, try generic "PROCEDURE"
    if not resp_committee:
        generic_sections = find_sections_by_header(soup, "PROCEDURE")
        for div in generic_sections:
            code = extract_committee_from_table(div, "Committee responsible")
            if code:
                resp_committee = code
                resp_procedure_div = div
                break

    # ---- Collect responsible roll‑call tables ----
    resp_rollcall_divs = find_all_divs_after_header(soup, "FINAL VOTE BY ROLL CALL IN COMMITTEE RESPONSIBLE")
    resp_rollcall_exists = len(resp_rollcall_divs) > 0

    # ---- Collect opinion committees with their source tables ----
    # We'll store tuples (committee_code, procedure_div)
    opinion_entries = []

    # 1. Sections with header "PROCEDURE – COMMITTEE ASKED FOR OPINION"
    opinion_sections = find_sections_by_header(soup, "PROCEDURE – COMMITTEE ASKED FOR OPINION")
    for div in opinion_sections:
        codes = extract_committees_from_table(div, ["Opinion by", "Committee(s) asked for opinion(s)"])
        for code in codes:
            opinion_entries.append((code, div))

    # 2. Sections with header "OPINION"
    if not opinion_entries:
        opinion_headers_sections = find_sections_by_header(soup, "OPINION")
        for div in opinion_headers_sections:
            codes = extract_committees_from_table(div, ["Opinion by", "Committee(s) asked for opinion(s)"])
            for code in codes:
                opinion_entries.append((code, div))

    # 3. Generic "PROCEDURE" sections (only if we haven't found any yet, and avoid duplicates)
    if not opinion_entries:
        generic_sections = find_sections_by_header(soup, "PROCEDURE")
        existing = set()
        for div in generic_sections:
            codes = extract_committees_from_table(div, ["Opinion by", "Committee(s) asked for opinion(s)"])
            for code in codes:
                if code.upper() not in existing:
                    opinion_entries.append((code, div))
                    existing.add(code.upper())

    # ---- Collect opinion roll‑call tables ----
    opinion_rollcall_divs = find_all_divs_after_header(soup, "FINAL VOTE BY ROLL CALL IN COMMITTEE ASKED FOR OPINION")
    opinion_rollcall_count = len(opinion_rollcall_divs)

    # ---- Build rows ----
    rows = []

    # Responsible committee row
    if resp_committee:
        proc_html = str(resp_procedure_div) if resp_procedure_div else ''
        roll_html = ''.join(str(div) for div in resp_rollcall_divs) if resp_rollcall_exists else ''
        rows.append({
            'procedure': procedure_number,
            'committee': resp_committee,
            'designation': 'lead',
            'procedure_available': resp_procedure_div is not None,
            'rollcall_available': resp_rollcall_exists,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })
    elif resp_sections:  # section exists but no committee code found
        proc_html = str(resp_sections[0]) if resp_sections else ''
        roll_html = ''.join(str(div) for div in resp_rollcall_divs) if resp_rollcall_exists else ''
        rows.append({
            'procedure': procedure_number,
            'committee': 'UNKNOWN',
            'designation': 'lead',
            'procedure_available': True,
            'rollcall_available': resp_rollcall_exists,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })

    # Opinion committee rows
    for i, (comm, proc_div) in enumerate(opinion_entries):
        proc_html = str(proc_div) if proc_div else ''
        # rollcall available only for the first `opinion_rollcall_count` committees
        roll_avail = i < opinion_rollcall_count
        roll_html = ''.join(str(div) for div in opinion_rollcall_divs) if roll_avail else ''
        rows.append({
            'procedure': procedure_number,
            'committee': comm,
            'designation': 'opinion',
            'procedure_available': True,  # we have a procedure div
            'rollcall_available': roll_avail,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })

    return pd.DataFrame(rows)

In [8]:
df = get_committee_votes_with_html(
    url='https://www.europarl.europa.eu/doceo/document/A-4-1995-0221_EN.html',
    procedure_number='2021/0366(COD)'
)

df

,procedure,committee,designation,procedure_available,rollcall_available,procedure_string,rollcall_string
0,2021/0366(COD),other format,other format,False,False,,


In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import re

def parse_procedure_table(html_str: str, committee_type: str = 'lead') -> list:
    """
    Parse the procedure table HTML and extract relevant rows.

    Parameters
    ----------
    html_str : str
        The HTML content of the div.table-responsive containing the procedure table.
    committee_type : str
        Either 'lead' or 'opinion'. Determines which header texts to look for.

    Returns
    -------
    list of dict
        Each dict has keys 'head' and 'value'. The head is the cleaned left cell text.
    """
    soup = BeautifulSoup(html_str, 'html.parser')
    table = soup.find('table')
    if not table:
        return []

    rows = table.find_all('tr')
    extracted = []

    # Mapping of expected head texts for lead vs opinion
    # We'll use these to decide how to parse each row
    for tr in rows:
        cells = tr.find_all(['td', 'th'])
        if not cells:
            continue

        # First cell is the head / label
        head_cell = cells[0]
        head = head_cell.get_text(strip=True)
        if not head:
            continue

        # Normalise head (strip, lower for matching, but keep original for output)
        head_lower = head.lower()

        # Skip rows that are not of interest (like empty or just formatting)
        # But we'll keep all and filter later; better to only process known heads.
        # However, the user may want all rows; we'll process all and let the caller decide.
        # We'll handle special cases based on head text.

        # Determine number of value cells
        value_cells = cells[1:]

        # Special handling based on head
        if 'result of final vote' in head_lower:
            # Expect pattern: +: number, -: number, 0: number interleaved
            # Iterate through value cells, treat pairs (label, number)
            i = 0
            while i < len(value_cells):
                cell = value_cells[i]
                text = cell.get_text(strip=True)
                # Check if this cell contains a vote symbol
                if text in ['+:', '–:', '-:', '0:']:
                    # Next cell should contain the number
                    if i+1 < len(value_cells):
                        num_cell = value_cells[i+1]
                        num_text = num_cell.get_text(strip=True)
                        # Determine head suffix
                        if text in ['+:', '+']:
                            suffix = 'pos'
                        elif text in ['–:', '-:']:
                            suffix = 'neg'
                        elif text == '0:':
                            suffix = 'abs'
                        else:
                            suffix = text.strip(':')
                        extracted.append({
                            'head': f'Result {suffix}',
                            'value': num_text
                        })
                        i += 2
                    else:
                        i += 1
                else:
                    # No symbol, just treat as normal? Should not happen.
                    # But if the numbers are all in one cell? In example they are separate.
                    # We'll skip or handle as fallback.
                    i += 1
            continue  # already added rows

        elif 'rapporteur' in head_lower and 'opinion' not in head_lower:
            # For lead: "Rapporteurs" (maybe multiple rapporteurs? But usually one)
            # Expect first value cell = name, second = date
            # If there are multiple rapporteurs? Unlikely, but could be.
            # We'll extract all non-empty cells, assume alternating name/date? Not reliable.
            # Simpler: treat first value cell as name, second as date (if exists)
            name = ''
            date = ''
            for cell in value_cells:
                text = cell.get_text(strip=True)
                if text and not name:
                    name = text
                elif text and not date:
                    date = text
                # ignore extra cells
            if name:
                extracted.append({'head': 'Rapporteur name', 'value': name})
            if date:
                extracted.append({'head': 'Rapporteur date appointed', 'value': date})

        elif 'rapporteur for the opinion' in head_lower:
            # Opinion rapporteur
            name = ''
            date = ''
            for cell in value_cells:
                text = cell.get_text(strip=True)
                if text and not name:
                    name = text
                elif text and not date:
                    date = text
            if name:
                extracted.append({'head': 'Rapporteur opinion name', 'value': name})
            if date:
                extracted.append({'head': 'Rapporteur opinion date appointed', 'value': date})

        elif any(x in head_lower for x in ['members present', 'substitutes present']):
            # These rows contain lists of names. Collect all text from all value cells,
            # then split by commas or just take the whole text and clean.
            all_text = ' '.join(cell.get_text(separator=' ', strip=True) for cell in value_cells)
            # Clean multiple spaces, commas, etc. Keep as is.
            # But we need to ensure names are separated by commas.
            # The text may already have commas; if not, we can split by common patterns.
            # For simplicity, we'll just store the whole cleaned text.
            extracted.append({'head': head, 'value': all_text})

        elif head_lower in ['title', 'committee responsible', 'opinion by', 'committee(s) asked for opinion(s)',
                            'discussed in committee', 'date adopted']:
            # Simple single value: collect text from all value cells and join.
            value = ' '.join(cell.get_text(separator=' ', strip=True) for cell in value_cells)
            extracted.append({'head': head, 'value': value})

        else:
            # For any other row, we still capture it? Maybe skip. But the user might want all.
            # We'll capture anyway.
            value = ' '.join(cell.get_text(separator=' ', strip=True) for cell in value_cells)
            if value:
                extracted.append({'head': head, 'value': value})

    return extracted


def expand_committee_procedure(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes the DataFrame from get_committee_votes_with_html and expands the procedure_string
    into a long-format table with columns: procedure, committee, designation, head, value.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: procedure, committee, designation, procedure_available, procedure_string.

    Returns
    -------
    pd.DataFrame
        Long-format table with extracted head-value pairs.
    """
    rows = []
    for _, row in df.iterrows():
        if not row['procedure_available']:
            continue
        html = row['procedure_string']
        if not html or pd.isna(html):
            continue
        # Determine committee type based on designation
        ctype = 'lead' if row['designation'] == 'lead' else 'opinion'
        extracted = parse_procedure_table(html, committee_type=ctype)
        for item in extracted:
            rows.append({
                'procedure': row['procedure'],
                'committee': row['committee'],
                'designation': row['designation'],
                'head': item['head'],
                'value': item['value']
            })
    return pd.DataFrame(rows)




In [3]:
df = pd.read_csv('bld\\test_procedures_committee_vote_info.csv')

In [34]:
from bs4 import BeautifulSoup
import pandas as pd
import re

def parse_procedure_table(html_str: str, committee_type: str = 'lead') -> list:
    """
    Parse the procedure table HTML and extract relevant rows as head-value pairs.
    Handles multi-line heads and multiple value cells.

    Parameters
    ----------
    html_str : str
        The HTML content of the div.table-responsive containing the procedure table.
    committee_type : str
        Either 'lead' or 'opinion'. Used for context, but not heavily used now.

    Returns
    -------
    list of dict
        Each dict has keys 'head' and 'value'.
    """
    soup = BeautifulSoup(html_str, 'html.parser')
    table = soup.find('table')
    if not table:
        return []

    rows = table.find_all('tr')
    extracted = []

    for tr in rows:
        cells = tr.find_all(['td', 'th'])
        if len(cells) < 2:
            continue

        # Head cell (first cell)
        head_cell = cells[0]
        head_text = head_cell.get_text('\n', strip=True)  # preserve line breaks
        # Split head by lines, ignore empty
        head_parts = [h.strip() for h in head_text.split('\n') if h.strip()]

        # Value cells (all after first)
        value_cells = cells[1:]

        # --- Special handling for result of final vote ---
        if any('result of final vote' in part.lower() for part in head_parts):
            symbols = []
            numbers = []
            # Collect lines from all value cells
            for cell in value_cells:
                cell_text = cell.get_text('\n', strip=True)
                lines = [ln.strip() for ln in cell_text.split('\n') if ln.strip()]
                for line in lines:
                    # Check if line is a symbol (maybe with colon)
                    if re.match(r'^[+\-–0]:?$', line):
                        sym = line.rstrip(':')
                        symbols.append(sym)
                    elif line.isdigit():
                        numbers.append(line)
                    # ignore any other content (should not happen)
            # Pair symbols and numbers in order
            for sym, num in zip(symbols, numbers):
                extracted.append({
                    'head': f'Result of final vote {sym}',
                    'value': num
                })
            continue

        # --- Special handling for members/substitutes (keep as one row) ---
        if any(key in head_text.lower() for key in ['members present', 'substitutes present']):
            all_text = ' '.join(cell.get_text(separator=' ', strip=True) for cell in value_cells)
            # Clean up multiple spaces, but keep names as a block
            all_text = re.sub(r'\s+', ' ', all_text).strip()
            extracted.append({'head': head_text, 'value': all_text})
            continue

        # --- General case: align head parts with value parts ---
        # Collect all text from value cells, each cell may contain multiple lines
        value_parts = []
        for cell in value_cells:
            cell_text = cell.get_text('\n', strip=True)
            parts = [p.strip() for p in cell_text.split('\n') if p.strip()]
            value_parts.extend(parts)

        # If number of head parts equals number of value parts, zip them
        if len(head_parts) == len(value_parts):
            for h, v in zip(head_parts, value_parts):
                extracted.append({'head': h, 'value': v})
        else:
            # Possible cases:
            # 1. One head part, multiple value parts (e.g., list of committees in separate cells)
            #    We'll treat each value part as a separate row with same head.
            if len(head_parts) == 1 and len(value_parts) > 1:
                for v in value_parts:
                    extracted.append({'head': head_parts[0], 'value': v})
            # 2. Multiple head parts, but value parts are grouped per cell (e.g., committees with dates)
            #    We need to pair head parts with groups of value parts from each cell.
            #    For example, head = ["Committee asked", "Date announced"] and each value cell contains two lines.
            #    We'll assume each value cell has as many lines as head_parts.
            elif len(head_parts) > 1:
                # Process each value cell separately
                for cell in value_cells:
                    cell_lines = cell.get_text('\n', strip=True).split('\n')
                    cell_lines = [ln.strip() for ln in cell_lines if ln.strip()]
                    if len(cell_lines) == len(head_parts):
                        for h, v in zip(head_parts, cell_lines):
                            extracted.append({'head': h, 'value': v})
                    else:
                        # fallback: treat whole cell as one value for each head? Not good.
                        # Instead, just join and give one row per head? We'll skip for now.
                        pass
            else:
                # Fallback: just join all value parts and assign to first head
                extracted.append({'head': head_parts[0] if head_parts else head_text,
                                  'value': ' '.join(value_parts)})

    return extracted


def expand_committee_procedure(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes the DataFrame from get_committee_votes_with_html and expands the procedure_string
    into a long-format table with columns: procedure, committee, designation, head, value.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: procedure, committee, designation, procedure_available, procedure_string.

    Returns
    -------
    pd.DataFrame
        Long-format table with extracted head-value pairs.
    """
    rows = []
    for _, row in df.iterrows():
        if not row['procedure_available']:
            continue
        html = row['procedure_string']
        if not html or pd.isna(html):
            continue
        # Determine committee type based on designation
        ctype = 'lead' if row['designation'] == 'lead' else 'opinion'
        extracted = parse_procedure_table(html, committee_type=ctype)
        for item in extracted:
            rows.append({
                'procedure': row['procedure'],
                'committee': row['committee'],
                'designation': row['designation'],
                'head': item['head'],
                'value': item['value']
            })
    return pd.DataFrame(rows)




In [45]:
df_long = expand_committee_procedure(df)

df_long.head(50)

,procedure,committee,designation,head,value
0,2021/0366(COD),ENVI,lead,Title,Making available on the Union market as well a...
1,2021/0366(COD),ENVI,lead,References,COM(2021)0706
2,2021/0366(COD),ENVI,lead,References,– C9-0430/2021 –
3,2021/0366(COD),ENVI,lead,References,2021/0366(COD)
4,2021/0366(COD),ENVI,lead,Date submitted to Parliament,18.11.2021
5,2021/0366(COD),ENVI,lead,Committee responsible,ENVI
6,2021/0366(COD),ENVI,lead,Date announced in plenary,17.1.2022
7,2021/0366(COD),ENVI,lead,Committees asked for opinions,DEVE
8,2021/0366(COD),ENVI,lead,Date announced in plenary,7.4.2022
9,2021/0366(COD),ENVI,lead,Committees asked for opinions,INTA


In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import re

def parse_rollcall_table(html_str: str) -> list:
    """
    Parse the HTML of a roll‑call vote section (may contain multiple tables)
    and extract vote details.

    Parameters
    ----------
    html_str : str
        The HTML content containing one or more div.table-responsive with roll‑call tables.

    Returns
    -------
    list of dict
        Each dict has keys: vote_type, group, members.
        (The total count is stored in the first row of each table, but we can also
        include it in each row if needed – here we add it as 'total'.)
    """
    soup = BeautifulSoup(html_str, 'html.parser')
    tables = soup.find_all('table')
    results = []

    for table in tables:
        rows = table.find_all('tr')
        if not rows:
            continue

        # First row: contains total count and vote symbol
        first_row_cells = rows[0].find_all(['td', 'th'])
        if len(first_row_cells) < 2:
            continue
        total_text = first_row_cells[0].get_text(strip=True)
        symbol_cell = first_row_cells[1]
        # The symbol may be inside a <p> or directly; get the text and strip
        symbol = symbol_cell.get_text(strip=True)
        # Clean symbol: could be '+', '–', '-', '0' (sometimes with a colon)
        vote_type = symbol.rstrip(':')  # remove colon if present
        # total count might be a number; we keep as string
        total = total_text

        # Subsequent rows: group and members
        for row in rows[1:]:
            cells = row.find_all(['td', 'th'])
            if len(cells) < 2:
                continue
            group = cells[0].get_text(strip=True)
            # Members cell may contain a list of names, possibly split across <p> or <span>
            # We'll get all text and clean up.
            members_cell = cells[1]
            members_text = members_cell.get_text(separator=', ', strip=True)
            # Remove any extra whitespace
            members = re.sub(r'\s+', ' ', members_text).strip()
            # Sometimes names are separated by commas already, but get_text with separator adds extra commas.
            # We'll keep as is, but could split later if needed.
            results.append({
                'vote_type': vote_type,
                'total': total,
                'group': group,
                'members': members
            })

    return results


def expand_committee_rollcall(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes the DataFrame from get_committee_votes_with_html and expands the rollcall_string
    into a long-format table with columns:
        procedure, committee, designation, vote_type, group, members.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: procedure, committee, designation, rollcall_available, rollcall_string.

    Returns
    -------
    pd.DataFrame
        Long-format table with extracted roll‑call details.
    """
    rows = []
    for _, row in df.iterrows():
        if not row['rollcall_available']:
            continue
        html = row['rollcall_string']
        if not html or pd.isna(html):
            continue
        extracted = parse_rollcall_table(html)
        for item in extracted:
            rows.append({
                'procedure': row['procedure'],
                'committee': row['committee'],
                'designation': row['designation'],
                'vote_type': item['vote_type'],
                'total': item['total'],
                'group': item['group'],
                'members': item['members']
            })
    return pd.DataFrame(rows)

In [4]:
# Assuming df is your original DataFrame with procedure_string and rollcall_string
# For testing, pick the first row where rollcall_available is True
test_row = df[df['rollcall_available']].iloc[0]
html = test_row['rollcall_string']
parsed = parse_rollcall_table(html)
print(pd.DataFrame(parsed))

                                            vote_type  total  \
0   Making available on the Union market as well a...  Title   
1   Making available on the Union market as well a...  Title   
2   Making available on the Union market as well a...  Title   
3   Making available on the Union market as well a...  Title   
4   Making available on the Union market as well a...  Title   
..                                                ...    ...   
93                                                  -      2   
94                                                  0     13   
95                                                  0     13   
96                                                  0     13   
97                                                  0     13   

                                                group  \
0                                          References   
1      Committee responsibleDate announced in plenary   
2                 Opinion byDate announced in plenary   
3  

In [5]:
df1 = pd.DataFrame(parsed)

df1

,vote_type,total,group,members
0,Making available on the Union market as well a...,Title,References,"COM(2021)0706, – C9-0430/2021 –, 2021/0366(COD)"
1,Making available on the Union market as well a...,Title,Committee responsibleDate announced in plenary,"ENVI, 17.1.2022"
2,Making available on the Union market as well a...,Title,Opinion byDate announced in plenary,"INTA, 17.1.2022"
3,Making available on the Union market as well a...,Title,Associated committees - date announced in plenary,7.4.2022
4,Making available on the Union market as well a...,Title,Rapporteur for the opinionDate appointed,"Karin Karlsbro, 9.12.2021"
...,...,...,...,...
93,-,2,ECR,"Mazaly Aguilar, Hermann Tertsch"
94,0,13,ECR,"Sergio Berlato, Nicola Procaccini, Grzegorz To..."
95,0,13,ID,"Matteo Adinolfi, Simona Baldassarre, Aurélia B..."
96,0,13,NI,Edina Tóth


In [7]:
import requests
from bs4 import BeautifulSoup
import time
import re

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def parse_search_results(html):
    """Extract all available information from search page HTML"""
    soup = BeautifulSoup(html, 'html.parser')
    procedures = []
    
    for doc in soup.find_all('div', class_='es_document-wrapper'):
        proc = {}
        
        # Find the main title/header area
        title_tag = doc.find('h3', class_='es_document-title')
        if title_tag:
            link_tag = title_tag.find('a')
            if link_tag:
                proc['url'] = "https://oeil.europarl.europa.eu" + link_tag['href']
                # Extract reference from URL
                match = re.search(r'reference=([^&]+)', link_tag['href'])
                if match:
                    proc['reference'] = match.group(1)
            
            # Extract committees from badges inside the title
            committees = []
            badge_spans = title_tag.find_all('span', class_='es_badge-committee')
            for badge in badge_spans:
                committees.append(badge.get_text(strip=True))
            if committees:
                proc['committees'] = ', '.join(committees)
        
        # Get the title text
        body = doc.find('div', class_='es_document-body')
        if body:
            title_p = body.find('p')
            if title_p:
                proc['title'] = title_p.get_text(strip=True)
        
        # Extract rapporteur(s) - improved
        rapporteur_div = doc.find('div', class_='es_document-subtitle', 
                                string=lambda text: text and 'Rapporteur' in text)
        if rapporteur_div:
            # Find all author spans
            author_spans = rapporteur_div.find_all('span', class_='es_document-subtitle-author')
            rapporteurs = [span.get_text(strip=True) for span in author_spans]
            if rapporteurs:
                proc['rapporteur'] = '; '.join(rapporteurs)
        
        # Only add if we found something
        if proc:
            procedures.append(proc)
    
    return procedures

def get_total_results_from_page(html):
    soup = BeautifulSoup(html, 'html.parser')
    total_span = soup.find('span', class_='document-search-total-results-count')
    if total_span:
        # Remove commas and convert to int
        total_text = total_span.get_text(strip=True).replace(',', '')
        return int(total_text)
    return None

def fetch_all_pages():
    all_procedures = []
    page = 1
    results_per_page = 30
    
    # First, fetch page 1 to get total
    first_url = f"https://oeil.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD&pageIndex=1"
    print("Fetching first page to get total...")
    response = requests.get(first_url, headers=HEADERS, timeout=10)
    response.raise_for_status()
    
    total_results = get_total_results_from_page(response.text)
    if total_results is None:
        print("Could not find total results count. Using hardcoded 22062.")
        total_results = 22062
    
    total_pages = (total_results + results_per_page - 1) // results_per_page
    print(f"Total results: {total_results}, total pages: {total_pages}")
    
    # Parse first page
    first_page_results = parse_search_results(response.text)
    all_procedures.extend(first_page_results)
    print(f"  Page 1: found {len(first_page_results)} procedures")
    
    # Loop for remaining pages
    page = 2
    while page <= total_pages:
        print(f"Fetching page {page}/{total_pages}...")
        url = f"https://oeil.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD&pageIndex={page}"
        
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
            response.raise_for_status()
            
            page_results = parse_search_results(response.text)
            all_procedures.extend(page_results)
            
            print(f"  Found {len(page_results)} procedures on this page")
            
            page += 1
            time.sleep(1)
            
        except Exception as e:
            print(f"Error on page {page}: {e}")
            break
    
    return all_procedures

In [8]:
import pandas as pd

In [12]:
df = pd.read_csv(r"bld\oeil_all_procedures_landing_page.csv")

In [13]:
df

,url,reference,committees,title
0,https://oeil.europarl.europa.eu/oeil/en/proced...,2025/0394(COD),"Environment, Climate and Food Safety",Simplification of some requirements and reduct...
1,https://oeil.europarl.europa.eu/oeil/en/proced...,2025/0393(COD),"Environment, Climate and Food Safety",Simplification of certain requirements for the...
2,https://oeil.europarl.europa.eu/oeil/en/proced...,2025/0391(COD),"Environment, Climate and Food Safety",Speeding-up environmental assessments
3,https://oeil.europarl.europa.eu/oeil/en/proced...,2025/0384(COD),"International Trade, Civil Liberties, Justice ...",Monitoring and controlling drug precursors
4,https://oeil.europarl.europa.eu/oeil/en/proced...,2025/3502(COD),"Security and Defence, International Trade, Bud...",Establishing the Reparations Loan to Ukraine
...,...,...,...,...
22026,https://oeil.europarl.europa.eu/oeil/en/proced...,C(2025)0713,"Civil Liberties, Justice and Home Affairs",Inclusion of Romania and Bulgaria in the annua...
22027,https://oeil.europarl.europa.eu/oeil/en/proced...,C(2025)0712,"Civil Liberties, Justice and Home Affairs",Provision of timeschedule for the periodic eva...
22028,https://oeil.europarl.europa.eu/oeil/en/proced...,C(2025)0954,Constitutional Affairs,EU in 2024. General report on the activities o...
22029,https://oeil.europarl.europa.eu/oeil/en/proced...,C(2024)3860,Internal Market and Consumer Protection,European citizens’ initiative: ‘Stop Destroyin...


In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re
import pandas as pd
from IPython.display import display

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def parse_search_results(html):
    """Extract reference, title, committees, rapporteur, stage, url."""
    soup = BeautifulSoup(html, 'html.parser')
    procedures = []
    for doc in soup.find_all('div', class_='es_document-wrapper'):
        proc = {}
        title_tag = doc.find('h3', class_='es_document-title')
        if title_tag:
            link = title_tag.find('a')
            if link and link.get('href'):
                proc['url'] = "https://oeil.europarl.europa.eu" + link['href']
                ref_match = re.search(r'reference=([^&]+)', link['href'])
                if ref_match:
                    proc['reference'] = ref_match.group(1)
            badges = title_tag.find_all('span', class_='es_badge-committee')
            if badges:
                proc['committees'] = ', '.join(b.get_text(strip=True) for b in badges)
            t_item = title_tag.find('span', class_='t-item')
            if t_item:
                text = t_item.get_text(separator=' ', strip=True)
                parts = text.split()
                if len(parts) >= 2 and re.match(r'^\*+[I]+$', parts[1]):
                    proc['stage'] = parts[1]
        body = doc.find('div', class_='es_document-body')
        if body:
            p = body.find('p')
            if p:
                proc['title'] = p.get_text(strip=True)
        rapporteur_div = doc.find('div', string=re.compile(r'Rapporteur\s*:'))
        if rapporteur_div:
            authors = rapporteur_div.find_all('span', class_='es_document-subtitle-author')
            if authors:
                proc['rapporteur'] = '; '.join(a.get_text(strip=True) for a in authors)
        if proc.get('reference') or proc.get('title'):
            procedures.append(proc)
    return procedures

def get_total_results(html):
    soup = BeautifulSoup(html, 'html.parser')
    span = soup.find('span', class_='document-search-total-results-count')
    if span:
        return int(span.get_text(strip=True).replace(',', ''))
    return None

def fetch_page_with_retry(url, retries=3, delay=2):
    """Fetch a page, retrying on failure."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            resp.raise_for_status()
            procs = parse_search_results(resp.text)
            if procs:
                return procs, resp.text
            print(f"  (attempt {attempt+1} returned 0, retrying...)")
            time.sleep(delay * (attempt + 1))
        except Exception as e:
            print(f"  (attempt {attempt+1} failed: {e}, retrying...)")
            time.sleep(delay * (attempt + 1))
    return [], None

def fetch_all_procedures(procedure_type=8):
    all_procs = []
    page_stats = {}
    page = 1
    expected_total = None
    results_per_page = 30
    empty_streak = 0
    max_empty_pages = 3   # stop after this many consecutive empty pages
    max_pages = 200       # safety limit

    while page <= max_pages:
        url = f"https://oeil.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD&pageIndex={page}&procedureType={procedure_type}"
        print(f"Fetching page {page}...", end=' ')

        # Get total from first page
        if page == 1:
            try:
                r = requests.get(url, headers=HEADERS, timeout=10)
                r.raise_for_status()
                expected_total = get_total_results(r.text)
                if expected_total:
                    print(f"Total results: {expected_total}")
                else:
                    print("Could not determine total.")
                first_procs = parse_search_results(r.text)
                all_procs.extend(first_procs)
                count = len(first_procs)
                page_stats[page] = count
                print(f"found {count} procedures")
                page += 1
                time.sleep(1)
                continue
            except Exception as e:
                print(f"Error on first page: {e}")
                break

        # Fetch subsequent pages with retry
        procs, html = fetch_page_with_retry(url, retries=3, delay=2)
        count = len(procs)
        page_stats[page] = count
        all_procs.extend(procs)
        print(f"found {count} procedures")

        # # Save HTML for pages that have fewer than 30 (and aren't the last expected)
        # if expected_total:
        #     last_expected_page = (expected_total + results_per_page - 1) // results_per_page
        #     if page < last_expected_page and count < results_per_page:
        #         print(f"  ⚠️  Page {page} has only {count} (expected {results_per_page}). Saving HTML.")
        #         with open(f"debug_page_{page}.html", "w", encoding="utf-8") as f:
        #             f.write(html)

        # Stop condition: three consecutive empty pages
        if count == 0:
            empty_streak += 1
            if empty_streak >= max_empty_pages:
                print(f"{max_empty_pages} consecutive empty pages – stopping.")
                break
        else:
            empty_streak = 0

        # Optional: also stop if we've passed the last expected page and have empty pages
        if expected_total and page > last_expected_page and empty_streak >= 1:
            # Already beyond last expected, and this page is empty – safe to stop
            pass  # but we already have the three-empty condition

        page += 1
        time.sleep(1)

    return all_procs, page_stats, expected_total

# --- Run in Jupyter ---
print("Scraping COD procedures...")
data, stats, expected = fetch_all_procedures(procedure_type=8)
df = pd.DataFrame(data)

# Reorder columns for clarity
desired_order = ['reference', 'title', 'stage', 'committees', 'rapporteur', 'url']
df = df[[col for col in desired_order if col in df.columns]]

print(f"\nCollected: {len(df)} procedures")
if expected:
    print(f"Expected (from first page): {expected}")
    if len(df) != expected:
        print("⚠️  Discrepancy! Pages with unusual counts:")
        total_pages = (expected + 30 - 1) // 30
        for p in range(1, total_pages + 1):
            cnt = stats.get(p, 0)
            if p < total_pages and cnt != 30:
                print(f"  Page {p}: {cnt} (expected 30)")
            elif p == total_pages:
                expected_last = expected % 30 or 30
                if cnt != expected_last:
                    print(f"  Page {p} (last): {cnt} (expected {expected_last})")
    else:
        print("✅ All pages match expected count.")
else:
    print("No expected total available.")

display(df.head(10))

Scraping COD procedures...
Fetching page 1... Total results: 2903
found 30 procedures
Fetching page 2... found 30 procedures
Fetching page 3... found 30 procedures
Fetching page 4... found 30 procedures
Fetching page 5... found 30 procedures
Fetching page 6... found 30 procedures
Fetching page 7... found 30 procedures
Fetching page 8... found 30 procedures
Fetching page 9... found 30 procedures
Fetching page 10... found 30 procedures
Fetching page 11... found 30 procedures
Fetching page 12... found 30 procedures
Fetching page 13... found 30 procedures
Fetching page 14... found 30 procedures
Fetching page 15... found 30 procedures
Fetching page 16... found 30 procedures
Fetching page 17... found 30 procedures
Fetching page 18... found 30 procedures
Fetching page 19... found 30 procedures
Fetching page 20... found 30 procedures
Fetching page 21... found 30 procedures
Fetching page 22... found 30 procedures
Fetching page 23... found 30 procedures
Fetching page 24... found 30 procedures
Fe

,reference,title,stage,committees,url
0,2025/0394(COD),Simplification of some requirements and reduct...,***I,"Environment, Climate and Food Safety",https://oeil.europarl.europa.eu/oeil/en/proced...
1,2025/0393(COD),Simplification of certain requirements for the...,***I,"Environment, Climate and Food Safety",https://oeil.europarl.europa.eu/oeil/en/proced...
2,2025/0391(COD),Speeding-up environmental assessments,***I,"Environment, Climate and Food Safety",https://oeil.europarl.europa.eu/oeil/en/proced...
3,2025/0384(COD),Monitoring and controlling drug precursors,***I,"International Trade, Civil Liberties, Justice ...",https://oeil.europarl.europa.eu/oeil/en/proced...
4,2025/3502(COD),Establishing the Reparations Loan to Ukraine,***I,"Security and Defence, International Trade, Bud...",https://oeil.europarl.europa.eu/oeil/en/proced...
5,2025/0385(COD),Amendments to the Critical Raw Materials Act,***I,"Industry, Research and Energy",https://oeil.europarl.europa.eu/oeil/en/proced...
6,2025/0383(COD),Amending certain Regulations as regards the fu...,***I,Economic and Monetary Affairs,https://oeil.europarl.europa.eu/oeil/en/proced...
7,2025/0382(COD),Amending certain Directives as regards the fur...,***I,Economic and Monetary Affairs,https://oeil.europarl.europa.eu/oeil/en/proced...
8,2025/0381(COD),Banks and financial markets: settlement finali...,***I,Economic and Monetary Affairs,https://oeil.europarl.europa.eu/oeil/en/proced...
9,2025/0380(COD),"Market stability reserve for the buildings, ro...",***I,"Environment, Climate and Food Safety",https://oeil.europarl.europa.eu/oeil/en/proced...


In [37]:
df.isna().sum()

reference       0
title           0
stage           0
committees    249
url             0
dtype: int64

In [40]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from IPython.display import display

# ---------------------------
# Helper functions
# ---------------------------
def fetch_html(url):
    """Fetch HTML from a URL."""
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.text

def extract_committee_from_table(div, label="Committee responsible"):
    """
    From a div containing a procedure table, extract the committee code
    by looking for a row whose first cell contains the label.
    """
    table = div.find('table')
    if not table:
        return None
    for row in table.find_all('tr'):
        cells = row.find_all('td')
        if len(cells) < 2:
            continue
        if label.lower() in cells[0].get_text(strip=True).lower():
            # committee code is in the second cell, often inside a <p>
            p = cells[1].find('p')
            return p.get_text(strip=True) if p else cells[1].get_text(strip=True)
    return None

def extract_committee_from_opinion_title(title):
    """
    Given a title like "OPINION of the Committee on Legal Affairs" or
    "OPINION OF THE COMMITTEE ON INTERNATIONAL TRADE", extract the committee name.
    Returns the raw name (e.g., "Legal Affairs", "INTERNATIONAL TRADE").
    """
    # Remove "OPINION" and any leading/trailing punctuation
    # Common patterns:
    # 1. "OPINION of the Committee on ..."
    # 2. "OPINION OF THE COMMITTEE ON ..."
    # 3. "OPINION ..." (fallback)
    title_clean = re.sub(r'^OPINION\s+', '', title, flags=re.IGNORECASE).strip()
    # Try to extract after "on " (case-insensitive)
    match = re.search(r'on\s+(.+)', title_clean, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    # Fallback: return the whole cleaned string
    return title_clean

# Optional mapping from committee names to standard codes
COMMITTEE_MAP = {
    "Legal Affairs": "JURI",
    "International Trade": "INTA",
    "Civil Liberties, Justice and Home Affairs": "LIBE",
    "Development": "DEVE",
    "Agriculture and Rural Development": "AGRI",
    "Internal Market and Consumer Protection": "IMCO",
    "Environment, Climate and Food Safety": "ENVI",  # etc.
    # Add more as needed
}
def map_committee(name):
    """Map a committee name (case-insensitive) to its standard code."""
    name_lower = name.lower()
    for key, code in COMMITTEE_MAP.items():
        if key.lower() in name_lower:
            return code
    return name  # fallback to raw name

# ---------------------------
# Main extraction function
# ---------------------------
def extract_procedure_data(url, procedure_number):
    """
    For a given document URL, extract lead and opinion committee information
    following the new approach: first parse the section links, then extract tables.
    """
    html = fetch_html(url)
    soup = BeautifulSoup(html, 'html.parser')

    # ----- Check for placeholder (document not in HTML) -----
    placeholder = soup.select_one('div.card.mb-3 div.card-body p.bg-white.alert.alert-info.mb-3')
    if placeholder and "This document does not exist in HTML" in placeholder.get_text():
        return pd.DataFrame([{
            'procedure': procedure_number,
            'committee': 'other format',
            'designation': 'other format',
            'procedure_available': False,
            'rollcall_available': False,
            'procedure_string': '',
            'rollcall_string': ''
        }])

    # ----- Find the section links list -----
    links_div = soup.find('div', class_='es_links-list')
    if not links_div:
        # If no links list, maybe the page has a different structure – return empty
        return pd.DataFrame()

    sections = []
    for li in links_div.find_all('li'):
        a = li.find('a')
        if not a:
            continue
        href = a.get('href', '')
        # The visible text is inside a span with class "t-x" (or nested spans)
        span = a.find('span', class_='t-x')
        if span:
            title = span.get_text(separator=' ', strip=True)
        else:
            title = a.get_text(strip=True)
        sections.append({'href': href, 'title': title})

    # ----- Identify lead procedure section -----
    lead_proc_section = None
    lead_proc_title = None
    # Look for section containing "PROCEDURE" but not "OPINION" (to avoid opinion sections)
    for sec in sections:
        title_upper = sec['title'].upper()
        if 'PROCEDURE' in title_upper and 'OPINION' not in title_upper:
            lead_proc_section = sec
            lead_proc_title = sec['title']
            break

    # ----- Identify lead roll call section -----
    lead_roll_section = None
    for sec in sections:
        if 'FINAL VOTE BY ROLL CALL IN COMMITTEE RESPONSIBLE' in sec['title'].upper():
            lead_roll_section = sec
            break

    # ----- Identify opinion sections (in order) -----
    opinion_sections = []
    for sec in sections:
        if 'OPINION' in sec['title'].upper():
            opinion_sections.append(sec)

    # ----- Identify opinion roll call sections (usually one section with multiple tables) -----
    opinion_roll_sections = []
    for sec in sections:
        if 'FINAL VOTE BY ROLL CALL IN COMMITTEE ASKED FOR OPINION' in sec['title'].upper():
            opinion_roll_sections.append(sec)

    # ----- Helper to get content div for a section -----
    def get_content_div(href):
        if not href or not href.startswith('#'):
            return None
        section_id = href[1:]  # remove '#'
        heading = soup.find(id=section_id)
        if heading:
            # The content is usually the next sibling div (or the next div after any whitespace)
            # We'll look for the next div that is not empty
            next_el = heading.find_next_sibling()
            if next_el and next_el.name == 'div':
                return next_el
            # Sometimes there's a div wrapped inside something else; try find_next('div')
            return heading.find_next('div')
        return None

    # ----- Process lead committee -----
    rows = []
    if lead_proc_section:
        content_div = get_content_div(lead_proc_section['href'])
        if content_div:
            proc_html = str(content_div)
            committee_code = extract_committee_from_table(content_div, "Committee responsible")
            if not committee_code:
                committee_code = "UNKNOWN"
        else:
            proc_html = ''
            committee_code = "UNKNOWN"

        # Lead roll call
        roll_html = ''
        roll_avail = False
        if lead_roll_section:
            roll_div = get_content_div(lead_roll_section['href'])
            if roll_div:
                roll_html = str(roll_div)
                roll_avail = True

        rows.append({
            'procedure': procedure_number,
            'committee': committee_code,
            'designation': 'lead',
            'procedure_available': bool(proc_html),
            'rollcall_available': roll_avail,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })
    else:
        # No lead procedure section found – maybe it's missing entirely
        pass

    # ----- Process opinion committees -----
    # Collect all opinion roll call divs (if any) in order
    opinion_roll_divs = []
    for sec in opinion_roll_sections:
        div = get_content_div(sec['href'])
        if div:
            # The div may contain multiple tables – we treat the whole div as the roll call content
            opinion_roll_divs.append(div)

    for idx, sec in enumerate(opinion_sections):
        content_div = get_content_div(sec['href'])
        if content_div:
            proc_html = str(content_div)
            proc_avail = True
            # Extract committee name from title
            raw_committee = extract_committee_from_opinion_title(sec['title'])
            committee = map_committee(raw_committee)
        else:
            proc_html = ''
            proc_avail = False
            committee = "UNKNOWN"

        # Does a corresponding roll call exist?
        roll_avail = idx < len(opinion_roll_divs)
        roll_html = str(opinion_roll_divs[idx]) if roll_avail else ''

        rows.append({
            'procedure': procedure_number,
            'committee': committee,
            'designation': 'opinion',
            'procedure_available': proc_avail,
            'rollcall_available': roll_avail,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })

    return pd.DataFrame(rows)

# ---------------------------
# Test the three URLs
# ---------------------------
urls = [
    ("https://www.europarl.europa.eu/doceo/document/A-8-2017-0375_EN.html", "2015/0287(COD)"),
    ("https://www.europarl.europa.eu/doceo/document/A-9-2022-0219_EN.html", "2021/0366(COD)"),
    ("https://www.europarl.europa.eu/doceo/document/A-7-2013-0315_EN.html", "2012/0364(COD)")
]

all_dfs = []
for url, proc_num in urls:
    print(f"Processing {proc_num} ...")
    df = extract_procedure_data(url, proc_num)
    # all_dfs.append(df)
    time.sleep(1)  # be polite

final_df = pd.concat(all_dfs, ignore_index=True)

# Show results
print("\nFinal DataFrame:")
display(final_df[['procedure', 'committee', 'designation', 'procedure_available', 'rollcall_available']])

# Optionally show first few rows with HTML snippets truncated
for col in ['procedure_string', 'rollcall_string']:
    final_df[col] = final_df[col].apply(lambda x: x[:200] + '...' if len(x) > 200 else x)
display(final_df.head(10))

Processing 2015/0287(COD) ...
Processing 2021/0366(COD) ...
Processing 2012/0364(COD) ...


ValueError: No objects to concatenate

In [42]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from IPython.display import display

HEADERS = {'User-Agent': 'Mozilla/5.0'}

def fetch_html(url):
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    return resp.text

def extract_committee_from_table(div, label="Committee responsible"):
    table = div.find('table')
    if not table:
        return None
    for row in table.find_all('tr'):
        cells = row.find_all('td')
        if len(cells) < 2:
            continue
        if label.lower() in cells[0].get_text(strip=True).lower():
            p = cells[1].find('p')
            return p.get_text(strip=True) if p else cells[1].get_text(strip=True)
    return None

def extract_committee_from_opinion_title(title):
    title_clean = re.sub(r'^OPINION\s+', '', title, flags=re.IGNORECASE).strip()
    match = re.search(r'on\s+(.+)', title_clean, re.IGNORECASE)
    return match.group(1).strip() if match else title_clean

COMMITTEE_MAP = {
    "Legal Affairs": "JURI",
    "International Trade": "INTA",
    "Civil Liberties, Justice and Home Affairs": "LIBE",
    "Development": "DEVE",
    "Agriculture and Rural Development": "AGRI",
    "Internal Market and Consumer Protection": "IMCO",
    "Environment, Public Health and Food Safety": "ENVI",
}
def map_committee(name):
    name_lower = name.lower()
    for key, code in COMMITTEE_MAP.items():
        if key.lower() in name_lower:
            return code
    return name

def extract_procedure_data(url, procedure_number):
    print(f"  Fetching {url} ...")
    html = fetch_html(url)
    soup = BeautifulSoup(html, 'html.parser')

    # ---- Check for placeholder (document not in HTML) ----
    placeholder = soup.select_one('div.card.mb-3 div.card-body p.bg-white.alert.alert-info.mb-3')
    if placeholder and "This document does not exist in HTML" in placeholder.get_text():
        print("  Document not in HTML format.")
        return pd.DataFrame([{
            'procedure': procedure_number,
            'committee': 'other format',
            'designation': 'other format',
            'procedure_available': False,
            'rollcall_available': False,
            'procedure_string': '',
            'rollcall_string': ''
        }])

    # ---- Find section links ----
    links_div = soup.find('div', class_='es_links-list')
    if not links_div:
        print("  No section links found – not a plenary document?")
        return pd.DataFrame(columns=['procedure','committee','designation',
                                     'procedure_available','rollcall_available',
                                     'procedure_string','rollcall_string'])

    sections = []
    for li in links_div.find_all('li'):
        a = li.find('a')
        if not a:
            continue
        href = a.get('href', '')
        span = a.find('span', class_='t-x')
        title = span.get_text(separator=' ', strip=True) if span else a.get_text(strip=True)
        sections.append({'href': href, 'title': title})
    print(f"  Found {len(sections)} sections.")

    # ---- Identify lead sections ----
    lead_proc = None
    lead_roll = None
    opinion_sections = []
    opinion_roll_sections = []

    for sec in sections:
        title_up = sec['title'].upper()
        if 'PROCEDURE' in title_up and 'OPINION' not in title_up:
            lead_proc = sec
        if 'FINAL VOTE BY ROLL CALL IN COMMITTEE RESPONSIBLE' in title_up:
            lead_roll = sec
        if 'OPINION' in title_up:
            opinion_sections.append(sec)
        if 'FINAL VOTE BY ROLL CALL IN COMMITTEE ASKED FOR OPINION' in title_up:
            opinion_roll_sections.append(sec)

    print(f"  Lead procedure section: {lead_proc['title'] if lead_proc else 'None'}")
    print(f"  Lead roll‑call section: {lead_roll['title'] if lead_roll else 'None'}")
    print(f"  Opinion sections: {len(opinion_sections)}")
    print(f"  Opinion roll‑call sections: {len(opinion_roll_sections)}")

    def get_content_div(href):
        if not href or not href.startswith('#'):
            return None
        section_id = href[1:]
        heading = soup.find(id=section_id)
        if heading:
            return heading.find_next('div')
        return None

    rows = []

    # ---- Lead committee ----
    if lead_proc:
        content = get_content_div(lead_proc['href'])
        proc_html = str(content) if content else ''
        committee = extract_committee_from_table(content, "Committee responsible") if content else None
        committee = committee if committee else "UNKNOWN"
        # lead roll‑call
        roll_html = ''
        roll_avail = False
        if lead_roll:
            roll_div = get_content_div(lead_roll['href'])
            if roll_div:
                roll_html = str(roll_div)
                roll_avail = True
        rows.append({
            'procedure': procedure_number,
            'committee': committee,
            'designation': 'lead',
            'procedure_available': bool(content),
            'rollcall_available': roll_avail,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })

    # ---- Opinion committees ----
    opinion_roll_divs = [get_content_div(sec['href']) for sec in opinion_roll_sections if get_content_div(sec['href'])]
    for idx, sec in enumerate(opinion_sections):
        content = get_content_div(sec['href'])
        proc_html = str(content) if content else ''
        proc_avail = bool(content)
        raw_committee = extract_committee_from_opinion_title(sec['title'])
        committee = map_committee(raw_committee)
        roll_avail = idx < len(opinion_roll_divs)
        roll_html = str(opinion_roll_divs[idx]) if roll_avail else ''
        rows.append({
            'procedure': procedure_number,
            'committee': committee,
            'designation': 'opinion',
            'procedure_available': proc_avail,
            'rollcall_available': roll_avail,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })

    df = pd.DataFrame(rows)
    print(f"  Extracted {len(df)} rows.")
    return df

# --------------------------------------------
# TEST ONE PROCEDURE (correct TA URL)
# --------------------------------------------
test_url = "https://www.europarl.europa.eu/doceo/document/A-9-2022-0219_EN.html"
test_proc = "2021/0366(COD)"

df_one = extract_procedure_data(test_url, test_proc)

print("\nResulting DataFrame:")
display(df_one[['procedure','committee','designation','procedure_available','rollcall_available']])

# If you want to see the first few characters of the HTML strings:
for col in ['procedure_string','rollcall_string']:
    if col in df_one.columns:
        df_one[col] = df_one[col].apply(lambda x: x[:200] + '...' if len(x) > 200 else x)
display(df_one)

  Fetching https://www.europarl.europa.eu/doceo/document/A-9-2022-0219_EN.html ...
  Found 2 sections.
  Lead procedure section: None
  Lead roll‑call section: None
  Opinion sections: 0
  Opinion roll‑call sections: 0
  Extracted 0 rows.

Resulting DataFrame:


KeyError: "None of [Index(['procedure', 'committee', 'designation', 'procedure_available',\n       'rollcall_available'],\n      dtype='str')] are in the [columns]"

In [43]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import time
from typing import List, Optional

HEADERS = {'User-Agent': 'Mozilla/5.0'}

# -------------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------------
def fetch_html(url: str) -> str:
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    return resp.text

def extract_committee_from_table(div, label="Committee responsible") -> Optional[str]:
    """Extract committee code from a procedure table."""
    table = div.find('table')
    if not table:
        return None
    for row in table.find_all('tr'):
        cells = row.find_all('td')
        if len(cells) < 2:
            continue
        if label.lower() in cells[0].get_text(strip=True).lower():
            p = cells[1].find('p')
            return p.get_text(strip=True) if p else cells[1].get_text(strip=True)
    return None

def extract_committee_from_opinion_title(title: str) -> str:
    """Extract committee name from an opinion section title."""
    title_clean = re.sub(r'^OPINION\s+', '', title, flags=re.IGNORECASE).strip()
    # Pattern: "of the Committee on ..." or "OF THE COMMITTEE ON ..."
    match = re.search(r'(?:of the )?Committee on\s+(.+)', title_clean, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return title_clean

# Mapping from committee names to standard codes (extend as needed)
COMMITTEE_MAP = {
    "Legal Affairs": "JURI",
    "International Trade": "INTA",
    "Civil Liberties, Justice and Home Affairs": "LIBE",
    "Development": "DEVE",
    "Agriculture and Rural Development": "AGRI",
    "Internal Market and Consumer Protection": "IMCO",
    "Environment, Public Health and Food Safety": "ENVI",
    "Environment, Climate and Food Safety": "ENVI",   # newer name
    "Budgets": "BUDG",
    "Foreign Affairs": "AFET",
    "Security and Defence": "SEDE",
    "Industry, Research and Energy": "ITRE",
    "Transport and Tourism": "TRAN",
    "Regional Development": "REGI",
    "Culture and Education": "CULT",
    "Constitutional Affairs": "AFCO",
    "Women’s Rights and Gender Equality": "FEMM",
    "Petitions": "PETI",
    "Human Rights": "DROI",
    "Fisheries": "PECH",
    "Economic and Monetary Affairs": "ECON",
    "Employment and Social Affairs": "EMPL",
}

def map_committee(name: str) -> str:
    """Convert a committee name (or partial name) to its standard code."""
    name_lower = name.lower()
    for key, code in COMMITTEE_MAP.items():
        if key.lower() in name_lower:
            return code
    return name   # fallback

# -------------------------------------------------------------------
# TA‑specific extraction (adopted texts with section links)
# -------------------------------------------------------------------
def extract_ta_document(url: str, procedure_number: str, soup: BeautifulSoup) -> pd.DataFrame:
    """Extract from TA documents that contain an es_links-list navigation."""
    links_div = soup.find('div', class_='es_links-list')
    if not links_div:
        # No links – maybe a different layout; return empty
        return pd.DataFrame(columns=['procedure','committee','designation',
                                     'procedure_available','rollcall_available',
                                     'procedure_string','rollcall_string'])

    sections = []
    for li in links_div.find_all('li'):
        a = li.find('a')
        if not a:
            continue
        href = a.get('href', '')
        span = a.find('span', class_='t-x')
        title = span.get_text(separator=' ', strip=True) if span else a.get_text(strip=True)
        sections.append({'href': href, 'title': title})

    # Identify lead and opinion sections
    lead_proc = None
    lead_roll = None
    opinion_sections = []
    opinion_roll_sections = []

    for sec in sections:
        title_up = sec['title'].upper()
        if 'PROCEDURE' in title_up and 'OPINION' not in title_up:
            lead_proc = sec
        if 'FINAL VOTE BY ROLL CALL IN COMMITTEE RESPONSIBLE' in title_up:
            lead_roll = sec
        if 'OPINION' in title_up:
            opinion_sections.append(sec)
        if 'FINAL VOTE BY ROLL CALL IN COMMITTEE ASKED FOR OPINION' in title_up:
            opinion_roll_sections.append(sec)

    def get_content_div(href):
        if not href or not href.startswith('#'):
            return None
        section_id = href[1:]
        heading = soup.find(id=section_id)
        if heading:
            return heading.find_next('div')
        return None

    rows = []

    # ---- Lead committee ----
    if lead_proc:
        content = get_content_div(lead_proc['href'])
        proc_html = str(content) if content else ''
        committee = extract_committee_from_table(content, "Committee responsible") if content else None
        committee = committee if committee else "UNKNOWN"
        # lead roll‑call
        roll_html = ''
        roll_avail = False
        if lead_roll:
            roll_div = get_content_div(lead_roll['href'])
            if roll_div:
                roll_html = str(roll_div)
                roll_avail = True
        rows.append({
            'procedure': procedure_number,
            'committee': committee,
            'designation': 'lead',
            'procedure_available': bool(content),
            'rollcall_available': roll_avail,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })

    # ---- Opinion committees ----
    opinion_roll_divs = [get_content_div(sec['href']) for sec in opinion_roll_sections if get_content_div(sec['href'])]
    for idx, sec in enumerate(opinion_sections):
        content = get_content_div(sec['href'])
        proc_html = str(content) if content else ''
        proc_avail = bool(content)
        raw_committee = extract_committee_from_opinion_title(sec['title'])
        committee = map_committee(raw_committee)
        roll_avail = idx < len(opinion_roll_divs)
        roll_html = str(opinion_roll_divs[idx]) if roll_avail else ''
        rows.append({
            'procedure': procedure_number,
            'committee': committee,
            'designation': 'opinion',
            'procedure_available': proc_avail,
            'rollcall_available': roll_avail,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })

    return pd.DataFrame(rows)

# -------------------------------------------------------------------
# A9‑specific extraction (committee reports without section links)
# -------------------------------------------------------------------
def extract_a9_document(url: str, procedure_number: str, soup: BeautifulSoup) -> pd.DataFrame:
    """Extract from A9 committee reports by parsing the header."""
    rows = []

    # ---- Lead committee ----
    # Usually appears as: "Committee on ..." followed by "Rapporteur: ..."
    lead_committee = None
    # Look for a line that starts with "Committee on" and contains no "Rapporteur"
    for elem in soup.find_all(['p', 'div']):
        text = elem.get_text(strip=True)
        if text.startswith("Committee on") and "Rapporteur" not in text:
            lead_committee = text
            break
    if not lead_committee:
        # Fallback: search for "Committee on" before the Rapporteur line
        rapporteur_elem = soup.find(string=re.compile(r'Rapporteur'))
        if rapporteur_elem:
            for elem in rapporteur_elem.find_all_previous(['p', 'div']):
                text = elem.get_text(strip=True)
                if text.startswith("Committee on"):
                    lead_committee = text
                    break

    if lead_committee:
        # Extract the part after "Committee on"
        match = re.search(r'Committee on\s+(.+)', lead_committee, re.IGNORECASE)
        if match:
            raw = match.group(1).strip()
            committee_code = map_committee(raw)
        else:
            committee_code = "UNKNOWN"
        rows.append({
            'procedure': procedure_number,
            'committee': committee_code,
            'designation': 'lead',
            'procedure_available': True,      # the report itself is the procedure
            'rollcall_available': False,      # no roll‑call tables in A9
            'procedure_string': '',           # no separate table
            'rollcall_string': ''
        })

    # ---- Opinion committees ----
    # Look for the line "Rapporteurs for the opinion (*):" followed by names and committees
    opinion_line = soup.find(string=re.compile(r'Rapporteurs? for the opinion', re.IGNORECASE))
    if opinion_line:
        line_text = opinion_line.strip()
        # Remove the label up to the colon
        if ':' in line_text:
            line_text = line_text.split(':', 1)[1].strip()
        # Split by semicolon to get individual rapporteur+committee pairs
        pairs = line_text.split(';')
        for pair in pairs:
            pair = pair.strip()
            if not pair:
                continue
            # Extract committee after "Committee on"
            match = re.search(r'Committee on\s+(.+?)(?:\s*$|;|,|\.)', pair, re.IGNORECASE)
            if match:
                raw_committee = match.group(1).strip()
                committee_code = map_committee(raw_committee)
                rows.append({
                    'procedure': procedure_number,
                    'committee': committee_code,
                    'designation': 'opinion',
                    'procedure_available': True,   # opinion is part of the report
                    'rollcall_available': False,
                    'procedure_string': '',
                    'rollcall_string': ''
                })
            else:
                # Sometimes only the rapporteur name is given; we can't extract committee
                pass

    return pd.DataFrame(rows)

# -------------------------------------------------------------------
# Main dispatcher function
# -------------------------------------------------------------------
def extract_procedure_data(url: str, procedure_number: str) -> pd.DataFrame:
    """
    Extract committee information from a Europarl document.
    Works for both A9 (committee report) and TA (adopted text) URLs.
    Returns a DataFrame with columns:
        procedure, committee, designation, procedure_available,
        rollcall_available, procedure_string, rollcall_string.
    """
    html = fetch_html(url)
    soup = BeautifulSoup(html, 'html.parser')

    # Check for placeholder (document not in HTML)
    placeholder = soup.select_one('div.card.mb-3 div.card-body p.bg-white.alert.alert-info.mb-3')
    if placeholder and "This document does not exist in HTML" in placeholder.get_text():
        return pd.DataFrame([{
            'procedure': procedure_number,
            'committee': 'other format',
            'designation': 'other format',
            'procedure_available': False,
            'rollcall_available': False,
            'procedure_string': '',
            'rollcall_string': ''
        }])

    # Detect document type by URL
    if '/TA-' in url:
        return extract_ta_document(url, procedure_number, soup)
    else:
        return extract_a9_document(url, procedure_number, soup)

# -------------------------------------------------------------------
# Test block (when run directly)
# -------------------------------------------------------------------
if __name__ == "__main__":
    # Test with one A9 and one TA document
    test_cases = [
        ("https://www.europarl.europa.eu/doceo/document/A-9-2022-0219_EN.html", "2021/0366(COD)"),
        ("https://www.europarl.europa.eu/doceo/document/TA-9-2022-0219_EN.html", "2021/0366(COD)"),
    ]

    for url, proc in test_cases:
        print(f"\n--- Testing {url} ---")
        df = extract_procedure_data(url, proc)
        print(df[['procedure','committee','designation','procedure_available','rollcall_available']])
        # Optional: show first 200 chars of HTML strings
        for col in ['procedure_string','rollcall_string']:
            if col in df.columns:
                df[col] = df[col].apply(lambda x: x[:200] + '...' if len(x) > 200 else x)
        print(df.head())


--- Testing https://www.europarl.europa.eu/doceo/document/A-9-2022-0219_EN.html ---
        procedure committee designation  procedure_available  \
0  2021/0366(COD)      INTA        lead                 True   

   rollcall_available  
0               False  
        procedure committee designation  procedure_available  \
0  2021/0366(COD)      INTA        lead                 True   

   rollcall_available procedure_string rollcall_string  
0               False                                   

--- Testing https://www.europarl.europa.eu/doceo/document/TA-9-2022-0219_EN.html ---
Empty DataFrame
Columns: [procedure, committee, designation, procedure_available, rollcall_available]
Index: []
Empty DataFrame
Columns: [procedure, committee, designation, procedure_available, rollcall_available, procedure_string, rollcall_string]
Index: []


In [46]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from typing import List, Optional, Union, Tuple

# ---------- existing functions (unchanged, kept for reference) ----------
def fetch_html(url: str) -> str:
    response = requests.get(url)
    response.raise_for_status()
    return response.text

def find_sections_by_header(soup: BeautifulSoup, header_text: str) -> List[BeautifulSoup]:
    sections = []
    pattern = re.compile(re.escape(header_text), re.IGNORECASE)
    for elem in soup.find_all(string=pattern):
        parent = elem.parent
        while parent and parent.name not in ['h1','h2','h3','h4','h5','h6','p','div','section']:
            parent = parent.parent
        if not parent:
            parent = elem.parent
        next_div = parent.find_next('div', class_='table-responsive')
        if next_div and next_div not in sections:
            sections.append(next_div)
    return sections

def extract_committee_from_table(procedure_div: BeautifulSoup, label_texts: Union[str, List[str]]) -> Optional[str]:
    if isinstance(label_texts, str):
        label_texts = [label_texts]
    table = procedure_div.find('table')
    if not table:
        return None
    rows = table.find_all('tr')
    for row in rows:
        cells = row.find_all('td')
        if len(cells) < 2:
            continue
        label = cells[0].get_text(strip=True)
        if any(lt.lower() in label.lower() for lt in label_texts):
            p = cells[1].find('p')
            if p:
                return p.get_text(strip=True)
            return cells[1].get_text(strip=True)
    return None

def count_rollcall_sections(soup: BeautifulSoup, header_text: str) -> int:
    pattern = re.compile(re.escape(header_text), re.IGNORECASE)
    return len(list(soup.find_all(string=pattern)))

# ---------- new helper functions ----------
def find_all_divs_after_header(soup: BeautifulSoup, header_text: str, class_name: str = 'table-responsive') -> List[BeautifulSoup]:
    """
    Find all divs with class `class_name` that appear after an element containing header_text.
    Returns a list of those divs.
    """
    divs = []
    pattern = re.compile(re.escape(header_text), re.IGNORECASE)
    for elem in soup.find_all(string=pattern):
        parent = elem.parent
        while parent and parent.name not in ['h1','h2','h3','h4','h5','h6','p','div','section']:
            parent = parent.parent
        if not parent:
            parent = elem.parent
        # find all following divs with the given class
        for sibling in parent.find_all_next('div', class_=class_name):
            # stop if we hit another header? For simplicity we collect all until end of section.
            # But the page structure is consistent: all such divs under the same heading belong to it.
            if sibling not in divs:
                divs.append(sibling)
    return divs

def extract_committees_from_table(procedure_div: BeautifulSoup, label_texts: Union[str, List[str]]) -> List[str]:
    """
    Extract all committee codes from a procedure table.
    Finds the row whose first cell contains any of the label_texts (case‑insensitive),
    then collects the text from all subsequent cells (or their first paragraphs).
    Returns a list of committee codes.
    """
    if isinstance(label_texts, str):
        label_texts = [label_texts]
    table = procedure_div.find('table')
    if not table:
        return []
    rows = table.find_all('tr')
    for row in rows:
        cells = row.find_all('td')
        if len(cells) < 2:
            continue
        label = cells[0].get_text(strip=True)
        if any(lt.lower() in label.lower() for lt in label_texts):
            committees = []
            # start from cell 1 (index 1) because cell 0 is the label
            for cell in cells[1:]:
                # committee code is usually in the first paragraph, otherwise whole cell text
                p = cell.find('p')
                text = p.get_text(strip=True) if p else cell.get_text(strip=True)
                if text:  # ignore empty cells
                    committees.append(text)
            return committees
    return []

# ---------- new main function ----------
def get_committee_votes_with_html(url: str, procedure_number: str) -> pd.DataFrame:
    """
    Fetch the HTML from the given Europarl document URL and extract
    information about responsible and opinion committees, including the
    full HTML of the procedure and roll‑call tables.

    Parameters
    ----------
    url : str
        The URL of the document (e.g. https://www.europarl.europa.eu/doceo/document/...)
    procedure_number : str
        The procedure identifier, e.g. '2021/0366(COD)'

    Returns
    -------
    pd.DataFrame
        Columns: procedure, committee, designation, procedure_available,
                 rollcall_available, procedure_string, rollcall_string
    """
    html = fetch_html(url)
    soup = BeautifulSoup(html, 'html.parser')

    # ----- Check for "document not in HTML" placeholder -----
    placeholder = soup.select_one('div.card.mb-3 div.card-body p.bg-white.alert.alert-info.mb-3')
    if placeholder and "This document does not exist in HTML but is available in the following formats:" in placeholder.get_text():
        return pd.DataFrame([{
            'procedure': procedure_number,
            'committee': 'other format',
            'designation': 'other format',
            'procedure_available': False,
            'rollcall_available': False,
            'procedure_string': '',
            'rollcall_string': ''
        }])

    # ---- Collect responsible committee and its procedure table ----
    resp_procedure_div = None
    resp_committee = None

    # Try specific header first
    resp_sections = find_sections_by_header(soup, "PROCEDURE – COMMITTEE RESPONSIBLE")
    for div in resp_sections:
        code = extract_committee_from_table(div, "Committee responsible")
        if code:
            resp_committee = code
            resp_procedure_div = div
            break

    # If not found, try generic "PROCEDURE"
    if not resp_committee:
        generic_sections = find_sections_by_header(soup, "PROCEDURE")
        for div in generic_sections:
            code = extract_committee_from_table(div, "Committee responsible")
            if code:
                resp_committee = code
                resp_procedure_div = div
                break

    # ---- Collect responsible roll‑call tables ----
    resp_rollcall_divs = find_all_divs_after_header(soup, "FINAL VOTE BY ROLL CALL IN COMMITTEE RESPONSIBLE")
    resp_rollcall_exists = len(resp_rollcall_divs) > 0

    # ---- Collect opinion committees with their source tables ----
    # We'll store tuples (committee_code, procedure_div)
    opinion_entries = []

    # 1. Sections with header "PROCEDURE – COMMITTEE ASKED FOR OPINION"
    opinion_sections = find_sections_by_header(soup, "PROCEDURE – COMMITTEE ASKED FOR OPINION")
    for div in opinion_sections:
        codes = extract_committees_from_table(div, ["Opinion by", "Committee(s) asked for opinion(s)"])
        for code in codes:
            opinion_entries.append((code, div))

    # 2. Sections with header "OPINION"
    if not opinion_entries:
        opinion_headers_sections = find_sections_by_header(soup, "OPINION")
        for div in opinion_headers_sections:
            codes = extract_committees_from_table(div, ["Opinion by", "Committee(s) asked for opinion(s)"])
            for code in codes:
                opinion_entries.append((code, div))

    # 3. Generic "PROCEDURE" sections (only if we haven't found any yet, and avoid duplicates)
    if not opinion_entries:
        generic_sections = find_sections_by_header(soup, "PROCEDURE")
        existing = set()
        for div in generic_sections:
            codes = extract_committees_from_table(div, ["Opinion by", "Committee(s) asked for opinion(s)"])
            for code in codes:
                if code.upper() not in existing:
                    opinion_entries.append((code, div))
                    existing.add(code.upper())

    # ---- Collect opinion roll‑call tables ----
    opinion_rollcall_divs = find_all_divs_after_header(soup, "FINAL VOTE BY ROLL CALL IN COMMITTEE ASKED FOR OPINION")
    opinion_rollcall_count = len(opinion_rollcall_divs)

    # ---- Build rows ----
    rows = []

    # Responsible committee row
    if resp_committee:
        proc_html = str(resp_procedure_div) if resp_procedure_div else ''
        roll_html = ''.join(str(div) for div in resp_rollcall_divs) if resp_rollcall_exists else ''
        rows.append({
            'procedure': procedure_number,
            'committee': resp_committee,
            'designation': 'lead',
            'procedure_available': resp_procedure_div is not None,
            'rollcall_available': resp_rollcall_exists,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })
    elif resp_sections:  # section exists but no committee code found
        proc_html = str(resp_sections[0]) if resp_sections else ''
        roll_html = ''.join(str(div) for div in resp_rollcall_divs) if resp_rollcall_exists else ''
        rows.append({
            'procedure': procedure_number,
            'committee': 'UNKNOWN',
            'designation': 'lead',
            'procedure_available': True,
            'rollcall_available': resp_rollcall_exists,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })

    # Opinion committee rows
    for i, (comm, proc_div) in enumerate(opinion_entries):
        proc_html = str(proc_div) if proc_div else ''
        # rollcall available only for the first `opinion_rollcall_count` committees
        roll_avail = i < opinion_rollcall_count
        roll_html = ''.join(str(div) for div in opinion_rollcall_divs) if roll_avail else ''
        rows.append({
            'procedure': procedure_number,
            'committee': comm,
            'designation': 'opinion',
            'procedure_available': True,  # we have a procedure div
            'rollcall_available': roll_avail,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })

    return pd.DataFrame(rows)

import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

import pandas as pd
from functions.create_com_vote_info import get_committee_votes_with_html   # updated import

# Determine the project root (parent of the 'scripts' folder)
script_dir = os.path.dirname(os.path.abspath(__file__))
project_root = os.path.dirname(script_dir)  # go up one level
BLD = os.path.join(project_root, 'bld')

def main():
    # Path to the input CSV
    input_csv = os.path.join(BLD, 'test_procedures_timeline.csv')
    
    # Read the timeline data
    df_timeline = pd.read_csv(input_csv)
    
    # Filter rows where event is "Committee report tabled for plenary, 1st reading"
    mask = df_timeline['event'] == "Committee report tabled for plenary, 1st reading"
    df_filtered = df_timeline[mask].copy()
    
    if df_filtered.empty:
        print("No matching events found.")
        return
    
    # Prepare list to collect DataFrames
    all_dfs = []
    
    # Iterate over each filtered row
    for idx, row in df_filtered.iterrows():
        procedure = row['procedure']
        hyperlink = row['hyperlink']
        
        # Skip if hyperlink is missing or NaN
        if pd.isna(hyperlink) or not hyperlink:
            print(f"Skipping procedure {procedure}: no hyperlink")
            continue
        
        print(f"Processing {procedure} -> {hyperlink}")
        
        try:
            # Use the new function that returns additional HTML columns
            df_committees = get_committee_votes_with_html(hyperlink, procedure)
            all_dfs.append(df_committees)
        except Exception as e:
            print(f"Error processing {hyperlink}: {e}")
            continue
    
    if not all_dfs:
        print("No data collected.")
        return
    
    # Concatenate all results
    df_final = pd.concat(all_dfs, ignore_index=True)
    
    # Reorder columns to put 'procedure' first, then the rest
    cols = ['procedure'] + [c for c in df_final.columns if c != 'procedure']
    df_final = df_final[cols]
    
    # # Save output
    # output_csv = os.path.join(BLD, 'test_procedures_committee_vote_info.csv')
    # df_final.to_csv(output_csv, index=False)
    print(f"Saved {len(df_final)} rows to {output_csv}")
    return df_final  # return the final DataFrame for further use if needed

if __name__ == "__main__":
    main()

NameError: name '__file__' is not defined

In [48]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from typing import List, Optional, Union, Tuple

# ---------- existing functions (unchanged, kept for reference) ----------
def fetch_html(url: str) -> str:
    response = requests.get(url)
    response.raise_for_status()
    return response.text

def find_sections_by_header(soup: BeautifulSoup, header_text: str) -> List[BeautifulSoup]:
    sections = []
    pattern = re.compile(re.escape(header_text), re.IGNORECASE)
    for elem in soup.find_all(string=pattern):
        parent = elem.parent
        while parent and parent.name not in ['h1','h2','h3','h4','h5','h6','p','div','section']:
            parent = parent.parent
        if not parent:
            parent = elem.parent
        next_div = parent.find_next('div', class_='table-responsive')
        if next_div and next_div not in sections:
            sections.append(next_div)
    return sections

def extract_committee_from_table(procedure_div: BeautifulSoup, label_texts: Union[str, List[str]]) -> Optional[str]:
    if isinstance(label_texts, str):
        label_texts = [label_texts]
    table = procedure_div.find('table')
    if not table:
        return None
    rows = table.find_all('tr')
    for row in rows:
        cells = row.find_all('td')
        if len(cells) < 2:
            continue
        label = cells[0].get_text(strip=True)
        if any(lt.lower() in label.lower() for lt in label_texts):
            p = cells[1].find('p')
            if p:
                return p.get_text(strip=True)
            return cells[1].get_text(strip=True)
    return None

def count_rollcall_sections(soup: BeautifulSoup, header_text: str) -> int:
    pattern = re.compile(re.escape(header_text), re.IGNORECASE)
    return len(list(soup.find_all(string=pattern)))

# ---------- new helper functions ----------
def find_all_divs_after_header(soup: BeautifulSoup, header_text: str, class_name: str = 'table-responsive') -> List[BeautifulSoup]:
    """
    Find all divs with class `class_name` that appear after an element containing header_text.
    Returns a list of those divs.
    """
    divs = []
    pattern = re.compile(re.escape(header_text), re.IGNORECASE)
    for elem in soup.find_all(string=pattern):
        parent = elem.parent
        while parent and parent.name not in ['h1','h2','h3','h4','h5','h6','p','div','section']:
            parent = parent.parent
        if not parent:
            parent = elem.parent
        # find all following divs with the given class
        for sibling in parent.find_all_next('div', class_=class_name):
            # stop if we hit another header? For simplicity we collect all until end of section.
            # But the page structure is consistent: all such divs under the same heading belong to it.
            if sibling not in divs:
                divs.append(sibling)
    return divs

def extract_committees_from_table(procedure_div: BeautifulSoup, label_texts: Union[str, List[str]]) -> List[str]:
    """
    Extract all committee codes from a procedure table.
    Finds the row whose first cell contains any of the label_texts (case‑insensitive),
    then collects the text from all subsequent cells (or their first paragraphs).
    Returns a list of committee codes.
    """
    if isinstance(label_texts, str):
        label_texts = [label_texts]
    table = procedure_div.find('table')
    if not table:
        return []
    rows = table.find_all('tr')
    for row in rows:
        cells = row.find_all('td')
        if len(cells) < 2:
            continue
        label = cells[0].get_text(strip=True)
        if any(lt.lower() in label.lower() for lt in label_texts):
            committees = []
            # start from cell 1 (index 1) because cell 0 is the label
            for cell in cells[1:]:
                # committee code is usually in the first paragraph, otherwise whole cell text
                p = cell.find('p')
                text = p.get_text(strip=True) if p else cell.get_text(strip=True)
                if text:  # ignore empty cells
                    committees.append(text)
            return committees
    return []

# ---------- new main function ----------
def get_committee_votes_with_html(url: str, procedure_number: str) -> pd.DataFrame:
    """
    Fetch the HTML from the given Europarl document URL and extract
    information about responsible and opinion committees, including the
    full HTML of the procedure and roll‑call tables.

    Parameters
    ----------
    url : str
        The URL of the document (e.g. https://www.europarl.europa.eu/doceo/document/...)
    procedure_number : str
        The procedure identifier, e.g. '2021/0366(COD)'

    Returns
    -------
    pd.DataFrame
        Columns: procedure, committee, designation, procedure_available,
                 rollcall_available, procedure_string, rollcall_string
    """
    html = fetch_html(url)
    soup = BeautifulSoup(html, 'html.parser')

    # ----- Check for "document not in HTML" placeholder -----
    placeholder = soup.select_one('div.card.mb-3 div.card-body p.bg-white.alert.alert-info.mb-3')
    if placeholder and "This document does not exist in HTML but is available in the following formats:" in placeholder.get_text():
        return pd.DataFrame([{
            'procedure': procedure_number,
            'committee': 'other format',
            'designation': 'other format',
            'procedure_available': False,
            'rollcall_available': False,
            'procedure_string': '',
            'rollcall_string': ''
        }])

    # ---- Collect responsible committee and its procedure table ----
    resp_procedure_div = None
    resp_committee = None

    # Try specific header first
    resp_sections = find_sections_by_header(soup, "PROCEDURE – COMMITTEE RESPONSIBLE")
    for div in resp_sections:
        code = extract_committee_from_table(div, "Committee responsible")
        if code:
            resp_committee = code
            resp_procedure_div = div
            break

    # If not found, try generic "PROCEDURE"
    if not resp_committee:
        generic_sections = find_sections_by_header(soup, "PROCEDURE")
        for div in generic_sections:
            code = extract_committee_from_table(div, "Committee responsible")
            if code:
                resp_committee = code
                resp_procedure_div = div
                break

    # ---- Collect responsible roll‑call tables ----
    resp_rollcall_divs = find_all_divs_after_header(soup, "FINAL VOTE BY ROLL CALL IN COMMITTEE RESPONSIBLE")
    resp_rollcall_exists = len(resp_rollcall_divs) > 0

    # ---- Collect opinion committees with their source tables ----
    # We'll store tuples (committee_code, procedure_div)
    opinion_entries = []

    # 1. Sections with header "PROCEDURE – COMMITTEE ASKED FOR OPINION"
    opinion_sections = find_sections_by_header(soup, "PROCEDURE – COMMITTEE ASKED FOR OPINION")
    for div in opinion_sections:
        codes = extract_committees_from_table(div, ["Opinion by", "Committee(s) asked for opinion(s)"])
        for code in codes:
            opinion_entries.append((code, div))

    # 2. Sections with header "OPINION"
    if not opinion_entries:
        opinion_headers_sections = find_sections_by_header(soup, "OPINION")
        for div in opinion_headers_sections:
            codes = extract_committees_from_table(div, ["Opinion by", "Committee(s) asked for opinion(s)"])
            for code in codes:
                opinion_entries.append((code, div))

    # 3. Generic "PROCEDURE" sections (only if we haven't found any yet, and avoid duplicates)
    if not opinion_entries:
        generic_sections = find_sections_by_header(soup, "PROCEDURE")
        existing = set()
        for div in generic_sections:
            codes = extract_committees_from_table(div, ["Opinion by", "Committee(s) asked for opinion(s)"])
            for code in codes:
                if code.upper() not in existing:
                    opinion_entries.append((code, div))
                    existing.add(code.upper())

    # ---- Collect opinion roll‑call tables ----
    opinion_rollcall_divs = find_all_divs_after_header(soup, "FINAL VOTE BY ROLL CALL IN COMMITTEE ASKED FOR OPINION")
    opinion_rollcall_count = len(opinion_rollcall_divs)

    # ---- Build rows ----
    rows = []

    # Responsible committee row
    if resp_committee:
        proc_html = str(resp_procedure_div) if resp_procedure_div else ''
        roll_html = ''.join(str(div) for div in resp_rollcall_divs) if resp_rollcall_exists else ''
        rows.append({
            'procedure': procedure_number,
            'committee': resp_committee,
            'designation': 'lead',
            'procedure_available': resp_procedure_div is not None,
            'rollcall_available': resp_rollcall_exists,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })
    elif resp_sections:  # section exists but no committee code found
        proc_html = str(resp_sections[0]) if resp_sections else ''
        roll_html = ''.join(str(div) for div in resp_rollcall_divs) if resp_rollcall_exists else ''
        rows.append({
            'procedure': procedure_number,
            'committee': 'UNKNOWN',
            'designation': 'lead',
            'procedure_available': True,
            'rollcall_available': resp_rollcall_exists,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })

    # Opinion committee rows
    for i, (comm, proc_div) in enumerate(opinion_entries):
        proc_html = str(proc_div) if proc_div else ''
        # rollcall available only for the first `opinion_rollcall_count` committees
        roll_avail = i < opinion_rollcall_count
        roll_html = ''.join(str(div) for div in opinion_rollcall_divs) if roll_avail else ''
        rows.append({
            'procedure': procedure_number,
            'committee': comm,
            'designation': 'opinion',
            'procedure_available': True,  # we have a procedure div
            'rollcall_available': roll_avail,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })

    return pd.DataFrame(rows)

# import sys
# import os
# sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

# import pandas as pd
# from functions.create_com_vote_info import get_committee_votes_with_html   # updated import

# # Determine the project root (parent of the 'scripts' folder)
# script_dir = os.path.dirname(os.path.abspath(__file__))
# project_root = os.path.dirname(script_dir)  # go up one level
# BLD = os.path.join(project_root, 'bld')

def main():
    # Path to the input CSV
    input_csv = os.path.join(BLD, 'test_procedures_timeline.csv')
    
    # Read the timeline data
    df_timeline = pd.read_csv(input_csv)
    
    # Filter rows where event is "Committee report tabled for plenary, 1st reading"
    mask = df_timeline['event'] == "Committee report tabled for plenary, 1st reading"
    df_filtered = df_timeline[mask].copy()
    
    if df_filtered.empty:
        print("No matching events found.")
        return
    
    # Prepare list to collect DataFrames
    all_dfs = []
    
    # Iterate over each filtered row
    for idx, row in df_filtered.iterrows():
        procedure = row['procedure']
        hyperlink = row['hyperlink']
        
        # Skip if hyperlink is missing or NaN
        if pd.isna(hyperlink) or not hyperlink:
            print(f"Skipping procedure {procedure}: no hyperlink")
            continue
        
        print(f"Processing {procedure} -> {hyperlink}")
        
        try:
            # Use the new function that returns additional HTML columns
            df_committees = get_committee_votes_with_html(hyperlink, procedure)
            all_dfs.append(df_committees)
        except Exception as e:
            print(f"Error processing {hyperlink}: {e}")
            continue
    
    if not all_dfs:
        print("No data collected.")
        return
    
    # Concatenate all results
    df_final = pd.concat(all_dfs, ignore_index=True)
    
    # Reorder columns to put 'procedure' first, then the rest
    cols = ['procedure'] + [c for c in df_final.columns if c != 'procedure']
    df_final = df_final[cols]
    
    # # Save output
    # output_csv = os.path.join(BLD, 'test_procedures_committee_vote_info.csv')
    # df_final.to_csv(output_csv, index=False)
    print(f"Saved {len(df_final)} rows to {output_csv}")
    return df_final  # return the final DataFrame for further use if needed

if __name__ == "__main__":
    main()

NameError: name 'BLD' is not defined

In [49]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from typing import List, Dict, Any

# ---------- Configuration ----------
CSV_PATH = r'C:\Users\User\eu_legislation\scrapper\bld\test_procedures_timeline.csv'
# Regex for reference starting with A + digit (e.g. A9‑0123/2021)
REF_PATTERN = re.compile(r'^A\d')

# ---------- Helper functions ----------
def fetch_html(url: str) -> str:
    """Fetch HTML from a URL with error handling."""
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        return resp.text
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return ""

def extract_committee_name_from_opinion_button(text: str) -> str:
    """
    Extract the committee name from an opinion button text.
    Examples:
        "OPINION OF THE COMMITTEE ON INTERNATIONAL TRADE" -> "COMMITTEE ON INTERNATIONAL TRADE"
        "OPINION of the Committee on Civil Liberties, Justice and Home Affairs" -> "Committee on Civil Liberties, Justice and Home Affairs"
    """
    # Remove leading "OPINION" and any "OF THE", "BY THE" (case‑insensitive)
    match = re.search(r'OPINION\s+(?:OF\s+THE\s+)?(?:BY\s+THE\s+)?(.*)', text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    # Fallback: just remove the first word if it is "OPINION"
    words = text.split()
    if words and words[0].upper() == 'OPINION':
        return ' '.join(words[1:]).strip()
    return text.strip()

def parse_page(html: str, procedure: str, event: str, reference: str, hyperlink: str) -> List[Dict[str, Any]]:
    """
    Parse a single document page and return a list of rows (one per committee found).
    """
    soup = BeautifulSoup(html, 'html.parser')
    rows = []

    # ----- Check for "other format" placeholder -----
    other_format = False
    placeholder = soup.find('p', class_='bg-white alert alert-info mb-3')
    if placeholder and "This document does not exist in HTML" in placeholder.get_text():
        other_format = True
        rows.append({
            'procedure': procedure,
            'event': event,
            'reference': reference,
            'hyperlink': hyperlink,
            'committee_name': None,
            'committee_type': None,
            'lead_procedure_exists': False,
            'lead_roll_call_exists': False,
            'opinion_procedure_exists': False,
            'opinion_roll_call_exists': False,
            'other_format': True
        })
        return rows

    # ----- Initialise flags -----
    lead_proc = False
    lead_roll = False
    opinion_proc = False
    opinion_roll = False
    opinion_committees = []  # store extracted committee names

    # ----- Parse the button list (table of contents) -----
    button_list = soup.find('div', class_='es_links-list mb-3')
    if button_list:
        for li in button_list.find_all('li', recursive=False):
            a = li.find('a')
            if not a:
                continue
            # The text is usually inside a <span class="t-x">
            span = a.find('span', class_='t-x')
            text = span.get_text(strip=True) if span else a.get_text(strip=True)
            text_upper = text.upper()

            # Detect buttons
            if 'PROCEDURE' in text_upper:
                lead_proc = True
            if 'FINAL VOTE' in text_upper and 'COMMITTEE RESPONSIBLE' in text_upper:
                lead_roll = True
            if 'FINAL VOTE' in text_upper and 'OPINION' in text_upper:
                opinion_roll = True

            # Opinion committee button (exclude those that are purely roll‑call)
            if 'OPINION' in text_upper and 'FINAL VOTE' not in text_upper:
                opinion_proc = True
                comm_name = extract_committee_name_from_opinion_button(text)
                if comm_name:
                    opinion_committees.append(comm_name)

    # ----- Extract lead committee(s) from the PROCEDURE table -----
    lead_committees = []
    if lead_proc:
        # Find the first header containing "PROCEDURE" (case‑insensitive)
        header = soup.find('h2', string=re.compile(r'PROCEDURE', re.IGNORECASE))
        if header:
            # The table is usually inside a <div class="table-responsive"> following the header
            table_div = header.find_next('div', class_='table-responsive')
            if table_div:
                table = table_div.find('table')
            else:
                table = header.find_next('table')
            if table:
                for row in table.find_all('tr'):
                    cells = row.find_all('td')
                    if len(cells) < 2:
                        continue
                    label = cells[0].get_text(strip=True).lower()
                    if 'committee responsible' in label:
                        # Collect text from all following cells
                        for cell in cells[1:]:
                            p = cell.find('p')
                            comm_text = p.get_text(strip=True) if p else cell.get_text(strip=True)
                            if comm_text and comm_text.strip():
                                lead_committees.append(comm_text.strip())
                        break  # assume only one such row

    # ----- Build rows -----
    # Deduplicate opinion committees (simple string equality)
    seen_opinion = set()
    unique_opinion = []
    for comm in opinion_committees:
        if comm not in seen_opinion:
            seen_opinion.add(comm)
            unique_opinion.append(comm)

    # Lead committee rows
    for comm in lead_committees:
        rows.append({
            'procedure': procedure,
            'event': event,
            'reference': reference,
            'hyperlink': hyperlink,
            'committee_name': comm,
            'committee_type': 'lead',
            'lead_procedure_exists': lead_proc,
            'lead_roll_call_exists': lead_roll,
            'opinion_procedure_exists': opinion_proc,
            'opinion_roll_call_exists': opinion_roll,
            'other_format': False
        })

    # Opinion committee rows
    for comm in unique_opinion:
        rows.append({
            'procedure': procedure,
            'event': event,
            'reference': reference,
            'hyperlink': hyperlink,
            'committee_name': comm,
            'committee_type': 'opinion',
            'lead_procedure_exists': lead_proc,
            'lead_roll_call_exists': lead_roll,
            'opinion_procedure_exists': opinion_proc,
            'opinion_roll_call_exists': opinion_roll,
            'other_format': False
        })

    # If no committees at all, still add a row to preserve the flags
    if not lead_committees and not unique_opinion:
        rows.append({
            'procedure': procedure,
            'event': event,
            'reference': reference,
            'hyperlink': hyperlink,
            'committee_name': None,
            'committee_type': None,
            'lead_procedure_exists': lead_proc,
            'lead_roll_call_exists': lead_roll,
            'opinion_procedure_exists': opinion_proc,
            'opinion_roll_call_exists': opinion_roll,
            'other_format': False
        })

    return rows

# ---------- Main processing ----------
# 1. Load the CSV
df = pd.read_csv(CSV_PATH)

# 2. Filter rows where reference starts with A + digit
mask = df['reference'].astype(str).str.match(REF_PATTERN)
filtered = df[mask].copy()
print(f"Found {len(filtered)} rows to process.")

# 3. Process each row (with a simple cache to avoid repeated fetches of the same URL)
cache = {}
all_rows = []

for idx, row in filtered.iterrows():
    proc = row['procedure']
    event = row['event']
    ref = row['reference']
    url = row['hyperlink']

    # Use cached HTML if available
    if url not in cache:
        print(f"Fetching: {url}")
        cache[url] = fetch_html(url)
    html = cache[url]

    if not html:
        print(f"  -> Failed to retrieve, skipping.")
        continue

    # Parse the page and collect rows
    page_rows = parse_page(html, proc, event, ref, url)
    all_rows.extend(page_rows)

# 4. Create the final DataFrame
df_info = pd.DataFrame(all_rows)

# (Optional) reorder columns for clarity
column_order = [
    'procedure', 'event', 'reference', 'hyperlink',
    'committee_name', 'committee_type',
    'lead_procedure_exists', 'lead_roll_call_exists',
    'opinion_procedure_exists', 'opinion_roll_call_exists',
    'other_format'
]
df_info = df_info[column_order]

print("\nDone. Result shape:", df_info.shape)
df_info.head()

Found 54 rows to process.
Fetching: https://www.europarl.europa.eu/doceo/document/A-9-2022-0219_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-8-2015-0029_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-7-2013-0315_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-7-2012-0043_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-7-2011-0339_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-6-2008-0329_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-5-2000-0309_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-5-2001-0133_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-5-2001-0313_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-4-1995-0221_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-4-1996-0194_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-7-2012-0273_EN.html
Fetching: https://www.europarl.europa.

,procedure,event,reference,hyperlink,committee_name,committee_type,lead_procedure_exists,lead_roll_call_exists,opinion_procedure_exists,opinion_roll_call_exists,other_format
0,2021/0366(COD),"Committee report tabled for plenary, 1st readi...",A9-0219/2022,https://www.europarl.europa.eu/doceo/document/...,None,None,False,False,False,False,False
1,2021/0366(COD),"Committee report tabled for plenary, 1st reading",A9-0219/2022,https://www.europarl.europa.eu/doceo/document/...,None,None,False,False,False,False,False
2,2014/0011(COD),"Committee report tabled for plenary, 1st readi...",A8-0029/2015,https://www.europarl.europa.eu/doceo/document/...,None,None,False,False,False,False,False
3,2014/0011(COD),"Committee report tabled for plenary, 1st reading",A8-0029/2015,https://www.europarl.europa.eu/doceo/document/...,None,None,False,False,False,False,False
4,2012/0364(COD),"Committee report tabled for plenary, 1st reading",A7-0315/2013,https://www.europarl.europa.eu/doceo/document/...,None,None,False,False,False,False,False


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from typing import List, Dict, Any
import time

# ---------- Configuration ----------
CSV_PATH = r'C:\Users\User\eu_legislation\scrapper\bld\test_procedures_timeline.csv'
REF_PATTERN = re.compile(r'^A\d')  # references starting with A + digit

# ---------- Helper functions ----------
def fetch_html(url: str, retries=2) -> str:
    """Fetch HTML from a URL with error handling and retries."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=15)
            resp.raise_for_status()
            # Check if it's actually HTML (not PDF)
            content_type = resp.headers.get('Content-Type', '')
            if 'application/pdf' in content_type or url.lower().endswith('.pdf'):
                # Try HTML version by replacing .pdf with .html
                if url.lower().endswith('.pdf'):
                    html_url = url[:-4] + '.html'
                    print(f"  PDF detected, trying HTML: {html_url}")
                    return fetch_html(html_url, retries-1)
                else:
                    return ""
            return resp.text
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(2)
    return ""

def extract_committee_name_from_opinion_text(text: str) -> str:
    """
    Extract committee name from text like:
    "OPINION of the Committee on Civil Liberties, Justice and Home Affairs" -> "Committee on Civil Liberties, Justice and Home Affairs"
    "OPINION OF THE COMMITTEE ON INTERNATIONAL TRADE" -> "COMMITTEE ON INTERNATIONAL TRADE"
    """
    # Remove leading "OPINION" and any "OF THE", "BY THE" (case‑insensitive)
    match = re.search(r'OPINION\s+(?:OF\s+THE\s+)?(?:BY\s+THE\s+)?(.*)', text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    # Fallback: just remove the first word if it is "OPINION"
    words = text.split()
    if words and words[0].upper() == 'OPINION':
        return ' '.join(words[1:]).strip()
    return text.strip()

def parse_page(html: str, procedure: str, event: str, reference: str, hyperlink: str) -> List[Dict[str, Any]]:
    """
    Parse a single document page and return a list of rows (one per committee found).
    """
    if not html:
        return [{
            'procedure': procedure,
            'event': event,
            'reference': reference,
            'hyperlink': hyperlink,
            'committee_name': None,
            'committee_type': None,
            'lead_procedure_exists': False,
            'lead_roll_call_exists': False,
            'opinion_procedure_exists': False,
            'opinion_roll_call_exists': False,
            'other_format': False,
            'error': 'No HTML content'
        }]

    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    error = None

    # ----- Check for "other format" placeholder -----
    placeholder = soup.find('p', class_=re.compile(r'alert alert-info'))
    if placeholder and "This document does not exist in HTML" in placeholder.get_text():
        rows.append({
            'procedure': procedure,
            'event': event,
            'reference': reference,
            'hyperlink': hyperlink,
            'committee_name': None,
            'committee_type': None,
            'lead_procedure_exists': False,
            'lead_roll_call_exists': False,
            'opinion_procedure_exists': False,
            'opinion_roll_call_exists': False,
            'other_format': True,
            'error': None
        })
        return rows

    # ----- Initialise flags and collections -----
    lead_proc = False
    lead_roll = False
    opinion_proc = False
    opinion_roll = False
    opinion_committees = set()  # use set to avoid duplicates
    lead_committees = []

    # ----- Parse the button list (table of contents) -----
    button_div = soup.find('div', class_=re.compile(r'es_links-list'))
    if button_div:
        for li in button_div.find_all('li', recursive=False):
            a = li.find('a')
            if not a:
                continue
            # Text is often inside a span with class 't-x'
            span = a.find('span', class_='t-x')
            text = span.get_text(strip=True) if span else a.get_text(strip=True)
            text_upper = text.upper()

            # Detect buttons
            if 'PROCEDURE' in text_upper:
                lead_proc = True
            if 'FINAL VOTE' in text_upper and 'COMMITTEE RESPONSIBLE' in text_upper:
                lead_roll = True
            if 'FINAL VOTE' in text_upper and 'OPINION' in text_upper:
                opinion_roll = True
            if 'OPINION' in text_upper and 'FINAL VOTE' not in text_upper:
                opinion_proc = True
                comm = extract_committee_name_from_opinion_text(text)
                if comm:
                    opinion_committees.add(comm)

    # ----- If button list not found, scan all headings (h2, h3) -----
    if not button_div:
        for heading in soup.find_all(['h2', 'h3']):
            heading_text = heading.get_text(strip=True).upper()
            if 'PROCEDURE' in heading_text:
                lead_proc = True
            if 'FINAL VOTE' in heading_text and 'COMMITTEE RESPONSIBLE' in heading_text:
                lead_roll = True
            if 'FINAL VOTE' in heading_text and 'OPINION' in heading_text:
                opinion_roll = True
            if 'OPINION' in heading_text and 'FINAL VOTE' not in heading_text:
                opinion_proc = True
                comm = extract_committee_name_from_opinion_text(heading.get_text(strip=True))
                if comm:
                    opinion_committees.add(comm)

    # ----- Extract lead committees from the PROCEDURE table -----
    # Look for the header containing "PROCEDURE – COMMITTEE RESPONSIBLE"
    proc_header = None
    for heading in soup.find_all(['h2', 'h3']):
        if 'PROCEDURE' in heading.get_text().upper() and 'COMMITTEE RESPONSIBLE' in heading.get_text().upper():
            proc_header = heading
            break
    if not proc_header:
        # fallback to any header with "PROCEDURE"
        for heading in soup.find_all(['h2', 'h3']):
            if 'PROCEDURE' in heading.get_text().upper():
                proc_header = heading
                break

    if proc_header:
        # The table is usually in a div.table-responsive following the header
        table_div = proc_header.find_next('div', class_='table-responsive')
        if table_div:
            table = table_div.find('table')
        else:
            table = proc_header.find_next('table')
        if table:
            # Look for row with first cell containing "Committee responsible" or "Committees responsible"
            for row in table.find_all('tr'):
                cells = row.find_all('td')
                if len(cells) < 2:
                    continue
                label = cells[0].get_text(strip=True).lower()
                if 'committee responsible' in label:
                    # Collect text from all following cells
                    for cell in cells[1:]:
                        p = cell.find('p')
                        comm_text = p.get_text(strip=True) if p else cell.get_text(strip=True)
                        if comm_text and comm_text.strip():
                            lead_committees.append(comm_text.strip())
                    break  # assume only one such row
                elif 'committees responsible' in label:
                    for cell in cells[1:]:
                        p = cell.find('p')
                        comm_text = p.get_text(strip=True) if p else cell.get_text(strip=True)
                        if comm_text and comm_text.strip():
                            lead_committees.append(comm_text.strip())
                    break

    # ----- Build rows -----
    # Lead committee rows
    for comm in lead_committees:
        rows.append({
            'procedure': procedure,
            'event': event,
            'reference': reference,
            'hyperlink': hyperlink,
            'committee_name': comm,
            'committee_type': 'lead',
            'lead_procedure_exists': lead_proc,
            'lead_roll_call_exists': lead_roll,
            'opinion_procedure_exists': opinion_proc,
            'opinion_roll_call_exists': opinion_roll,
            'other_format': False,
            'error': None
        })

    # Opinion committee rows
    for comm in opinion_committees:
        rows.append({
            'procedure': procedure,
            'event': event,
            'reference': reference,
            'hyperlink': hyperlink,
            'committee_name': comm,
            'committee_type': 'opinion',
            'lead_procedure_exists': lead_proc,
            'lead_roll_call_exists': lead_roll,
            'opinion_procedure_exists': opinion_proc,
            'opinion_roll_call_exists': opinion_roll,
            'other_format': False,
            'error': None
        })

    # If no committees at all, add a row with flags only
    if not lead_committees and not opinion_committees:
        rows.append({
            'procedure': procedure,
            'event': event,
            'reference': reference,
            'hyperlink': hyperlink,
            'committee_name': None,
            'committee_type': None,
            'lead_procedure_exists': lead_proc,
            'lead_roll_call_exists': lead_roll,
            'opinion_procedure_exists': opinion_proc,
            'opinion_roll_call_exists': opinion_roll,
            'other_format': False,
            'error': error
        })

    return rows

# ---------- Main processing ----------
# 1. Load the CSV
df = pd.read_csv(CSV_PATH)

# 2. Filter rows where reference starts with A + digit
mask = df['reference'].astype(str).str.match(REF_PATTERN)
filtered = df[mask].copy()
print(f"Found {len(filtered)} rows to process.")

# 3. Process each row (with a simple cache to avoid repeated fetches of the same URL)
cache = {}
all_rows = []

for idx, row in filtered.iterrows():
    proc = row['procedure']
    event = row['event']
    ref = row['reference']
    url = row['hyperlink']

    # Use cached HTML if available
    if url not in cache:
        print(f"Fetching: {url}")
        cache[url] = fetch_html(url)
    html = cache[url]

    # Parse the page and collect rows
    page_rows = parse_page(html, proc, event, ref, url)
    all_rows.extend(page_rows)

# 4. Create the final DataFrame
df_info = pd.DataFrame(all_rows)

# (Optional) reorder columns for clarity
column_order = [
    'procedure', 'event', 'reference', 'hyperlink',
    'committee_name', 'committee_type',
    'lead_procedure_exists', 'lead_roll_call_exists',
    'opinion_procedure_exists', 'opinion_roll_call_exists',
    'other_format', 'error'
]
# Ensure all columns exist
for col in column_order:
    if col not in df_info.columns:
        df_info[col] = None

df_info = df_info[column_order]

print("\nDone. Result shape:", df_info.shape)

Found 54 rows to process.
Fetching: https://www.europarl.europa.eu/doceo/document/A-9-2022-0219_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-8-2015-0029_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-7-2013-0315_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-7-2012-0043_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-7-2011-0339_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-6-2008-0329_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-5-2000-0309_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-5-2001-0133_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-5-2001-0313_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-4-1995-0221_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-4-1996-0194_EN.html
Fetching: https://www.europarl.europa.eu/doceo/document/A-7-2012-0273_EN.html
Fetching: https://www.europarl.europa.

,procedure,event,reference,hyperlink,committee_name,committee_type,lead_procedure_exists,lead_roll_call_exists,opinion_procedure_exists,opinion_roll_call_exists,other_format,error
0,2021/0366(COD),"Committee report tabled for plenary, 1st readi...",A9-0219/2022,https://www.europarl.europa.eu/doceo/document/...,ENVI,lead,False,False,False,False,False,None
1,2021/0366(COD),"Committee report tabled for plenary, 1st reading",A9-0219/2022,https://www.europarl.europa.eu/doceo/document/...,ENVI,lead,False,False,False,False,False,None
2,2014/0011(COD),"Committee report tabled for plenary, 1st readi...",A8-0029/2015,https://www.europarl.europa.eu/doceo/document/...,ENVI,lead,False,False,False,False,False,None
3,2014/0011(COD),"Committee report tabled for plenary, 1st reading",A8-0029/2015,https://www.europarl.europa.eu/doceo/document/...,ENVI,lead,False,False,False,False,False,None
4,2012/0364(COD),"Committee report tabled for plenary, 1st reading",A7-0315/2013,https://www.europarl.europa.eu/doceo/document/...,ECON,lead,False,False,False,False,False,None
5,2012/0364(COD),"Committee report tabled for plenary, 1st readi...",A7-0315/2013,https://www.europarl.europa.eu/doceo/document/...,ECON,lead,False,False,False,False,False,None
6,2010/0380(COD),"Committee report tabled for plenary, 1st reading",A7-0043/2012,https://www.europarl.europa.eu/doceo/document/...,EMPL,lead,False,False,False,False,False,None
7,2010/0380(COD),"Committee report tabled for plenary, 1st readi...",A7-0043/2012,https://www.europarl.europa.eu/doceo/document/...,EMPL,lead,False,False,False,False,False,None
8,2010/0377(COD),"Committee report tabled for plenary, 1st readi...",A7-0339/2011,https://www.europarl.europa.eu/doceo/document/...,ENVI,lead,False,False,False,False,False,None
9,2010/0377(COD),"Committee report tabled for plenary, 1st reading",A7-0339/2011,https://www.europarl.europa.eu/doceo/document/...,ENVI,lead,False,False,False,False,False,None


In [63]:
df_info1 = df_info.drop_duplicates(subset=['reference']).reset_index(drop=True)

df_info1

,procedure,event,reference,hyperlink,committee_name,committee_type,lead_procedure_exists,lead_roll_call_exists,opinion_procedure_exists,opinion_roll_call_exists,other_format,error
0,2021/0366(COD),"Committee report tabled for plenary, 1st readi...",A9-0219/2022,https://www.europarl.europa.eu/doceo/document/...,ENVI,lead,False,False,False,False,False,None
1,2014/0011(COD),"Committee report tabled for plenary, 1st readi...",A8-0029/2015,https://www.europarl.europa.eu/doceo/document/...,ENVI,lead,False,False,False,False,False,None
2,2012/0364(COD),"Committee report tabled for plenary, 1st reading",A7-0315/2013,https://www.europarl.europa.eu/doceo/document/...,ECON,lead,False,False,False,False,False,None
3,2010/0380(COD),"Committee report tabled for plenary, 1st reading",A7-0043/2012,https://www.europarl.europa.eu/doceo/document/...,EMPL,lead,False,False,False,False,False,None
4,2010/0377(COD),"Committee report tabled for plenary, 1st readi...",A7-0339/2011,https://www.europarl.europa.eu/doceo/document/...,ENVI,lead,False,False,False,False,False,None
5,2007/0295(COD),"Committee report tabled for plenary, 1st reading",A6-0329/2008,https://www.europarl.europa.eu/doceo/document/...,ENVI,lead,False,False,False,False,False,None
6,2000/0073(COD),"Committee report tabled for plenary, 1st reading",A5-0309/2000,https://www.europarl.europa.eu/doceo/document/...,NaN,NaN,False,False,False,False,True,None
7,2000/0073(COD),"Committee recommendation tabled for plenary, 2...",A5-0133/2001,https://www.europarl.europa.eu/doceo/document/...,NaN,NaN,False,False,False,False,True,None
8,2000/0073(COD),Report tabled for plenary by Parliament delega...,A5-0313/2001,https://www.europarl.europa.eu/doceo/document/...,NaN,NaN,False,False,False,False,False,None
9,1994/0272(COD),"Committee report tabled for plenary, 1st readi...",A4-0221/1995,https://www.europarl.europa.eu/doceo/document/...,NaN,NaN,False,False,False,False,True,None


In [56]:
df_info1

0     A9-0219/2022
2     A8-0029/2015
4     A7-0315/2013
6     A7-0043/2012
8     A7-0339/2011
10    A6-0329/2008
12    A5-0309/2000
14    A5-0133/2001
16    A5-0313/2001
18    A4-0221/1995
20    A4-0194/1996
22    A7-0273/2012
24    A7-0116/2014
26    A7-0034/2013
28    A7-0255/2013
30    A7-0142/2014
32    A8-0057/2014
34    A8-0295/2015
36    A8-0011/2016
38    A8-0378/2016
40    A8-0375/2017
44    A8-0405/2017
46    A9-0164/2024
48    A8-0274/2018
50    A8-0004/2019
52    A9-0255/2020
54    A9-0114/2021
Name: reference, dtype: str

In [1]:
import pandas as pd

In [11]:
df = pd.read_csv(r'C:\Users\User\eu_legislation\scrapper\bld\beta_test_procedures_committee_vote_info.csv')

df.head()

,procedure,committee,designation,procedure_available,rollcall_available,procedure_string,rollcall_string
0,2021/0366(COD),ENVI,lead,True,True,"<div class=""table-responsive""><table class=""ta...","<div class=""table-responsive""><table class=""ta..."
1,2021/0366(COD),AGRI,opinion,True,True,"<div class=""table-responsive""><table class=""ta...","<div class=""table-responsive""><table class=""ta..."
2,2021/0366(COD),DEVE,opinion,True,True,"<div class=""table-responsive""><table class=""ta...","<div class=""table-responsive""><table class=""ta..."
3,2021/0366(COD),IMCO,opinion,True,True,"<div class=""table-responsive""><table class=""ta...","<div class=""table-responsive""><table class=""ta..."
4,2021/0366(COD),INTA,opinion,True,True,"<div class=""table-responsive""><table class=""ta...","<div class=""table-responsive""><table class=""ta..."


In [3]:
df.shape

(70, 7)

In [7]:
df = pd.read_csv(r'C:\Users\User\eu_legislation\scrapper\bld\beta_test_procedures_committee_vote_info.csv')

df.tail()

,procedure,committee,designation,procedure_available,rollcall_available,procedure_string,rollcall_string
58,2019/0002(COD),AFET,no_op,False,False,NaN,NaN
59,2019/0002(COD),TRAN,no_op,False,False,NaN,NaN
60,2020/0262(COD),EMPL,lead,True,True,"<div class=""table-responsive""><table class=""ta...","<div class=""table-responsive""><table class=""ta..."
61,2020/0262(COD),JURI,opinion,True,True,"<div class=""table-responsive""><table class=""ta...","<div class=""table-responsive""><table class=""ta..."
62,2020/0262(COD),ENVI,no_op,False,False,NaN,NaN


In [10]:
pd.set_option('display.max_rows', None)

In [12]:
from bs4 import BeautifulSoup
import pandas as pd
import re

# ------------------------------------------------------------
# helpers
# ------------------------------------------------------------
def clean_text(text: str) -> str:
    if text is None:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()

def split_members(member_text: str) -> list[str]:
    """
    Split a comma-separated member string into individual names.
    """
    member_text = clean_text(member_text)
    if not member_text:
        return []
    return [m.strip() for m in member_text.split(",") if m.strip()]

def map_vote_symbol_to_type(symbol: str) -> str | None:
    """
    Map roll-call symbol to requested vote type labels.
    """
    symbol = clean_text(symbol)
    return {
        "+": "pos",
        "-": "neg",
        "0": "abs",
    }.get(symbol)

def parse_rollcall_html(
    rollcall_html: str,
    procedure: str,
    committee: str,
    designation: str
) -> list[dict]:
    """
    Parse one rollcall_string into member-level long rows.

    Output columns:
    procedure, committee, designation, vote_type, group, member
    """
    if pd.isna(rollcall_html) or not clean_text(rollcall_html):
        return []

    soup = BeautifulSoup(rollcall_html, "html.parser")
    tables = soup.find_all("table")
    out_rows = []

    for table in tables:
        trs = table.find_all("tr")
        if len(trs) < 2:
            continue

        # first row: total count + vote symbol
        header_cells = trs[0].find_all("td")
        if len(header_cells) < 2:
            continue

        vote_symbol = clean_text(header_cells[1].get_text(" ", strip=True))
        vote_type = map_vote_symbol_to_type(vote_symbol)

        if vote_type is None:
            continue

        # second row: groups and member blobs
        data_cells = trs[1].find_all("td")
        if len(data_cells) < 2:
            continue

        group_ps = [clean_text(p.get_text(" ", strip=True)) for p in data_cells[0].find_all("p")]
        member_ps = [clean_text(p.get_text(" ", strip=True)) for p in data_cells[1].find_all("p")]

        # remove empty paragraphs
        group_ps = [g for g in group_ps if g]
        member_ps = [m for m in member_ps if m]

        # Usually these align 1-to-1 after removing blanks
        n = min(len(group_ps), len(member_ps))

        for group, member_blob in zip(group_ps[:n], member_ps[:n]):
            members = split_members(member_blob)

            for member in members:
                out_rows.append({
                    "procedure": procedure,
                    "committee": committee,
                    "designation": designation,
                    "vote_type": vote_type,
                    "group": group,
                    "member": member,
                })

    return out_rows


# ------------------------------------------------------------
# build long dataframe from df
# assumes df already exists and has at least:
# procedure, committee, designation, rollcall_string
# ------------------------------------------------------------
all_rows = []

for _, row in df.iterrows():
    procedure = row["procedure"]
    committee = row["committee"]
    designation = row["designation"]
    rollcall_html = row.get("rollcall_string", "")

    parsed_rows = parse_rollcall_html(
        rollcall_html=rollcall_html,
        procedure=procedure,
        committee=committee,
        designation=designation
    )
    all_rows.extend(parsed_rows)

df_rollcalls_long = pd.DataFrame(
    all_rows,
    columns=[
        "procedure",
        "committee",
        "designation",
        "vote_type",
        "group",
        "member",
    ]
)

# ------------------------------------------------------------
# merge duplicated lead committees within the same procedure
# e.g. IMCO + JURI  ->  IMCO/JURI
# ------------------------------------------------------------
lead_combo_map = (
    df.loc[df["designation"].eq("lead")]
      .groupby("procedure")["committee"]
      .agg(lambda s: sorted(pd.unique(s)))
)

# keep only procedures with more than one lead committee
lead_combo_map = lead_combo_map[lead_combo_map.apply(len) > 1]

# convert list like ["IMCO", "JURI"] -> "IMCO/JURI"
lead_combo_map = {proc: "/".join(committees) for proc, committees in lead_combo_map.items()}

# relabel lead committee rows in the long dataframe
mask = (
    df_rollcalls_long["designation"].eq("lead")
    & df_rollcalls_long["procedure"].isin(lead_combo_map)
)

df_rollcalls_long.loc[mask, "committee"] = (
    df_rollcalls_long.loc[mask, "procedure"].map(lead_combo_map)
)

# remove exact duplicates just in case
df_rollcalls_long = df_rollcalls_long.drop_duplicates().reset_index(drop=True)

df_rollcalls_long.head()

,procedure,committee,designation,vote_type,group,member
0,2021/0366(COD),ENVI,lead,pos,PPE,Isabella Adinolfi
1,2021/0366(COD),ENVI,lead,pos,PPE,Bartosz Arłukowicz
2,2021/0366(COD),ENVI,lead,pos,PPE,Traian Băsescu
3,2021/0366(COD),ENVI,lead,pos,PPE,Alexander Bernhuber
4,2021/0366(COD),ENVI,lead,pos,PPE,Nathalie Colin-Oesterlé


In [14]:
df_rollcalls_long.sort_values(["procedure", "vote_type", "committee", "designation", "group", "member"], ascending= False, inplace=True)

In [15]:
df_rollcalls_long.shape

(308, 6)

In [36]:
df = pd.read_csv(r'bld\beta_test_procedures_committee_vote_info.csv')

In [37]:
df.head()

df1 = df[df['committee'] == 'other format']

df1

,procedure,committee,designation,procedure_available,rollcall_available,procedure_string,rollcall_string
15,2000/0073(COD),other format,other format,False,False,NaN,NaN
16,1994/0272(COD),other format,other format,False,False,NaN,NaN
36,2014/0091(COD),other format,other format,False,False,NaN,NaN


In [38]:
df_timeline = pd.read_csv(r'bld\test_procedures_timeline.csv')

In [40]:
import pandas as pd

df = pd.read_csv(r'bld\committee_other_format_text.csv')

df.head()

,procedure,reference,format,extracted_text
0,2000/0073(COD),A5-0309/2000,pdf,RR\423874EN.doc PE 293.670 EN EN EUROPEAN PARL...
1,1994/0272(COD),A4-0221/1995,pdf,28 September 1995 A4-0221/95 ╔═════╗ ║ ***I║ ╚...
2,2014/0091(COD),A8-0011/2016,pdf,RR\1085792EN.doc PE565.015v04-00 EN United in ...


In [ ]:
import io
import re
from pathlib import Path
from urllib.parse import urljoin
from typing import Dict, List, Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader
from IPython.display import display


# ---------------------------------------------------------------------
# paths
# ---------------------------------------------------------------------
BLD = Path("bld")


# ---------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------
def fetch_html(url: str, timeout: int = 30) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(url, headers=headers, timeout=timeout)
    response.raise_for_status()
    return response.text


def clean_text_block(text: Optional[str]) -> str:
    """
    Clean page text while keeping paragraph/line structure reasonably readable.
    """
    if text is None:
        return ""

    text = str(text).replace("\xa0", " ")
    lines = [re.sub(r"\s+", " ", line).strip() for line in text.splitlines()]
    lines = [line for line in lines if line]

    return "\n".join(lines).strip()


def get_other_format_links(procedure_url: str) -> Dict[str, Optional[str]]:
    """
    Find downloadable PDF / Word links on a procedure page.
    """
    html = fetch_html(procedure_url)
    soup = BeautifulSoup(html, "html.parser")

    links = {"pdf": None, "word": None}

    for a in soup.find_all("a", href=True):
        href = urljoin(procedure_url, a["href"])
        href_l = href.lower()

        text_l = a.get_text(" ", strip=True).lower()
        title_l = (a.get("title") or "").lower()

        is_pdf = (
            href_l.endswith(".pdf")
            or ".pdf" in href_l
            or text_l == "pdf"
            or " pdf" in text_l
            or title_l == "pdf"
            or "pdf" in title_l
        )

        is_word = (
            href_l.endswith(".doc")
            or href_l.endswith(".docx")
            or ".doc" in href_l
            or ".docx" in href_l
            or text_l == "word"
            or text_l == "doc"
            or text_l == "docx"
            or "word" in text_l
            or "doc" in title_l
        )

        if is_pdf and links["pdf"] is None:
            links["pdf"] = href

        if is_word and links["word"] is None:
            links["word"] = href

    return links


def extract_pdf_pages_from_url(
    pdf_url: str,
    procedure: str,
    reference: str
) -> List[Dict[str, object]]:
    """
    Extract one row per PDF page.

    Important:
    - page is the sequential PDF page index: 1, 2, 3, ...
    - this does NOT rely on footer parsing
    - so page 1 is always the first PDF page, even if no printed footer exists there
    """
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(pdf_url, headers=headers, timeout=120)
    response.raise_for_status()

    reader = PdfReader(io.BytesIO(response.content))
    rows: List[Dict[str, object]] = []

    for page_number, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text() or ""
        page_text = clean_text_block(raw_text)

        rows.append({
            "procedure": procedure,
            "reference": reference,
            "format": "pdf",
            "page": page_number,
            "text": page_text,
        })

    return rows


def build_other_format_page_table(
    df_vote_info: pd.DataFrame,
    df_timeline: pd.DataFrame
) -> pd.DataFrame:
    """
    Build a page-level extraction table with columns:
    procedure, reference, format, page, text
    """
    # identify "other format" procedures
    committee_series = df_vote_info.get("committee", pd.Series(dtype="object")).fillna("").astype(str).str.strip().str.lower()
    designation_series = df_vote_info.get("designation", pd.Series(dtype="object")).fillna("").astype(str).str.strip().str.lower()

    other_format_procedures = (
        df_vote_info.loc[
            (committee_series.eq("other format")) | (designation_series.eq("other format")),
            "procedure"
        ]
        .dropna()
        .drop_duplicates()
    )

    if other_format_procedures.empty:
        return pd.DataFrame(columns=["procedure", "reference", "format", "page", "text"])

    # match back to the relevant timeline event
    df_timeline_filtered = df_timeline.loc[
        df_timeline["event"].eq("Committee report tabled for plenary, 1st reading")
    ].copy()

    df_target = df_timeline_filtered[
        df_timeline_filtered["procedure"].isin(other_format_procedures)
    ].copy()

    if df_target.empty:
        return pd.DataFrame(columns=["procedure", "reference", "format", "page", "text"])

    keep_cols = [c for c in ["procedure", "reference", "hyperlink"] if c in df_target.columns]
    df_target = df_target[keep_cols].drop_duplicates(subset=["procedure"])

    rows: List[Dict[str, object]] = []

    for _, row in df_target.iterrows():
        procedure = row["procedure"]
        reference = row.get("reference", "")
        procedure_url = row.get("hyperlink", "")

        if pd.isna(procedure_url) or not str(procedure_url).strip():
            rows.append({
                "procedure": procedure,
                "reference": reference,
                "format": "",
                "page": pd.NA,
                "text": "",
            })
            continue

        try:
            links = get_other_format_links(procedure_url)
            pdf_url = links.get("pdf")
            word_url = links.get("word")

            if pdf_url:
                page_rows = extract_pdf_pages_from_url(
                    pdf_url=pdf_url,
                    procedure=procedure,
                    reference=reference
                )
                rows.extend(page_rows)

            elif word_url:
                rows.append({
                    "procedure": procedure,
                    "reference": reference,
                    "format": "word",
                    "page": pd.NA,
                    "text": "",
                })

            else:
                rows.append({
                    "procedure": procedure,
                    "reference": reference,
                    "format": "",
                    "page": pd.NA,
                    "text": "",
                })

        except Exception as e:
            rows.append({
                "procedure": procedure,
                "reference": reference,
                "format": "",
                "page": pd.NA,
                "text": f"Failed to process procedure page: {e}",
            })

    df_out = pd.DataFrame(rows, columns=["procedure", "reference", "format", "page", "text"])

    # optional: ensure nullable integer dtype for page
    if not df_out.empty:
        df_out["page"] = df_out["page"].astype("Int64")

    return df_out


# ---------------------------------------------------------------------
# run
# ---------------------------------------------------------------------
df_vote_info = pd.read_csv(BLD / "committee_vote_info.csv")
df_timeline = pd.read_csv(BLD / "timeline_table.csv")

df_page_text = build_other_format_page_table(df_vote_info, df_timeline)

print(f"Built page-level table with {len(df_page_text)} rows")
display(df_page_text.head(20))

# # optional save
# output_csv = "committee_other_format_page_text.csv"
# df_page_text.to_csv(output_csv, index=False)
# print(f"Saved to: {output_csv}")

Built page-level table with 190 rows


,procedure,reference,format,page,text
0,2000/0073(COD),A5-0309/2000,pdf,1,RR\423874EN.doc PE 293.670\nEN EN\nEUROPEAN PA...
1,2000/0073(COD),A5-0309/2000,pdf,2,PE 293.670 2/44 RR\423874EN.doc\nEN\nSymbols f...
2,2000/0073(COD),A5-0309/2000,pdf,3,RR\423874EN.doc 3/44 PE 293.670\nEN\nCONTENTS\...
3,2000/0073(COD),A5-0309/2000,pdf,4,PE 293.670 4/44 RR\423874EN.doc\nEN\nPROCEDURA...
4,2000/0073(COD),A5-0309/2000,pdf,5,RR\423874EN.doc 5/44 PE 293.670\nEN\nLEGISLATI...
5,2000/0073(COD),A5-0309/2000,pdf,6,PE 293.670 6/44 RR\423874EN.doc\nEN\nWith a vi...
6,2000/0073(COD),A5-0309/2000,pdf,7,RR\423874EN.doc 7/44 PE 293.670\nEN\ninstituti...
7,2000/0073(COD),A5-0309/2000,pdf,8,PE 293.670 8/44 RR\423874EN.doc\nEN\n(Amendmen...
8,2000/0073(COD),A5-0309/2000,pdf,9,RR\423874EN.doc 9/44 PE 293.670\nEN\nJustifica...
9,2000/0073(COD),A5-0309/2000,pdf,10,PE 293.670 10/44 RR\423874EN.doc\nEN\nsafe pro...


Saved to: committee_other_format_page_text.csv


In [56]:
import re
from pathlib import Path

import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------
# fallback: load df_page_text if it is not already in memory
# ---------------------------------------------------------------------
if "df_page_text" not in globals():
    df_page_text = pd.read_csv(Path("bld") / "committee_other_format_page_text.csv")


# ---------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------
EU_LANG_CODES = {
    "BG", "CS", "DA", "DE", "EL", "EN", "ES", "ET", "FI", "FR", "GA",
    "HR", "HU", "IT", "LT", "LV", "MT", "NL", "PL", "PT", "RO", "SK",
    "SL", "SV"
}


def as_text(x):
    if pd.isna(x):
        return ""
    return str(x).replace("\xa0", " ")


def norm_space(text):
    return re.sub(r"\s+", " ", as_text(text)).strip()


def norm_dashes(text):
    return re.sub(r"[–—−]", "-", as_text(text))


def strip_box_chars(text):
    """
    Remove box/ornament characters that show up in old PDFs.
    """
    text = as_text(text)
    text = re.sub(r"[╔╗║╚╝═]", " ", text)
    text = re.sub(r"[★☆■□▪▫◆◇•·]+", " ", text)
    return text


def clean_line(line):
    return re.sub(r"\s+", " ", strip_box_chars(line)).strip()


def extract_language_flag(text):
    """
    Detect a 2-letter language flag like EN / FR / DE.
    Returns the most frequent valid code found in the text.
    """
    text = strip_box_chars(text)
    tokens = re.findall(r"\b[A-Za-z]{2}\b", text)
    codes = [tok.upper() for tok in tokens if tok.upper() in EU_LANG_CODES]

    if not codes:
        return pd.NA

    return pd.Series(codes).value_counts().idxmax()


def extract_pe_ref(text):
    """
    Examples:
    - PE 293.670
    - PE 213.392/fin
    - PE565.015v04-00
    """
    text = strip_box_chars(text)
    pattern = r"\bPE\s*\d[\d\.]*(?:v\d{2}-\d{2})?(?:/[A-Za-z0-9.\-]+)?"
    m = re.search(pattern, text, flags=re.IGNORECASE)
    if not m:
        return pd.NA
    return re.sub(r"\s+", " ", m.group(0)).strip()


def extract_rr_ref(text):
    """
    Examples:
    - RR\423874EN.doc
    - RR\1085792EN.doc
    """
    text = strip_box_chars(text)
    m = re.search(r"RR\\\d+[A-Z]{2}\.doc\b", text)
    if not m:
        return pd.NA
    return m.group(0).strip()


def has_contents_page(page_texts):
    """
    Detects CONTENTS or spaced C O N T E N T S on any page.
    """
    pattern = re.compile(
        r"(^|\n)\s*(?:CONTENTS|C\s*O\s*N\s*T\s*E\s*N\s*T\s*S)\s*(\n|$)",
        flags=re.IGNORECASE
    )
    for txt in page_texts:
        if pattern.search(strip_box_chars(txt)):
            return True
    return False


def extract_doc_date(text):
    """
    Examples:
    - 20 October 2000
    - 28 September 1995
    - 3.2.2016
    """
    text = strip_box_chars(text)

    patterns = [
        r"\b\d{1,2}\s+[A-Z][a-z]+\s+\d{4}\b",
        r"\b\d{1,2}\.\d{1,2}\.\d{4}\b",
    ]

    for pat in patterns:
        m = re.search(pat, text)
        if m:
            return m.group(0).strip()

    return pd.NA


def extract_stage(text):
    """
    Handles things like:
    ***I
    **I
    ***II
    ***III

    including older boxed extraction like:
    ║ ***I║
    """
    for raw_line in as_text(text).splitlines():
        line = clean_line(raw_line)
        line = re.sub(r"[^*IVX]", "", line.upper())

        if line in {"***III", "***II", "***I", "***", "**II", "**I", "*"}:
            return line

    return pd.NA


def extract_title(text):
    """
    Title starts after the final '... Council ... on'
    and stops before '(COM'.

    Examples handled:
    - REPORT on the proposal for a European Parliament and Council directive on ...
    - REPORT on the proposal for a European Parliament and Council Directive on ...
    - REPORT on the proposal for a directive of the European Parliament and of the Council on ...
    """
    flat = norm_space(strip_box_chars(text))

    patterns = [
        r"REPORT\s+on\s+the\s+proposal\s+for\s+a\s+European\s+Parliament\s+and\s+Council\s+Directive\s+on\s+(.*?)\s+\(\s*COM",
        r"REPORT\s+on\s+the\s+proposal\s+for\s+a\s+European\s+Parliament\s+and\s+Council\s+directive\s+on\s+(.*?)\s+\(\s*COM",
        r"REPORT\s+on\s+the\s+proposal\s+for\s+a\s+directive\s+of\s+the\s+European\s+Parliament\s+and\s+of\s+the\s+Council\s+on\s+(.*?)\s+\(\s*COM",
    ]

    for pat in patterns:
        m = re.search(pat, flat, flags=re.IGNORECASE)
        if m:
            return m.group(1).strip()

    return pd.NA


def extract_ref_block(text):
    """
    Extract the whole reference block beginning with (COM ... )
    but without the outer opening '(' and final closing ')'.

    Example output:
    COM (2000) 139 – C5-0224/2000 – 2000/0073(COD)
    """
    text = strip_box_chars(text)

    start = re.search(r"\(COM", text, flags=re.IGNORECASE)
    if not start:
        return pd.NA

    tail = text[start.start():]

    # stop before Committee on ..., if present
    stop = re.search(r"\bCommittee\s+on\b", tail, flags=re.IGNORECASE)
    if stop:
        candidate = tail[:stop.start()]
    else:
        candidate = tail

    candidate = candidate.strip()

    # remove one outer opening and one outer closing parenthesis only
    if candidate.startswith("("):
        candidate = candidate[1:]
    candidate = candidate.strip()
    if candidate.endswith(")"):
        candidate = candidate[:-1]

    candidate = re.sub(r"\s+", " ", candidate).strip(" -–—")
    return candidate if candidate else pd.NA

def split_ref_block(ref_block):
    """
    Split COM / C-line / procedure refs on separator dashes,
    but do NOT split inside references like:
    - C4-OO2O/95
    - C5-0224/2000

    Rule:
    split on dashes unless the dash is immediately preceded by C<digits>.
    """
    if pd.isna(ref_block):
        return []

    block = norm_dashes(ref_block)

    # split on "-" only when it is NOT the internal dash of something like C5-0224/2000
    parts = [
        part.strip()
        for part in re.split(r"(?<!C\d)\s*-\s*", block, flags=re.IGNORECASE)
        if part.strip()
    ]
    return parts

# def split_ref_block(ref_block):
#     """
#     Split COM/C-line/procedure refs on dashes.
#     Returns a list.
#     """
#     if pd.isna(ref_block):
#         return []

#     block = norm_dashes(ref_block)
#     parts = [part.strip() for part in re.split(r"\s*-\s*", block) if part.strip()]
#     return parts


def extract_committee(text):
    """
    First 'Committee on ...' line from page 1.
    """
    text = strip_box_chars(text)
    m = re.search(r"(?mi)^\s*Committee\s+on\s+.+$", text)
    if not m:
        return pd.NA
    return norm_space(m.group(0))


def extract_rapporteur(text):
    """
    Handles:
    Rapporteur: Laura González Álvarez
    Rapporteur:Mr Allan Macartney
    """
    text = strip_box_chars(text)
    m = re.search(r"(?mi)^\s*Rapporteur\s*:\s*(.+)$", text)
    if not m:
        return pd.NA
    return norm_space(m.group(1))


# ---------------------------------------------------------------------
# builder
# ---------------------------------------------------------------------
def build_first_page_meta(df_page_text):
    df = df_page_text.copy()

    ref_col = "reference" if "reference" in df.columns else "web_reference"
    fmt_col = "format" if "format" in df.columns else None

    df["page"] = pd.to_numeric(df["page"], errors="coerce")

    rows = []

    group_cols = ["procedure", ref_col]
    if fmt_col is not None:
        group_cols.append(fmt_col)

    for keys, g in df.groupby(group_cols, dropna=False):
        if len(group_cols) == 3:
            procedure, web_reference, fmt = keys
        else:
            procedure, web_reference = keys
            fmt = pd.NA

        g = g.sort_values("page").copy()

        first_page = g[g["page"] == 1]
        if not first_page.empty:
            first_text = as_text(first_page.iloc[0]["text"])
        else:
            first_text = as_text(g.iloc[0]["text"])

        all_texts = g["text"].fillna("").astype(str).tolist()
        full_doc_text = "\n".join(all_texts)

        language = extract_language_flag(first_text)
        if pd.isna(language):
            language = extract_language_flag(full_doc_text)

        pe_ref = extract_pe_ref(first_text)
        if pd.isna(pe_ref):
            pe_ref = extract_pe_ref(full_doc_text)

        rr_ref = extract_rr_ref(first_text)
        if pd.isna(rr_ref):
            rr_ref = extract_rr_ref(full_doc_text)

        ref_block = extract_ref_block(first_text)
        ref_i = split_ref_block(ref_block)

        rows.append(
            {
                "procedure": procedure,
                "web_reference": web_reference,
                "format": fmt,
                "language": language,
                "pe_ref": pe_ref,
                "rr_ref": rr_ref,
                "contents": has_contents_page(all_texts),
                "doc_date": extract_doc_date(first_text),
                "stage": extract_stage(first_text),
                "title": extract_title(first_text),
                "references": ref_i,
                "committee": extract_committee(first_text),
                "designation": "lead",
                "rapporteur": extract_rapporteur(first_text),
            }
        )

    out = pd.DataFrame(rows)

    out = out[
        [
            "procedure",
            "web_reference",
            "format",
            "language",
            "pe_ref",
            "rr_ref",
            "contents",
            "doc_date",
            "stage",
            "title",
            "references",
            "committee",
            "designation",
            "rapporteur",
        ]
    ].sort_values(["procedure", "web_reference"], kind="stable").reset_index(drop=True)

    return out


# ---------------------------------------------------------------------
# run
# ---------------------------------------------------------------------
dff_first_page_meta = build_first_page_meta(df_page_text)

display(dff_first_page_meta)

,procedure,web_reference,format,language,pe_ref,rr_ref,contents,doc_date,stage,title,references,committee,designation,rapporteur
0,1994/0272(COD),A4-0221/1995,pdf,EN,PE 213.392/fin,NaN,True,28 September 1995,***I,energy efficiency requirements for household e...,"[COM(94)O521, C4-OO2O/95, 94/O272(COD)]","Committee on Research, Technological Developme...",lead,Mr Allan Macartney
1,2000/0073(COD),A5-0309/2000,pdf,EN,PE 293.670,RR\423874EN.doc,True,20 October 2000,***I,general product safety,"[COM (2000) 139, C5-0224/2000, 2000/0073(COD)]","Committee on the Environment, Public Health an...",lead,Laura González Álvarez
2,2014/0091(COD),A8-0011/2016,pdf,EN,PE565.015v04-00,RR\1085792EN.doc,True,3.2.2016,***I,the activities and supervision of institutions...,"[COM(2014)0167, C7-0112/2014, 2014/0091(COD)]",Committee on Economic and Monetary Affairs,lead,Brian Hayes


In [50]:
import re
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------
EU_LANG_CODES = {
    "BG", "CS", "DA", "DE", "EL", "EN", "ES", "ET", "FI", "FR", "GA",
    "HR", "HU", "IT", "LT", "LV", "MT", "NL", "PL", "PT", "RO", "SK",
    "SL", "SV"
}


def as_text(x):
    if pd.isna(x):
        return ""
    return str(x).replace("\xa0", " ")


def strip_ornaments(text):
    text = as_text(text)
    text = re.sub(r"[╔╗║╚╝═]", " ", text)
    text = re.sub(r"[★☆■□▪▫◆◇•·]+", " ", text)
    return text


def norm_space(text):
    return re.sub(r"\s+", " ", strip_ornaments(text)).strip()


def normalize_for_match(text):
    text = strip_ornaments(text)
    text = text.replace("–", "-").replace("—", "-").replace("−", "-")
    text = re.sub(r"\s+", " ", text).strip().upper()
    return text


def clean_line(line):
    line = strip_ornaments(line)
    line = re.sub(r"\s+", " ", line).strip()
    return line


def looks_like_furniture(line):
    """
    Skip obvious page furniture from contents pages.
    """
    raw = clean_line(line)
    if not raw:
        return True

    norm = normalize_for_match(raw)

    if norm in {"PAGE"}:
        return True

    if norm in EU_LANG_CODES:
        return True

    # old style: - 2 - PE 213.392/fin.
    if re.match(r"^-?\s*\d+\s*-?\s*PE\b", raw, flags=re.IGNORECASE):
        return True

    # PE / RR / language headers
    if re.search(r"RR\\\d+[A-Z]{2}\.doc", raw):
        return True

    if re.search(r"\bPE\s*\d", raw, flags=re.IGNORECASE):
        # keep only if it looks like page furniture, not real content
        if re.search(r"\b\d+\s*/\s*\d+\b", raw) or re.search(r"\bEN\b", raw):
            return True
        if len(raw) < 40:
            return True

    return False


def is_contents_heading(line):
    """
    CONTENTS or spaced C O N T E N T S
    """
    norm = normalize_for_match(line)
    norm_no_space = re.sub(r"\s+", "", norm)
    return norm == "CONTENTS" or norm_no_space == "CONTENTS"


def find_contents_page(group_df):
    """
    Returns (page_number, page_text) for the first page that contains a contents heading.
    """
    for _, row in group_df.sort_values("page").iterrows():
        txt = as_text(row["text"])
        for line in txt.splitlines():
            if is_contents_heading(line):
                return row["page"], txt
    return pd.NA, ""


def parse_contents_items(contents_text):
    """
    Parse entries of the form:
    ITEM ............ 4

    Handles wrapped multi-line items, e.g.
    OPINION OF THE COMMITTEE ON LEGAL AFFAIRS
    AND THE INTERNAL MARKET ............ 22
    """
    lines = [clean_line(line) for line in as_text(contents_text).splitlines()]

    # start only after the CONTENTS heading
    start_idx = None
    for i, line in enumerate(lines):
        if is_contents_heading(line):
            start_idx = i + 1
            break

    if start_idx is None:
        return []

    lines = lines[start_idx:]

    entries = []
    buffer = []

    for line in lines:
        if looks_like_furniture(line):
            continue

        if not line:
            continue

        # item .... 12
        m = re.match(r"^(.*?)(?:\.{2,}|…{2,})\s*(\d+)\s*$", line)
        if m:
            item_piece = m.group(1).strip(" -")
            page_number = int(m.group(2))

            parts = buffer + ([item_piece] if item_piece else [])
            item = " ".join(parts)
            item = re.sub(r"\s+,", ",", item)
            item = norm_space(item)

            if item:
                entries.append((item, page_number))

            buffer = []
        else:
            buffer.append(line)

    return entries


def build_contents_df(df_page_text, dff_first_page_meta):
    page_ref_col = "reference" if "reference" in df_page_text.columns else "web_reference"
    meta_ref_col = "web_reference" if "web_reference" in dff_first_page_meta.columns else "reference"

    rows = []

    meta_contents = dff_first_page_meta.loc[dff_first_page_meta["contents"] == True].copy()

    for _, meta_row in meta_contents.iterrows():
        procedure = meta_row["procedure"]
        web_reference = meta_row[meta_ref_col]

        g = df_page_text.loc[
            (df_page_text["procedure"] == procedure)
            & (df_page_text[page_ref_col] == web_reference)
        ].copy()

        if g.empty:
            continue

        contents_page, contents_text = find_contents_page(g)
        if not contents_text:
            continue

        items = parse_contents_items(contents_text)

        for item, page_number in items:
            rows.append({
                "procedure": procedure,
                "item": item,
                "page_number": page_number,
            })

    return pd.DataFrame(rows, columns=["procedure", "item", "page_number"])


def build_df_committees(contents_df, dff_first_page_meta):
    rows = []

    meta_ref_col = "web_reference" if "web_reference" in dff_first_page_meta.columns else "reference"

    for _, meta_row in dff_first_page_meta.iterrows():
        procedure = meta_row["procedure"]
        lead_committee = meta_row.get("committee", pd.NA)

        cdf = contents_df.loc[contents_df["procedure"] == procedure].copy()
        if cdf.empty:
            continue

        # 1) add lead committee only if PROCEDURAL PAGE exists in contents
        has_procedural_page = cdf["item"].apply(
            lambda x: (
                normalize_for_match(x).startswith("PROCEDURAL PAGE")
                or normalize_for_match(x).startswith("PROCEDURE - COMMITTEE RESPONSIBLE")
            )
        ).any()

        if has_procedural_page and pd.notna(lead_committee) and str(lead_committee).strip():
            rows.append({
                "procedure": procedure,
                "committee": lead_committee,
                "designation": "lead",
            })

        # 2) opinion committees from contents items
        prefix = "OPINION OF THE "

        for item in cdf["item"]:
            item_norm = normalize_for_match(item)

            if item_norm.startswith(prefix):
                committee = norm_space(item[len(prefix):])
                if committee:
                    rows.append({
                        "procedure": procedure,
                        "committee": committee,
                        "designation": "opinion",
                    })

    out = pd.DataFrame(rows, columns=["procedure", "committee", "designation"])

    if not out.empty:
        out = (
            out.assign(
                _committee_norm=out["committee"].map(normalize_for_match),
                _designation_norm=out["designation"].str.lower()
            )
            .drop_duplicates(subset=["procedure", "_committee_norm", "_designation_norm"])
            .drop(columns=["_committee_norm", "_designation_norm"])
            .sort_values(["procedure", "designation", "committee"], kind="stable")
            .reset_index(drop=True)
        )

    return out


# ---------------------------------------------------------------------
# build outputs
# ---------------------------------------------------------------------
contents_df = build_contents_df(df_page_text, dff_first_page_meta)
df_committees = build_df_committees(contents_df, dff_first_page_meta)

display(contents_df)
display(df_committees)

,procedure,item,page_number
0,1994/0272(COD),Procedural page,3
1,1994/0272(COD),A. LEGISLATIVE PROPOSAL,4
2,1994/0272(COD),DRAFT LEGISLATIVE RESOLUTION,10
3,1994/0272(COD),B. EXPLANATORY STATEMENT,11
4,1994/0272(COD),Opinion of the Committee on Economic and Monet...,15
5,1994/0272(COD),"Opinion of the Committee on the Environment, P...",20
6,1994/0272(COD),Commission position (Rule 56(4)/Rule 57(3),27
7,2000/0073(COD),PROCEDURAL PAGE,4
8,2000/0073(COD),LEGISLATIVE PROPOSAL,5
9,2000/0073(COD),DRAFT LEGISLATIVE RESOLUTION,18


,procedure,committee,designation
0,1994/0272(COD),"Committee on Research, Technological Developme...",lead
1,1994/0272(COD),Committee on Economic and Monetary Affairs and...,opinion
2,1994/0272(COD),"Committee on the Environment, Public Health an...",opinion
3,2000/0073(COD),"Committee on the Environment, Public Health an...",lead
4,2000/0073(COD),"COMMITTEE ON INDUSTRY, EXTERNAL TRADE, RESEARC...",opinion
5,2000/0073(COD),COMMITTEE ON LEGAL AFFAIRS AND THE INTERNAL MA...,opinion
6,2014/0091(COD),Committee on Economic and Monetary Affairs,lead
7,2014/0091(COD),COMMITTEE ON EMPLOYMENT AND SOCIAL AFFAIRS,opinion
8,2014/0091(COD),COMMITTEE ON WOMEN'S RIGHTS AND GENDER EQUALITY,opinion


In [57]:
import ast
import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------
MONTH_MAP = {
    "january": "January",
    "february": "February",
    "march": "March",
    "april": "April",
    "may": "May",
    "june": "June",
    "july": "July",
    "august": "August",
    "september": "September",
    "october": "October",
    "november": "November",
    "december": "December",
}
MONTH_PATTERN = r"(January|February|March|April|May|June|July|August|September|October|November|December)"


def as_text(x):
    if pd.isna(x):
        return ""
    return str(x).replace("\xa0", " ")


def strip_ornaments(text):
    text = as_text(text)
    text = re.sub(r"[╔╗║╚╝═]", " ", text)
    text = re.sub(r"[★☆■□▪▫◆◇•·]+", " ", text)
    return text


def norm_space(text):
    return re.sub(r"\s+", " ", strip_ornaments(text)).strip()


def normalize_for_match(text):
    text = strip_ornaments(text)
    text = text.replace("–", "-").replace("—", "-").replace("−", "-")
    text = re.sub(r"\s+", " ", text).strip().upper()
    return text


def ascii_fold(text):
    text = unicodedata.normalize("NFKD", as_text(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text


def flatten_text(text):
    text = strip_ornaments(text)
    text = text.replace("–", "-").replace("—", "-").replace("−", "-")
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]
    return re.sub(r"\s+", " ", " ".join(lines)).strip()


def split_sentences(text):
    flat = flatten_text(text)
    if not flat:
        return []
    parts = re.split(r"(?<=[\.\?!])\s+", flat)
    return [part.strip() for part in parts if part.strip()]


def strip_honorifics(name):
    name = norm_space(name)
    name = re.sub(
        r"^(Mr|Mrs|Ms|Miss|Mme|M\.|Sir|Dr|Professor|Prof\.)\s+",
        "",
        name,
        flags=re.IGNORECASE,
    )
    return norm_space(name)


def parse_references_field(x):
    """
    Works if references is already a list, tuple, or a string representation of a list.
    """
    if isinstance(x, (list, tuple)):
        return [norm_space(v) for v in x if norm_space(v)]

    s = as_text(x).strip()
    if not s:
        return []

    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [norm_space(v) for v in parsed if norm_space(v)]
        except Exception:
            pass

    return [norm_space(s)]


def procedural_page_score(text):
    s = ascii_fold(flatten_text(text)).lower()
    score = 0

    if "procedural page" in s:
        score += 100
    if "appointed" in s and "rapporteur" in s:
        score += 30
    if "considered the commission proposal" in s:
        score += 30
    if "adopted the draft legislative resolution" in s:
        score += 30
    if "the following were present for the vote:" in s:
        score += 20
    if "the following took part in the vote/were present for the vote:" in s:
        score += 20
    if "the opinions of" in s:
        score += 10

    return score


def locate_procedural_page(proc_pages, target_page):
    """
    Search target_page +/- 1 and choose the strongest procedural-page candidate.
    """
    candidate_pages = proc_pages.loc[
        proc_pages["page"].between(target_page - 1, target_page + 1, inclusive="both")
    ].copy()

    if candidate_pages.empty:
        return pd.NA, ""

    candidate_pages["score"] = candidate_pages["text"].map(procedural_page_score)
    candidate_pages["distance"] = (candidate_pages["page"] - target_page).abs()

    candidate_pages = candidate_pages.sort_values(
        ["score", "distance", "page"],
        ascending=[False, True, True],
        kind="stable"
    )

    best = candidate_pages.iloc[0]
    if best["score"] <= 0:
        return pd.NA, ""

    return int(best["page"]), as_text(best["text"])


def fix_ocr_day_zeroes(text):
    """
    Fix common OCR issues like:
    3O May -> 30 May
    2O June -> 20 June
    """
    text = as_text(text)
    text = re.sub(
        rf"\b(\d)O(?=\s+{MONTH_PATTERN}\b)",
        lambda m: f"{m.group(1)}0",
        text,
        flags=re.IGNORECASE,
    )
    return text


def canonical_date(day, month, year):
    return f"{int(day)} {MONTH_MAP[month.lower()]} {int(year)}"


def parse_date_string_to_ts(date_str):
    return pd.to_datetime(date_str, dayfirst=True, errors="coerce")


def extract_dates_from_sentence(text):
    """
    Handles:
    - 23 May 2000
    - 24 April, 30 May, 20 June, 18 July and 26-27 September 1995
    - 12 September, 19 September and 17 October 2000
    """
    text = fix_ocr_day_zeroes(text)
    text = text.replace("–", "-").replace("—", "-").replace("−", "-")
    text = norm_space(text)

    years = re.findall(r"\b(19|20)\d{2}\b", text)
    base_year_match = re.findall(r"\b((?:19|20)\d{2})\b", text)
    base_year = base_year_match[-1] if base_year_match else None

    results = []

    pattern = re.compile(
        rf"""
        (\b\d{{1,2}}\s*-\s*\d{{1,2}}\s+{MONTH_PATTERN}(?:\s+(?:19|20)\d{{2}})?\b)
        |
        (\b\d{{1,2}}\s+{MONTH_PATTERN}(?:\s+(?:19|20)\d{{2}})?\b)
        """,
        flags=re.IGNORECASE | re.VERBOSE,
    )

    for m in pattern.finditer(text):
        frag = m.group(0).strip()

        range_match = re.match(
            rf"^\b(\d{{1,2}})\s*-\s*(\d{{1,2}})\s+{MONTH_PATTERN}(?:\s+((?:19|20)\d{{2}}))?\b$",
            frag,
            flags=re.IGNORECASE,
        )
        if range_match:
            start_day = int(range_match.group(1))
            end_day = int(range_match.group(2))
            month = range_match.group(3)
            year = range_match.group(4) or base_year

            if year:
                for day in range(start_day, end_day + 1):
                    results.append(canonical_date(day, month, year))
            continue

        single_match = re.match(
            rf"^\b(\d{{1,2}})\s+{MONTH_PATTERN}(?:\s+((?:19|20)\d{{2}}))?\b$",
            frag,
            flags=re.IGNORECASE,
        )
        if single_match:
            day = int(single_match.group(1))
            month = single_match.group(2)
            year = single_match.group(3) or base_year

            if year:
                results.append(canonical_date(day, month, year))

    # preserve order, remove duplicates
    deduped = []
    seen = set()
    for d in results:
        if d not in seen:
            seen.add(d)
            deduped.append(d)

    return deduped


def rapporteur_name_variants(name):
    """
    Full name strongest, surname fallback.
    """
    clean = strip_honorifics(name)
    if not clean:
        return []

    full = ascii_fold(clean).lower()
    parts = clean.split()
    surname = ascii_fold(parts[-1]).lower() if parts else ""

    variants = [full]
    if surname and surname not in variants:
        variants.append(surname)

    return variants


def extract_date_appointed(procedural_text, rapporteur_name):
    variants = rapporteur_name_variants(rapporteur_name)
    if not variants:
        return pd.NA

    for sent in split_sentences(procedural_text):
        sent_fold = ascii_fold(sent).lower()
        if "appoint" not in sent_fold or "rapporteur" not in sent_fold:
            continue

        if not any(v in sent_fold for v in variants):
            continue

        dates = extract_dates_from_sentence(sent)
        if dates:
            return dates[0]

    return pd.NA


def extract_discussed_dates(procedural_text):
    for sent in split_sentences(procedural_text):
        sent_fold = ascii_fold(sent).lower()
        if "considered the commission proposal" not in sent_fold:
            continue

        dates = extract_dates_from_sentence(sent)
        if dates:
            return dates

    return []


def extract_vote_and_adoption(procedural_text, discussed_dates):
    adoption_found = False
    adopted_date = pd.NA
    vote_rows = []

    for sent in split_sentences(procedural_text):
        sent_fold = ascii_fold(sent).lower()
        if "adopted the draft legislative resolution" not in sent_fold:
            continue

        adoption_found = True

        # date adopted = latest discussed in committee date
        if discussed_dates:
            parsed = [(d, parse_date_string_to_ts(d)) for d in discussed_dates]
            parsed = [(d, ts) for d, ts in parsed if pd.notna(ts)]
            if parsed:
                adopted_date = max(parsed, key=lambda x: x[1])[0]

        if "unanimously" in sent_fold:
            vote_rows.append(("Result of the vote", "unanimous"))
            return adoption_found, adopted_date, vote_rows

        m = re.search(
            r"by\s+(\d+)\s+votes?\s+to\s*([0-9]+)\s*,?\s*with\s+(\d+)\s+abstentions?",
            sent,
            flags=re.IGNORECASE,
        )
        if m:
            vote_rows.append(("Result of final vote +", m.group(1)))
            vote_rows.append(("Result of final vote –", m.group(2)))
            vote_rows.append(("Result of final vote 0", m.group(3)))
            return adoption_found, adopted_date, vote_rows

        # fallback for no-comma or odd spacing
        m = re.search(
            r"by\s+(\d+)\s+votes?\s+to\s*([0-9]+).*?with\s+(\d+)\s+abstentions?",
            sent,
            flags=re.IGNORECASE,
        )
        if m:
            vote_rows.append(("Result of final vote +", m.group(1)))
            vote_rows.append(("Result of final vote –", m.group(2)))
            vote_rows.append(("Result of final vote 0", m.group(3)))
            return adoption_found, adopted_date, vote_rows

        return adoption_found, adopted_date, vote_rows

    return adoption_found, adopted_date, vote_rows


def extract_members_present(procedural_text):
    flat = flatten_text(procedural_text)

    patterns = [
        r"The following took part in the vote/were present for the vote:\s*(.*?)(?=The opinions of|The report was tabled on|The deadline for tabling amendments|$)",
        r"The following were present for the vote:\s*(.*?)(?=The opinions of|The report was tabled on|The deadline for tabling amendments|$)",
    ]

    for pat in patterns:
        m = re.search(pat, flat, flags=re.IGNORECASE | re.DOTALL)
        if m:
            return norm_space(m.group(1))

    return pd.NA


def add_row(rows, procedure, committee, designation, head, value):
    if pd.isna(value):
        return
    value = norm_space(value)
    if not value:
        return
    rows.append(
        {
            "procedure": procedure,
            "committee": committee,
            "designation": designation,
            "head": head,
            "value": value,
        }
    )


# ---------------------------------------------------------------------
# main builder: only procedures whose contents contain PROCEDURAL PAGE
# ---------------------------------------------------------------------
def build_lead_procedural_meta(df_page_text, contents_df, dff_first_page_meta):
    page_ref_col = "reference" if "reference" in df_page_text.columns else "web_reference"
    meta_ref_col = "web_reference" if "web_reference" in dff_first_page_meta.columns else "reference"

    eligible_contents = contents_df.loc[
        contents_df["item"].map(normalize_for_match).str.startswith("PROCEDURAL PAGE")
    ].copy()

    rows = []

    for procedure, proc_cdf in eligible_contents.groupby("procedure", sort=False):
        target_page = int(proc_cdf["page_number"].iloc[0])

        meta_match = dff_first_page_meta.loc[dff_first_page_meta["procedure"] == procedure].copy()
        if meta_match.empty:
            continue

        meta_row = meta_match.iloc[0]
        web_reference = meta_row[meta_ref_col]

        proc_pages = df_page_text.loc[
            (df_page_text["procedure"] == procedure)
            & (df_page_text[page_ref_col] == web_reference)
        ].copy()

        if proc_pages.empty:
            proc_pages = df_page_text.loc[df_page_text["procedure"] == procedure].copy()

        if proc_pages.empty:
            continue

        proc_pages["page"] = pd.to_numeric(proc_pages["page"], errors="coerce")
        proc_pages = proc_pages.dropna(subset=["page"]).copy()

        located_page, procedural_text = locate_procedural_page(proc_pages, target_page)

        if pd.isna(located_page) or not procedural_text:
            continue

        print(f"located for lead committee in procedure == {procedure}")

        committee = meta_row.get("committee", pd.NA)
        designation = "lead"

        # -------------------------------------------------------------
        # seed from first-page metadata
        # -------------------------------------------------------------
        add_row(rows, procedure, committee, designation, "Title", meta_row.get("title", pd.NA))

        references = parse_references_field(meta_row.get("references", []))
        for ref in references:
            add_row(rows, procedure, committee, designation, "References", ref)

        rapporteur_clean = strip_honorifics(meta_row.get("rapporteur", pd.NA))
        add_row(rows, procedure, committee, designation, "Rapporteurs", rapporteur_clean)

        add_row(rows, procedure, committee, designation, "Committee responsible", committee)

        # -------------------------------------------------------------
        # parse procedural page
        # -------------------------------------------------------------
        date_appointed = extract_date_appointed(procedural_text, rapporteur_clean)
        add_row(rows, procedure, committee, designation, "Date appointed", date_appointed)

        discussed_dates = extract_discussed_dates(procedural_text)
        for d in discussed_dates:
            add_row(rows, procedure, committee, designation, "Discussed in committee", d)

        adoption_found, adopted_date, vote_rows = extract_vote_and_adoption(procedural_text, discussed_dates)
        if adoption_found:
            add_row(rows, procedure, committee, designation, "Date adopted", adopted_date)
            for head, value in vote_rows:
                add_row(rows, procedure, committee, designation, head, value)

        members_present = extract_members_present(procedural_text)
        add_row(
            rows,
            procedure,
            committee,
            designation,
            "Members present for the final vote",
            members_present,
        )

    out = pd.DataFrame(rows, columns=["procedure", "committee", "designation", "head", "value"])

    if not out.empty:
        out = out.drop_duplicates().reset_index(drop=True)

    return out


# ---------------------------------------------------------------------
# run
# ---------------------------------------------------------------------
df_committee_meta_long_new = build_lead_procedural_meta(
    df_page_text=df_page_text,
    contents_df=contents_df,
    dff_first_page_meta=dff_first_page_meta,
)

# append into an existing long table if you already have one
if "df_committee_meta_long" in globals():
    df_committee_meta_long = (
        pd.concat([df_committee_meta_long, df_committee_meta_long_new], ignore_index=True)
        .drop_duplicates()
        .reset_index(drop=True)
    )
else:
    df_committee_meta_long = df_committee_meta_long_new.copy()

display(df_committee_meta_long_new)

located for lead committee in procedure == 1994/0272(COD)
located for lead committee in procedure == 2000/0073(COD)


,procedure,committee,designation,head,value
0,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Title,energy efficiency requirements for household e...
1,1994/0272(COD),"Committee on Research, Technological Developme...",lead,References,COM(94)O521
2,1994/0272(COD),"Committee on Research, Technological Developme...",lead,References,C4-OO2O/95
3,1994/0272(COD),"Committee on Research, Technological Developme...",lead,References,94/O272(COD)
4,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Rapporteurs,Allan Macartney
5,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Committee responsible,"Committee on Research, Technological Developme..."
6,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Date appointed,5 October 1994
7,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Discussed in committee,24 April 1995
8,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Discussed in committee,30 May 1995
9,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Discussed in committee,20 June 1995


In [65]:
import ast
import re
import unicodedata

import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------
MONTH_MAP = {
    "january": "January",
    "february": "February",
    "march": "March",
    "april": "April",
    "may": "May",
    "june": "June",
    "july": "July",
    "august": "August",
    "september": "September",
    "october": "October",
    "november": "November",
    "december": "December",
}
MONTH_PATTERN = r"(January|February|March|April|May|June|July|August|September|October|November|December)"

NUMBER_WORDS = {
    "none": 0,
    "zero": 0,
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "seven": 7,
    "eight": 8,
    "nine": 9,
    "ten": 10,
}


def as_text(x):
    if pd.isna(x):
        return ""
    return str(x).replace("\xa0", " ")


def strip_ornaments(text):
    text = as_text(text)
    text = re.sub(r"[╔╗║╚╝═]", " ", text)
    text = re.sub(r"[★☆■□▪▫◆◇•·]+", " ", text)
    return text


def norm_space(text):
    return re.sub(r"\s+", " ", strip_ornaments(text)).strip()


def normalize_for_match(text):
    text = strip_ornaments(text)
    text = text.replace("–", "-").replace("—", "-").replace("−", "-")
    text = re.sub(r"\s+", " ", text).strip().upper()
    return text


def ascii_fold(text):
    text = unicodedata.normalize("NFKD", as_text(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text


def flatten_text(text):
    text = strip_ornaments(text)
    text = text.replace("–", "-").replace("—", "-").replace("−", "-")
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]
    return re.sub(r"\s+", " ", " ".join(lines)).strip()


def split_sentences(text):
    flat = flatten_text(text)
    if not flat:
        return []
    parts = re.split(r"(?<=[\.\?!])\s+", flat)
    return [part.strip() for part in parts if part.strip()]


def strip_honorifics(name):
    name = norm_space(name)
    name = re.sub(
        r"^(Mr|Mrs|Ms|Miss|Mme|M\.|Sir|Dr|Professor|Prof\.)\s+",
        "",
        name,
        flags=re.IGNORECASE,
    )
    return norm_space(name)


def parse_references_field(x):
    """
    Works if references is already a list, tuple, or a string representation of a list.
    """
    if isinstance(x, (list, tuple)):
        return [norm_space(v) for v in x if norm_space(v)]

    s = as_text(x).strip()
    if not s:
        return []

    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [norm_space(v) for v in parsed if norm_space(v)]
        except Exception:
            pass

    return [norm_space(s)]


def add_row(rows, procedure, committee, designation, head, value):
    if pd.isna(value):
        return
    value = norm_space(value)
    if not value:
        return
    rows.append(
        {
            "procedure": procedure,
            "committee": committee,
            "designation": designation,
            "head": head,
            "value": value,
        }
    )


# ---------------------------------------------------------------------
# page location / scoring
# ---------------------------------------------------------------------
def lead_page_score(text):
    s = ascii_fold(flatten_text(text)).lower()
    score = 0

    if "procedural page" in s:
        score += 100
    if "appointed" in s and "rapporteur" in s:
        score += 30
    if "considered the commission proposal" in s:
        score += 30
    if "adopted the draft legislative resolution" in s:
        score += 30
    if "the following were present for the vote:" in s:
        score += 20
    if "the following took part in the vote/were present for the vote:" in s:
        score += 20
    if "the opinions of" in s:
        score += 10

    return score


def opinion_start_page_score(text, committee_name):
    s = ascii_fold(flatten_text(text)).lower()
    c = ascii_fold(committee_name).lower()
    score = 0

    if "opinion of the committee on" in s and c in s:
        score += 120
    if c in s:
        score += 60
    if "draftsman" in s:
        score += 25
    if re.search(r"\bprocedure\b", s):
        score += 15
    if "considered the draft opinion" in s:
        score += 20
    if "appointed" in s and "draftsman" in s:
        score += 20
    if "the following were present for the vote:" in s:
        score += 10
    if "the following took part in the vote:" in s:
        score += 10

    return score


def locate_page_with_window(proc_pages, target_page, score_func):
    candidate_pages = proc_pages.loc[
        proc_pages["page"].between(target_page - 1, target_page + 1, inclusive="both")
    ].copy()

    if candidate_pages.empty:
        return pd.NA, ""

    candidate_pages["score"] = candidate_pages["text"].map(score_func)
    candidate_pages["distance"] = (candidate_pages["page"] - target_page).abs()

    candidate_pages = candidate_pages.sort_values(
        ["score", "distance", "page"],
        ascending=[False, True, True],
        kind="stable"
    )

    best = candidate_pages.iloc[0]
    if best["score"] <= 0:
        return pd.NA, ""

    return int(best["page"]), as_text(best["text"])


def locate_lead_procedural_page(proc_pages, target_page):
    return locate_page_with_window(proc_pages, target_page, lead_page_score)


def locate_opinion_start_page(proc_pages, target_page, committee_name):
    return locate_page_with_window(
        proc_pages,
        target_page,
        lambda text: opinion_start_page_score(text, committee_name)
    )


def find_procedure_subsection_in_page(text):
    """
    Return subtext from PROCEDURE heading onward if present on this page,
    otherwise return empty string.
    """
    lines = as_text(text).splitlines()
    for i, line in enumerate(lines):
        if normalize_for_match(line) == "PROCEDURE":
            return "\n".join(lines[i:]).strip()
    return ""


def build_page_lookup(proc_pages):
    return {
        int(row["page"]): as_text(row["text"])
        for _, row in proc_pages.iterrows()
        if pd.notna(row["page"])
    }


def locate_opinion_procedure_section(proc_pages, start_page):
    """
    Preferred:
    - from opinion start page to start_page+2, find page with explicit PROCEDURE heading
    Fallback:
    - use the opinion start page text itself (older 1995-style case)
    """
    page_lookup = build_page_lookup(proc_pages)

    for p in range(start_page, start_page + 3):
        txt = page_lookup.get(p, "")
        subsection = find_procedure_subsection_in_page(txt)
        if subsection:
            return p, subsection, True

    # fallback: older style with no explicit PROCEDURE heading
    txt = page_lookup.get(start_page, "")
    if txt:
        return start_page, txt, False

    return pd.NA, "", False


# ---------------------------------------------------------------------
# dates / names / roles
# ---------------------------------------------------------------------
def fix_ocr_day_zeroes(text):
    """
    Fix common OCR issues like:
    3O May -> 30 May
    2O June -> 20 June
    """
    text = as_text(text)
    text = re.sub(
        rf"\b(\d)O(?=\s+{MONTH_PATTERN}\b)",
        lambda m: f"{m.group(1)}0",
        text,
        flags=re.IGNORECASE,
    )
    return text


def canonical_date(day, month, year):
    return f"{int(day)} {MONTH_MAP[month.lower()]} {int(year)}"


def parse_date_string_to_ts(date_str):
    return pd.to_datetime(date_str, dayfirst=True, errors="coerce")


def extract_dates_from_sentence(text):
    """
    Handles:
    - 23 May 2000
    - 24 April, 30 May, 20 June, 18 July and 26-27 September 1995
    - 12 September, 19 September and 17 October 2000
    """
    text = fix_ocr_day_zeroes(text)
    text = text.replace("–", "-").replace("—", "-").replace("−", "-")
    text = norm_space(text)

    base_year_match = re.findall(r"\b((?:19|20)\d{2})\b", text)
    base_year = base_year_match[-1] if base_year_match else None

    results = []

    pattern = re.compile(
        rf"""
        (\b\d{{1,2}}\s*-\s*\d{{1,2}}\s+{MONTH_PATTERN}(?:\s+(?:19|20)\d{{2}})?\b)
        |
        (\b\d{{1,2}}\s+{MONTH_PATTERN}(?:\s+(?:19|20)\d{{2}})?\b)
        """,
        flags=re.IGNORECASE | re.VERBOSE,
    )

    for m in pattern.finditer(text):
        frag = m.group(0).strip()

        range_match = re.match(
            rf"^\b(\d{{1,2}})\s*-\s*(\d{{1,2}})\s+{MONTH_PATTERN}(?:\s+((?:19|20)\d{{2}}))?\b$",
            frag,
            flags=re.IGNORECASE,
        )
        if range_match:
            start_day = int(range_match.group(1))
            end_day = int(range_match.group(2))
            month = range_match.group(3)
            year = range_match.group(4) or base_year
            if year:
                for day in range(start_day, end_day + 1):
                    results.append(canonical_date(day, month, year))
            continue

        single_match = re.match(
            rf"^\b(\d{{1,2}})\s+{MONTH_PATTERN}(?:\s+((?:19|20)\d{{2}}))?\b$",
            frag,
            flags=re.IGNORECASE,
        )
        if single_match:
            day = int(single_match.group(1))
            month = single_match.group(2)
            year = single_match.group(3) or base_year
            if year:
                results.append(canonical_date(day, month, year))

    deduped = []
    seen = set()
    for d in results:
        if d not in seen:
            seen.add(d)
            deduped.append(d)

    return deduped


def person_name_variants(name):
    clean = strip_honorifics(name)
    if not clean:
        return []

    full = ascii_fold(clean).lower()
    parts = clean.split()
    surname = ascii_fold(parts[-1]).lower() if parts else ""

    variants = [full]
    if surname and surname not in variants:
        variants.append(surname)

    return variants


def extract_role_holder_from_label(text, role_word):
    """
    Example:
    Draftsman: Mr Michael Spindelegger
    """
    flat = flatten_text(text)
    m = re.search(
        rf"\b{role_word}\s*:\s*(.+?)(?=At its meeting of|At its meetings of|The Committee on|$)",
        flat,
        flags=re.IGNORECASE,
    )
    if not m:
        return pd.NA
    return strip_honorifics(m.group(1))


def extract_role_holder_and_date(section_text, role_word, known_name=pd.NA):
    """
    Works for both rapporteur and draftsman.
    Returns (name, date_appointed)
    """
    role_word = role_word.lower()
    label_name = extract_role_holder_from_label(section_text, role_word) if role_word == "draftsman" else pd.NA
    chosen_name = strip_honorifics(known_name) if pd.notna(known_name) else pd.NA

    if pd.isna(chosen_name) and pd.notna(label_name):
        chosen_name = label_name

    variants = person_name_variants(chosen_name) if pd.notna(chosen_name) else []

    for sent in split_sentences(section_text):
        sent_fold = ascii_fold(sent).lower()
        if "appoint" not in sent_fold or role_word not in sent_fold:
            continue

        # if we know the person, enforce match
        if variants and not any(v in sent_fold for v in variants):
            continue

        # if we do not know the person yet, derive from sentence
        if pd.isna(chosen_name):
            m = re.search(
                rf"appointed\s+(.+?)\s+{role_word}\b",
                sent,
                flags=re.IGNORECASE,
            )
            if m:
                chosen_name = strip_honorifics(m.group(1))

        dates = extract_dates_from_sentence(sent)
        if dates:
            return chosen_name, dates[0]

    return chosen_name, pd.NA


# ---------------------------------------------------------------------
# discussion / adoption / votes / members
# ---------------------------------------------------------------------
def sentence_is_discussion_sentence(sent):
    s = ascii_fold(sent).lower()

    if "considered" not in s:
        return False

    discussion_markers = [
        "commission proposal",
        "draft report",
        "draft opinion",
        "draft opinions",
    ]

    return any(marker in s for marker in discussion_markers)


def extract_discussed_dates(section_text):
    results = []

    for sent in split_sentences(section_text):
        if not sentence_is_discussion_sentence(sent):
            continue
        results.extend(extract_dates_from_sentence(sent))

    deduped = []
    seen = set()
    for d in results:
        if d not in seen:
            seen.add(d)
            deduped.append(d)

    return deduped


def normalize_number_token(tok):
    tok = as_text(tok).strip().lower()
    if tok.isdigit():
        return str(int(tok))
    if tok in NUMBER_WORDS:
        return str(NUMBER_WORDS[tok])
    return tok


def parse_vote_rows_from_sentence(sent):
    s = ascii_fold(sent).lower()

    if "unanimously" in s:
        return [("Result of the vote", "unanimous")]

    sent_clean = fix_ocr_day_zeroes(sent)
    sent_clean = sent_clean.replace("–", "-").replace("—", "-").replace("−", "-")
    num = r"(\d+|none|zero|one|two|three|four|five|six|seven|eight|nine|ten)"

    patterns = [
        # by 48 votes to 1, with 1 abstention
        rf"by\s+{num}\s+votes?\s+to\s*{num}\s*,?\s*with\s+{num}\s+abstentions?",
        # by 38 votes to 8
        rf"by\s+{num}\s+votes?\s+to\s*{num}",
        # by 27 votes, with one abstention
        rf"by\s+{num}\s+votes?\s*,?\s*with\s+{num}\s+abstentions?",
    ]

    for pat in patterns:
        m = re.search(pat, sent_clean, flags=re.IGNORECASE)
        if not m:
            continue

        groups = [normalize_number_token(g) for g in m.groups()]

        if len(groups) == 3:
            return [
                ("Result of final vote +", groups[0]),
                ("Result of final vote –", groups[1]),
                ("Result of final vote 0", groups[2]),
            ]
        if len(groups) == 2 and "to" in m.group(0).lower():
            return [
                ("Result of final vote +", groups[0]),
                ("Result of final vote –", groups[1]),
            ]
        if len(groups) == 2:
            return [
                ("Result of final vote +", groups[0]),
                ("Result of final vote 0", groups[1]),
            ]

    return []


def sentence_is_adoption_sentence(sent, designation):
    s = ascii_fold(sent).lower()

    if "adopt" not in s:
        return False

    if designation == "lead":
        return "adopted the draft legislative resolution" in s

    # opinion committee
    opinion_markers = [
        "adopted the conclusions",
        "adopted the amendments below",
        "adopted the draft opinion",
    ]
    return any(marker in s for marker in opinion_markers)


def extract_vote_and_adoption(section_text, discussed_dates, designation):
    adoption_found = False
    adopted_date = pd.NA
    vote_rows = []

    for sent in split_sentences(section_text):
        if not sentence_is_adoption_sentence(sent, designation):
            continue

        adoption_found = True

        if discussed_dates:
            parsed = [(d, parse_date_string_to_ts(d)) for d in discussed_dates]
            parsed = [(d, ts) for d, ts in parsed if pd.notna(ts)]
            if parsed:
                adopted_date = max(parsed, key=lambda x: x[1])[0]

        vote_rows = parse_vote_rows_from_sentence(sent)
        return adoption_found, adopted_date, vote_rows

    return adoption_found, adopted_date, vote_rows


def extract_members_present(section_text):
    flat = flatten_text(section_text)

    patterns = [
        r"The following took part in the vote/were present for the vote:\s*(.*?)(?=The opinions of|The report was tabled on|The deadline for tabling amendments|AMENDMENTS|SHORT JUSTIFICATION|Commission proposal|POSITION OF THE COMMISSION|$)",
        r"The following were present for the vote:\s*(.*?)(?=The opinions of|The report was tabled on|The deadline for tabling amendments|AMENDMENTS|SHORT JUSTIFICATION|Commission proposal|POSITION OF THE COMMISSION|$)",
        r"The following took part in the vote:\s*(.*?)(?=The opinions of|The report was tabled on|The deadline for tabling amendments|AMENDMENTS|SHORT JUSTIFICATION|Commission proposal|POSITION OF THE COMMISSION|$)",
    ]

    for pat in patterns:
        m = re.search(pat, flat, flags=re.IGNORECASE | re.DOTALL)
        if m:
            return norm_space(m.group(1))

    return pd.NA


# ---------------------------------------------------------------------
# contents / committee matching
# ---------------------------------------------------------------------
def find_opinion_contents_row(proc_contents, committee_name):
    """
    Find the contents row corresponding to:
    OPINION OF THE COMMITTEE ON <committee_name>
    """
    target = normalize_for_match(committee_name)

    opinion_rows = proc_contents.loc[
        proc_contents["item"].map(normalize_for_match).str.startswith("OPINION OF THE ")
    ].copy()

    if opinion_rows.empty:
        return pd.DataFrame()

    opinion_rows["committee_extracted"] = opinion_rows["item"].map(
        lambda x: normalize_for_match(re.sub(r"^OPINION OF THE\s+", "", normalize_for_match(x)))
    )

    matched = opinion_rows.loc[opinion_rows["committee_extracted"] == target].copy()
    return matched


# ---------------------------------------------------------------------
# builders
# ---------------------------------------------------------------------
def build_lead_rows_from_first_page_meta(rows, procedure, meta_row):
    committee = meta_row.get("committee", pd.NA)
    designation = "lead"

    add_row(rows, procedure, committee, designation, "Title", meta_row.get("title", pd.NA))

    references = parse_references_field(meta_row.get("references", []))
    for ref in references:
        add_row(rows, procedure, committee, designation, "References", ref)

    rapporteur_clean = strip_honorifics(meta_row.get("rapporteur", pd.NA))
    add_row(rows, procedure, committee, designation, "Rapporteurs", rapporteur_clean)
    add_row(rows, procedure, committee, designation, "Committee responsible", committee)

    return rapporteur_clean


def parse_committee_section(rows, procedure, committee, designation, section_text, known_person_name=pd.NA):
    role_word = "rapporteur" if designation == "lead" else "draftsman"

    person_name, date_appointed = extract_role_holder_and_date(
        section_text=section_text,
        role_word=role_word,
        known_name=known_person_name,
    )

    add_row(rows, procedure, committee, designation, "Rapporteurs", person_name)
    add_row(rows, procedure, committee, designation, "Date appointed", date_appointed)

    discussed_dates = extract_discussed_dates(section_text)
    for d in discussed_dates:
        add_row(rows, procedure, committee, designation, "Discussed in committee", d)

    adoption_found, adopted_date, vote_rows = extract_vote_and_adoption(
        section_text=section_text,
        discussed_dates=discussed_dates,
        designation=designation,
    )

    if adoption_found:
        add_row(rows, procedure, committee, designation, "Date adopted", adopted_date)
        for head, value in vote_rows:
            add_row(rows, procedure, committee, designation, head, value)

    members_present = extract_members_present(section_text)
    add_row(rows, procedure, committee, designation, "Members present for the final vote", members_present)


def build_committee_meta_long_procedural_universe(
    df_page_text,
    contents_df,
    dff_first_page_meta,
    df_committees,
):
    """
    Strict universe only:
    procedures whose contents_df contains PROCEDURAL PAGE.

    For those procedures:
    - populate lead committee metadata
    - populate opinion committee metadata
    """
    page_ref_col = "reference" if "reference" in df_page_text.columns else "web_reference"
    meta_ref_col = "web_reference" if "web_reference" in dff_first_page_meta.columns else "reference"

    eligible_contents = contents_df.loc[
        contents_df["item"].map(normalize_for_match).str.startswith("PROCEDURAL PAGE")
    ].copy()

    eligible_procedures = set(eligible_contents["procedure"].dropna().unique())

    rows = []

    for procedure in eligible_procedures:
        proc_contents = contents_df.loc[contents_df["procedure"] == procedure].copy()
        meta_match = dff_first_page_meta.loc[dff_first_page_meta["procedure"] == procedure].copy()

        if meta_match.empty:
            continue

        meta_row = meta_match.iloc[0]
        web_reference = meta_row[meta_ref_col]

        proc_pages = df_page_text.loc[
            (df_page_text["procedure"] == procedure)
            & (df_page_text[page_ref_col] == web_reference)
        ].copy()

        if proc_pages.empty:
            proc_pages = df_page_text.loc[df_page_text["procedure"] == procedure].copy()

        if proc_pages.empty:
            continue

        proc_pages["page"] = pd.to_numeric(proc_pages["page"], errors="coerce")
        proc_pages = proc_pages.dropna(subset=["page"]).copy()

        # =============================================================
        # LEAD COMMITTEE
        # =============================================================
        lead_contents = proc_contents.loc[
            proc_contents["item"].map(normalize_for_match).str.startswith("PROCEDURAL PAGE")
        ].copy()

        if not lead_contents.empty:
            lead_target_page = int(lead_contents["page_number"].iloc[0])
            located_page, lead_text = locate_lead_procedural_page(proc_pages, lead_target_page)

            if pd.notna(located_page) and lead_text:
                print(f"located for lead committee in procedure == {procedure}")

                lead_committee = meta_row.get("committee", pd.NA)
                known_rapporteur = build_lead_rows_from_first_page_meta(rows, procedure, meta_row)

                parse_committee_section(
                    rows=rows,
                    procedure=procedure,
                    committee=lead_committee,
                    designation="lead",
                    section_text=lead_text,
                    known_person_name=known_rapporteur,
                )

        # =============================================================
        # OPINION COMMITTEES
        # =============================================================
        proc_committees = df_committees.loc[
            (df_committees["procedure"] == procedure)
            & (df_committees["designation"].str.lower() == "opinion")
        ].copy()

        if proc_committees.empty:
            continue

        for _, c_row in proc_committees.iterrows():
            opinion_committee = c_row["committee"]

            matched_contents = find_opinion_contents_row(proc_contents, opinion_committee)
            if matched_contents.empty:
                continue

            opinion_target_page = int(matched_contents["page_number"].iloc[0])

            opinion_start_page, opinion_start_text = locate_opinion_start_page(
                proc_pages=proc_pages,
                target_page=opinion_target_page,
                committee_name=opinion_committee,
            )

            if pd.isna(opinion_start_page) or not opinion_start_text:
                continue

            section_page, opinion_section_text, explicit_procedure_heading = locate_opinion_procedure_section(
                proc_pages=proc_pages,
                start_page=opinion_start_page,
            )

            if pd.isna(section_page) or not opinion_section_text:
                continue

            print(f"found opinion section for committee == {opinion_committee} in procedure == {procedure}")

            parse_committee_section(
                rows=rows,
                procedure=procedure,
                committee=opinion_committee,
                designation="opinion",
                section_text=opinion_section_text,
                known_person_name=pd.NA,
            )

    out = pd.DataFrame(rows, columns=["procedure", "committee", "designation", "head", "value"])

    if not out.empty:
        out = out.drop_duplicates().reset_index(drop=True)

    return out


# ---------------------------------------------------------------------
# run
# ---------------------------------------------------------------------
df_committee_meta_long_new = build_committee_meta_long_procedural_universe(
    df_page_text=df_page_text,
    contents_df=contents_df,
    dff_first_page_meta=dff_first_page_meta,
    df_committees=df_committees,
)

if "df_committee_meta_long" in globals():
    df_committee_meta_long = (
        pd.concat([df_committee_meta_long, df_committee_meta_long_new], ignore_index=True)
        .drop_duplicates()
        .reset_index(drop=True)
    )
else:
    df_committee_meta_long = df_committee_meta_long_new.copy()

display(df_committee_meta_long_new)

located for lead committee in procedure == 1994/0272(COD)
found opinion section for committee == Committee on Economic and Monetary Affairs and Industrial Policy in procedure == 1994/0272(COD)
found opinion section for committee == Committee on the Environment, Public Health and Consumer Protection in procedure == 1994/0272(COD)
located for lead committee in procedure == 2000/0073(COD)
found opinion section for committee == COMMITTEE ON INDUSTRY, EXTERNAL TRADE, RESEARCH AND ENERGY in procedure == 2000/0073(COD)
found opinion section for committee == COMMITTEE ON LEGAL AFFAIRS AND THE INTERNAL MARKET in procedure == 2000/0073(COD)


,procedure,committee,designation,head,value
0,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Title,energy efficiency requirements for household e...
1,1994/0272(COD),"Committee on Research, Technological Developme...",lead,References,COM(94)O521
2,1994/0272(COD),"Committee on Research, Technological Developme...",lead,References,C4-OO2O/95
3,1994/0272(COD),"Committee on Research, Technological Developme...",lead,References,94/O272(COD)
4,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Rapporteurs,Allan Macartney
5,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Committee responsible,"Committee on Research, Technological Developme..."
6,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Date appointed,5 October 1994
7,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Discussed in committee,24 April 1995
8,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Discussed in committee,30 May 1995
9,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Discussed in committee,20 June 1995


In [67]:
df_dict = pd.read_csv('bld\committee_dictionary.csv')

In [68]:
df_dict

,committee_code,committee,committee_type
0,EUDS,European Democracy Shield,Special committee
1,HOUS,Housing Crisis in the EU,Special committee
2,AFCO,Constitutional Affairs,Standing committee
3,AFET,Foreign Affairs,Standing committee
4,AGRI,Agriculture and Rural Development,Standing committee
5,BUDG,Budgets,Standing committee
6,CONT,Budgetary Control,Standing committee
7,CULT,Culture and Education,Standing committee
8,DEVE,Development,Standing committee
9,DROI,Human Rights,Standing committee


In [69]:
import re
import unicodedata
import pandas as pd
from IPython.display import display

# ============================================================
# structured PDF procedure-table parser
# universe:
#   contents contains either
#   - PROCEDURE
#   - PROCEDURE – COMMITTEE RESPONSIBLE
#
# output:
#   same long format as before:
#   procedure, committee, designation, head, value
#
# this is appended beneath the prose-parser output already built
# ============================================================

# ------------------------------------------------------------
# knobs
# ------------------------------------------------------------
STRUCTURED_HARD_ASSERT = False  # set True if you want mismatch to raise immediately


# ------------------------------------------------------------
# small helpers
# ------------------------------------------------------------
def st_as_text(x):
    if pd.isna(x):
        return ""
    return str(x).replace("\xa0", " ")


def st_strip_ornaments(text):
    text = st_as_text(text)
    text = re.sub(r"[╔╗║╚╝═]", " ", text)
    text = re.sub(r"[★☆■□▪▫◆◇•·]+", " ", text)
    return text


def st_norm_space(text):
    return re.sub(r"\s+", " ", st_strip_ornaments(text)).strip()


def st_normalize_for_match(text):
    text = st_strip_ornaments(text)
    text = (
        text.replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
        .replace("’", "'")
        .replace("‘", "'")
    )
    text = re.sub(r"\s+", " ", text).strip().upper()
    return text


def st_ascii_fold(text):
    text = unicodedata.normalize("NFKD", st_as_text(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text


def st_flatten_text(text):
    text = st_strip_ornaments(text)
    text = (
        text.replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
        .replace("’", "'")
        .replace("‘", "'")
    )
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]
    return re.sub(r"\s+", " ", " ".join(lines)).strip()


def st_add_row(rows, procedure, committee, designation, head, value):
    if pd.isna(value):
        return
    value = st_norm_space(value)
    if not value:
        return
    rows.append(
        {
            "procedure": procedure,
            "committee": committee,
            "designation": designation,
            "head": head,
            "value": value,
        }
    )


# ------------------------------------------------------------
# committee-code helpers
# ------------------------------------------------------------
def st_build_committee_code_lookup(df_dict):
    out = {}
    for _, row in df_dict.iterrows():
        code = st_norm_space(row.get("committee_code", "")).upper()
        committee = st_norm_space(row.get("committee", ""))
        if code:
            out[code] = committee
    return out


def st_normalize_committee_name_for_compare(text):
    s = st_ascii_fold(st_as_text(text)).lower()
    s = s.replace("&", " and ")
    s = s.replace("’", "'").replace("‘", "'")
    s = re.sub(r"^annex:\s*", "", s)
    s = re.sub(r"^opinion of the\s+", "", s)
    s = re.sub(r"^committee on the\s+", "", s)
    s = re.sub(r"^committee on\s+", "", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def st_committee_code_matches_expected(expected_committee, code, committee_code_lookup):
    """
    expected_committee comes from df_committees (outer committee column we want to keep)
    code comes from the structured table (Committee responsible or Opinion by)
    committee_code_lookup comes from df_dict

    returns:
        (is_match, mapped_committee_name_from_dict)
    """
    code = st_norm_space(code).upper()
    mapped = committee_code_lookup.get(code, "")

    if not mapped:
        return False, ""

    exp_norm = st_normalize_committee_name_for_compare(expected_committee)
    map_norm = st_normalize_committee_name_for_compare(mapped)

    # strong exact / contains checks
    if exp_norm == map_norm or exp_norm in map_norm or map_norm in exp_norm:
        return True, mapped

    # softer token overlap check
    stop = {"and", "the", "of", "on", "for", "in", "to"}
    exp_tokens = {t for t in exp_norm.split() if t not in stop}
    map_tokens = {t for t in map_norm.split() if t not in stop}

    overlap = len(exp_tokens & map_tokens)
    # intentionally conservative
    if overlap >= 2:
        return True, mapped

    return False, mapped


def st_assert_or_warn_committee_match(
    procedure,
    designation,
    expected_committee,
    code,
    committee_code_lookup,
):
    ok, mapped = st_committee_code_matches_expected(
        expected_committee=expected_committee,
        code=code,
        committee_code_lookup=committee_code_lookup,
    )

    if ok:
        return True

    msg = (
        f"[structured-parser] committee-code mismatch in procedure {procedure} "
        f"({designation}). code={code!r}, df_dict={mapped!r}, expected outer committee={expected_committee!r}"
    )

    if STRUCTURED_HARD_ASSERT:
        raise AssertionError(msg)

    print(msg)
    return False


# ------------------------------------------------------------
# locating the structured table pages
# ------------------------------------------------------------
def st_structured_table_page_score(text):
    s = st_ascii_fold(st_flatten_text(text)).lower()
    score = 0

    if "procedure - committee responsible" in s:
        score += 120
    if re.search(r"\bprocedure\b", s):
        score += 80
    if "title " in s:
        score += 20
    if "references " in s:
        score += 20
    if "date adopted" in s:
        score += 10
    if "result of final vote" in s:
        score += 20
    if "members present for the final vote" in s:
        score += 20
    if "committee responsible" in s:
        score += 10

    return score


def st_build_page_lookup(proc_pages):
    out = {}
    for _, row in proc_pages.iterrows():
        if pd.notna(row["page"]):
            out[int(row["page"])] = st_as_text(row["text"])
    return out


def st_locate_lead_structured_page(proc_pages, target_page):
    cand = proc_pages.loc[
        proc_pages["page"].between(target_page - 1, target_page + 1, inclusive="both")
    ].copy()

    if cand.empty:
        return pd.NA, ""

    cand["score"] = cand["text"].map(st_structured_table_page_score)
    cand["distance"] = (cand["page"] - target_page).abs()

    cand = cand.sort_values(
        ["score", "distance", "page"],
        ascending=[False, True, True],
        kind="stable",
    )

    best = cand.iloc[0]
    if best["score"] <= 0:
        return pd.NA, ""

    return int(best["page"]), st_as_text(best["text"])


def st_locate_opinion_structured_page(proc_pages, section_start_page, section_end_page):
    """
    Search backwards inside the opinion section.
    The PROCEDURE page is usually the last or second-last page of the section.
    """
    page_lookup = st_build_page_lookup(proc_pages)

    candidates = []
    for p in range(section_end_page, section_start_page - 1, -1):
        txt = page_lookup.get(p, "")
        if not txt:
            continue
        score = st_structured_table_page_score(txt)
        if score > 0:
            candidates.append((p, score, txt))

    if not candidates:
        return pd.NA, ""

    # prefer later page; among equally late, stronger score
    candidates = sorted(candidates, key=lambda x: (x[0], x[1]), reverse=True)
    page, _, txt = candidates[0]
    return page, txt


# ------------------------------------------------------------
# contents-row matching
# ------------------------------------------------------------
def st_find_lead_procedure_contents_row(proc_contents):
    """
    Prefer PROCEDURE – COMMITTEE RESPONSIBLE.
    Fallback to PROCEDURE.
    """
    cdf = proc_contents.copy()
    cdf["item_norm"] = cdf["item"].map(st_normalize_for_match)

    preferred = cdf.loc[cdf["item_norm"].str.startswith("PROCEDURE - COMMITTEE RESPONSIBLE")]
    if not preferred.empty:
        preferred = preferred.sort_values("page_number", ascending=False, kind="stable")
        return preferred.iloc[[0]]

    fallback = cdf.loc[cdf["item_norm"].eq("PROCEDURE")]
    if not fallback.empty:
        fallback = fallback.sort_values("page_number", ascending=False, kind="stable")
        return fallback.iloc[[0]]

    return pd.DataFrame(columns=proc_contents.columns)


def st_find_opinion_contents_row(proc_contents, committee_name):
    """
    Match the opinion contents row for the current committee.
    """
    cdf = proc_contents.copy()
    cdf = cdf.loc[cdf["item"].map(st_normalize_for_match).str.startswith("OPINION OF THE")].copy()
    if cdf.empty:
        return pd.DataFrame(columns=proc_contents.columns)

    expected = st_normalize_committee_name_for_compare(committee_name)

    def score_item(item):
        item_norm = st_normalize_committee_name_for_compare(item)
        score = 0
        if expected == item_norm:
            score += 100
        if expected in item_norm or item_norm in expected:
            score += 80

        exp_tokens = set(expected.split())
        item_tokens = set(item_norm.split())
        score += len(exp_tokens & item_tokens)

        if st_normalize_for_match(item).startswith("OPINION OF THE"):
            score += 10

        return score

    cdf["score"] = cdf["item"].map(score_item)
    cdf = cdf.sort_values(["score", "page_number"], ascending=[False, True], kind="stable")

    best = cdf.iloc[0]
    if best["score"] <= 0:
        return pd.DataFrame(columns=proc_contents.columns)

    return cdf.iloc[[0]]


def st_get_section_end_page(proc_contents_sorted, current_page):
    """
    end page = page before the next contents item
    """
    later = proc_contents_sorted.loc[proc_contents_sorted["page_number"] > current_page].copy()
    if later.empty:
        return current_page
    return int(later["page_number"].iloc[0]) - 1


# ------------------------------------------------------------
# structured-page line parser
# ------------------------------------------------------------
STRUCTURED_HEAD_SPECS = [
    ("Title", "simple_text", re.compile(r"^(Title)\b\s*(.*)$", flags=re.IGNORECASE)),
    ("References", "simple_text", re.compile(r"^(References)\b\s*(.*)$", flags=re.IGNORECASE)),
    (
        "Date submitted to Parliament",
        "simple_text",
        re.compile(r"^(Date submitted to Parliament)\b\s*(.*)$", flags=re.IGNORECASE),
    ),
    (
        "Committee responsible",
        "paired",
        re.compile(r"^(Committee responsible)\b\s*(.*)$", flags=re.IGNORECASE),
    ),
    ("Opinion by", "paired", re.compile(r"^(Opinion by)\b\s*(.*)$", flags=re.IGNORECASE)),
    (
        "Committees asked for opinions",
        "paired",
        re.compile(
            r"^(Committees asked for opinions|Committee\(s\) asked for opinion\(s\))\b\s*(.*)$",
            flags=re.IGNORECASE,
        ),
    ),
    (
        "Associated committees",
        "paired",
        re.compile(r"^(Associated committees)\b\s*(.*)$", flags=re.IGNORECASE),
    ),
    (
        "Not delivering opinions",
        "paired",
        re.compile(r"^(Not delivering opinions)\b\s*(.*)$", flags=re.IGNORECASE),
    ),
    (
        "Rapporteur",
        "paired",
        re.compile(r"^(Rapporteurs?|Draftsman|Drafts\(wo\)man)\b\s*(.*)$", flags=re.IGNORECASE),
    ),
    (
        "Discussed in committee",
        "date_list",
        re.compile(r"^(Discussed in committee)\b\s*(.*)$", flags=re.IGNORECASE),
    ),
    (
        "Date adopted",
        "simple_text",
        re.compile(r"^(Date adopted)\b\s*(.*)$", flags=re.IGNORECASE),
    ),
    (
        "Result of final vote",
        "vote",
        re.compile(r"^(Result of final vote)\b\s*(.*)$", flags=re.IGNORECASE),
    ),
    (
        "Members present for the final vote",
        "list_block",
        re.compile(r"^(Members present for the final vote)\b\s*(.*)$", flags=re.IGNORECASE),
    ),
    (
        "Substitutes present for the final vote",
        "list_block",
        re.compile(
            r"^(Substitutes present for the final vote|Substitute\(s\) present for the final vote)\b\s*(.*)$",
            flags=re.IGNORECASE,
        ),
    ),
    (
        "Substitutes under Rule present for the final vote",
        "list_block",
        re.compile(
            r"^(Substitutes under Rule .*?present(?: for)? the final vote|Substitute\(s\) under Rule .*?present(?: for)? the final vote)\b\s*(.*)$",
            flags=re.IGNORECASE,
        ),
    ),
    ("Date tabled", "simple_text", re.compile(r"^(Date tabled)\b\s*(.*)$", flags=re.IGNORECASE)),
]


def st_detect_structured_head(line):
    line = st_norm_space(line)
    for _, kind, pat in STRUCTURED_HEAD_SPECS:
        m = pat.match(line)
        if m:
            display_head = st_norm_space(m.group(1))
            remainder = st_norm_space(m.group(2))
            return {
                "display_head": display_head,
                "kind": kind,
                "remainder": remainder,
            }
    return None


def st_make_logical_lines(page_text):
    raw_lines = [line.strip() for line in st_as_text(page_text).splitlines()]
    raw_lines = [line for line in raw_lines if line]

    # start from the PROCEDURE heading
    start_idx = None
    for i, line in enumerate(raw_lines):
        if st_normalize_for_match(line).startswith("PROCEDURE"):
            start_idx = i
            break

    if start_idx is None:
        return []

    body = raw_lines[start_idx + 1 :]
    logical = []

    i = 0
    while i < len(body):
        line = body[i].strip()
        if not line:
            i += 1
            continue

        # if current line alone is not a head, try combining with the next line
        # this helps for wrapped heads like:
        #   Substitutes under Rule 200(2) present
        #   for the final vote
        if st_detect_structured_head(line) is None and i + 1 < len(body):
            combined = f"{line} {body[i + 1].strip()}"
            if st_detect_structured_head(combined) is not None:
                logical.append(combined)
                i += 2
                continue

        logical.append(line)
        i += 1

    return logical


def st_extract_structured_blocks(page_text):
    lines = st_make_logical_lines(page_text)
    blocks = []

    i = 0
    while i < len(lines):
        detected = st_detect_structured_head(lines[i])
        if detected is None:
            i += 1
            continue

        block = {
            "display_head": detected["display_head"],
            "kind": detected["kind"],
            "content_lines": [],
        }

        if detected["remainder"]:
            block["content_lines"].append(detected["remainder"])

        i += 1
        while i < len(lines):
            if st_detect_structured_head(lines[i]) is not None:
                break
            block["content_lines"].append(st_norm_space(lines[i]))
            i += 1

        blocks.append(block)

    return blocks


def st_extract_dot_dates(text):
    return re.findall(r"\b\d{1,2}\.\d{1,2}\.\d{4}\b", st_as_text(text))


def st_parse_vote_block(display_head, content_lines):
    items = []
    symbols = []
    numbers = []

    sym_map = {"+": "+", "-": "–", "–": "–", "0": "0"}

    for line in content_lines:
        for part in [p.strip() for p in re.split(r"\s+", st_as_text(line)) if p.strip()]:
            part_clean = part.rstrip(":")
            if part_clean in sym_map and part.endswith(":"):
                symbols.append(sym_map[part_clean])
            elif re.fullmatch(r"\d+", part_clean):
                numbers.append(part_clean)

    for sym, num in zip(symbols, numbers):
        items.append({"head": f"{display_head} {sym}", "value": num})

    return items


def st_parse_paired_block(display_head, content_lines):
    items = []
    content_lines = [st_norm_space(x) for x in content_lines if st_norm_space(x)]
    if not content_lines:
        return items

    subhead = None
    first = st_normalize_for_match(content_lines[0])

    if first in {"DATE ANNOUNCED IN PLENARY", "DATE OF DECISION", "DATE APPOINTED"}:
        subhead = st_norm_space(content_lines[0])
        content_lines = content_lines[1:]

    if not content_lines:
        return items

    if subhead:
        # pair sequentially: code/name, date, code/name, date ...
        idx = 0
        while idx < len(content_lines):
            items.append({"head": display_head, "value": content_lines[idx]})
            if idx + 1 < len(content_lines):
                items.append({"head": subhead, "value": content_lines[idx + 1]})
            idx += 2
    else:
        for val in content_lines:
            items.append({"head": display_head, "value": val})

    return items


def st_parse_structured_block(block):
    display_head = block["display_head"]
    kind = block["kind"]
    content_lines = [st_norm_space(x) for x in block["content_lines"] if st_norm_space(x)]

    items = []

    if kind == "simple_text":
        value = st_norm_space(" ".join(content_lines))
        if value:
            items.append({"head": display_head, "value": value})
        return items

    if kind == "date_list":
        joined = st_norm_space(" ".join(content_lines))
        dates = st_extract_dot_dates(joined)
        if dates:
            for d in dates:
                items.append({"head": display_head, "value": d})
        elif joined:
            items.append({"head": display_head, "value": joined})
        return items

    if kind == "vote":
        return st_parse_vote_block(display_head, content_lines)

    if kind == "list_block":
        value = st_norm_space(" ".join(content_lines))
        if value:
            items.append({"head": display_head, "value": value})
        return items

    if kind == "paired":
        return st_parse_paired_block(display_head, content_lines)

    return items


def st_parse_structured_procedure_page(page_text):
    blocks = st_extract_structured_blocks(page_text)
    extracted = []

    for block in blocks:
        extracted.extend(st_parse_structured_block(block))

    return extracted


# ------------------------------------------------------------
# builder
# ------------------------------------------------------------
def build_structured_committee_meta_long_pdf(
    df_page_text,
    contents_df,
    df_committees,
    df_dict,
):
    page_ref_col = "reference" if "reference" in df_page_text.columns else "web_reference"
    committee_code_lookup = st_build_committee_code_lookup(df_dict)

    proc_contents_all = contents_df.copy()
    proc_contents_all["page_number"] = pd.to_numeric(proc_contents_all["page_number"], errors="coerce")
    proc_contents_all = proc_contents_all.dropna(subset=["page_number"]).copy()

    # structured universe:
    # contents has PROCEDURE or PROCEDURE – COMMITTEE RESPONSIBLE
    structured_contents_mask = proc_contents_all["item"].map(st_normalize_for_match).apply(
        lambda x: x == "PROCEDURE" or x.startswith("PROCEDURE - COMMITTEE RESPONSIBLE")
    )
    structured_procedures = proc_contents_all.loc[structured_contents_mask, "procedure"].dropna().unique().tolist()

    rows = []

    for procedure in structured_procedures:
        proc_contents = (
            proc_contents_all.loc[proc_contents_all["procedure"] == procedure]
            .sort_values("page_number", kind="stable")
            .copy()
        )

        proc_pages = df_page_text.loc[df_page_text["procedure"] == procedure].copy()
        if proc_pages.empty:
            continue

        proc_pages["page"] = pd.to_numeric(proc_pages["page"], errors="coerce")
        proc_pages = proc_pages.dropna(subset=["page"]).copy()

        # ----------------------------------------------------
        # LEAD TABLE
        # ----------------------------------------------------
        lead_rows = df_committees.loc[
            (df_committees["procedure"] == procedure)
            & (df_committees["designation"].str.lower() == "lead")
        ].copy()

        if not lead_rows.empty:
            expected_lead_committee = st_norm_space(lead_rows.iloc[0]["committee"])

            lead_contents_row = st_find_lead_procedure_contents_row(proc_contents)
            if not lead_contents_row.empty:
                target_page = int(lead_contents_row.iloc[0]["page_number"])

                located_page, lead_text = st_locate_lead_structured_page(proc_pages, target_page)
                if pd.notna(located_page) and lead_text:
                    print(f"found structured lead table in procedure == {procedure} on page == {located_page}")

                    extracted = st_parse_structured_procedure_page(lead_text)
                    extracted_df = pd.DataFrame(extracted)

                    if not extracted_df.empty:
                        lead_code_rows = extracted_df.loc[
                            extracted_df["head"].map(st_normalize_for_match) == "COMMITTEE RESPONSIBLE"
                        ]
                        if not lead_code_rows.empty:
                            lead_code = st_norm_space(lead_code_rows.iloc[0]["value"]).upper()
                            st_assert_or_warn_committee_match(
                                procedure=procedure,
                                designation="lead",
                                expected_committee=expected_lead_committee,
                                code=lead_code,
                                committee_code_lookup=committee_code_lookup,
                            )

                        for item in extracted:
                            st_add_row(
                                rows=rows,
                                procedure=procedure,
                                committee=expected_lead_committee,   # outer committee stays the df_committees name
                                designation="lead",
                                head=item["head"],
                                value=item["value"],
                            )

        # ----------------------------------------------------
        # OPINION TABLES
        # ----------------------------------------------------
        opinion_rows = df_committees.loc[
            (df_committees["procedure"] == procedure)
            & (df_committees["designation"].str.lower() == "opinion")
        ].copy()

        if opinion_rows.empty:
            continue

        for _, c_row in opinion_rows.iterrows():
            expected_opinion_committee = st_norm_space(c_row["committee"])

            opinion_contents_row = st_find_opinion_contents_row(proc_contents, expected_opinion_committee)
            if opinion_contents_row.empty:
                print(
                    f"[structured-parser] could not match opinion contents row for committee == {expected_opinion_committee} "
                    f"in procedure == {procedure}"
                )
                continue

            section_start_page = int(opinion_contents_row.iloc[0]["page_number"])
            section_end_page = st_get_section_end_page(proc_contents, section_start_page)

            located_page, opinion_text = st_locate_opinion_structured_page(
                proc_pages=proc_pages,
                section_start_page=section_start_page,
                section_end_page=section_end_page,
            )

            if pd.isna(located_page) or not opinion_text:
                print(
                    f"[structured-parser] could not locate structured opinion PROCEDURE table "
                    f"for committee == {expected_opinion_committee} in procedure == {procedure}"
                )
                continue

            print(
                f"found structured opinion table for committee == {expected_opinion_committee} "
                f"in procedure == {procedure} on page == {located_page}"
            )

            extracted = st_parse_structured_procedure_page(opinion_text)
            extracted_df = pd.DataFrame(extracted)

            if not extracted_df.empty:
                opinion_code_rows = extracted_df.loc[
                    extracted_df["head"].map(st_normalize_for_match) == "OPINION BY"
                ]
                if not opinion_code_rows.empty:
                    opinion_code = st_norm_space(opinion_code_rows.iloc[0]["value"]).upper()
                    st_assert_or_warn_committee_match(
                        procedure=procedure,
                        designation="opinion",
                        expected_committee=expected_opinion_committee,
                        code=opinion_code,
                        committee_code_lookup=committee_code_lookup,
                    )

                for item in extracted:
                    st_add_row(
                        rows=rows,
                        procedure=procedure,
                        committee=expected_opinion_committee,   # outer committee stays the df_committees name
                        designation="opinion",
                        head=item["head"],
                        value=item["value"],
                    )

    out = pd.DataFrame(rows, columns=["procedure", "committee", "designation", "head", "value"])
    if not out.empty:
        out = out.drop_duplicates().reset_index(drop=True)

    return out


# ------------------------------------------------------------
# run structured parser
# ------------------------------------------------------------
df_committee_meta_long_structured_new = build_structured_committee_meta_long_pdf(
    df_page_text=df_page_text,
    contents_df=contents_df,
    df_committees=df_committees,
    df_dict=df_dict,
)

# ------------------------------------------------------------
# append beneath the prose-parser output already built
# ------------------------------------------------------------
if "df_committee_meta_long" in globals():
    df_committee_meta_long = (
        pd.concat([df_committee_meta_long, df_committee_meta_long_structured_new], ignore_index=True)
        .drop_duplicates()
        .reset_index(drop=True)
    )
elif "df_committee_meta_long_new" in globals():
    df_committee_meta_long = (
        pd.concat([df_committee_meta_long_new, df_committee_meta_long_structured_new], ignore_index=True)
        .drop_duplicates()
        .reset_index(drop=True)
    )
else:
    df_committee_meta_long = df_committee_meta_long_structured_new.copy()

display(df_committee_meta_long_structured_new)
display(df_committee_meta_long)

found structured lead table in procedure == 2014/0091(COD) on page == 119
found structured opinion table for committee == COMMITTEE ON EMPLOYMENT AND SOCIAL AFFAIRS in procedure == 2014/0091(COD) on page == 101
found structured opinion table for committee == COMMITTEE ON WOMEN'S RIGHTS AND GENDER EQUALITY in procedure == 2014/0091(COD) on page == 118


,procedure,committee,designation,head,value
0,2014/0091(COD),Committee on Economic and Monetary Affairs,lead,Title,Activities and supervision of institutions for...
1,2014/0091(COD),Committee on Economic and Monetary Affairs,lead,References,COM(2014)0167 – C7-0112/2014 – 2014/0091(COD)
2,2014/0091(COD),Committee on Economic and Monetary Affairs,lead,Date submitted to Parliament,19.3.2014
3,2014/0091(COD),Committee on Economic and Monetary Affairs,lead,Committee responsible,ECON
4,2014/0091(COD),Committee on Economic and Monetary Affairs,lead,Date announced in plenary,14.4.2014
5,2014/0091(COD),Committee on Economic and Monetary Affairs,lead,Committees asked for opinions,EMPL
6,2014/0091(COD),Committee on Economic and Monetary Affairs,lead,Committees asked for opinions,JURI
7,2014/0091(COD),Committee on Economic and Monetary Affairs,lead,Committees asked for opinions,FEMM
8,2014/0091(COD),Committee on Economic and Monetary Affairs,lead,Not delivering opinions,JURI
9,2014/0091(COD),Committee on Economic and Monetary Affairs,lead,Date of decision,3.9.2014


,procedure,committee,designation,head,value
0,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Title,energy efficiency requirements for household e...
1,1994/0272(COD),"Committee on Research, Technological Developme...",lead,References,COM(94)O521
2,1994/0272(COD),"Committee on Research, Technological Developme...",lead,References,C4
3,1994/0272(COD),"Committee on Research, Technological Developme...",lead,References,OO2O/95
4,1994/0272(COD),"Committee on Research, Technological Developme...",lead,References,94/O272(COD)
5,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Rapporteurs,Allan Macartney
6,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Committee responsible,"Committee on Research, Technological Developme..."
7,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Date appointed,5 October 1994
8,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Discussed in committee,24 April 1995
9,1994/0272(COD),"Committee on Research, Technological Developme...",lead,Discussed in committee,30 May 1995
